# [실습] 계층 모형 요율표 — 관측이 적은 세그먼트 추정

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## 오늘의 질문과 데이터

<div style="background:#FFFFFF;color:#000000;padding:40px;border-radius:12px;border:1px solid #E6E6E6;border-left:4px solid #051C2C">
<div style="font-size:0.9em;color:#656565">모형추정 심화통계 · Day 5</div>
<h1 style="color:#000000;margin:10px 0">계층 모형 요율표 — 관측이 적은 세그먼트의 추정</h1>
<div style="font-size:1.05em;color:#656565">5일 동안 진행한 요율 산정 프로젝트의 마지막 날 — 프랑스 자동차보험 실계약 678,013건</div>
</div>

## 오늘 할 일

4일 동안 만든 요율표에는 구간을 좁히지 못한 세그먼트가 남아 있습니다. **"계약 63건인 세그먼트의 요율을 다른 세그먼트의 기록까지 함께 사용해 추정하려 합니다. 그 값을 신뢰할 수 있는지는 무엇으로 판정하고, 요율표에는 사람이 지정한 사전분포가 낸 값과 데이터로 추정한 사전분포가 낸 값 가운데 무엇을 적습니까."**

미션 3개로 계층 모형을 만들고 사후 표본의 수렴을 판정한 뒤, 세그먼트별 요율표를 만드는 미니 프로젝트로 마칩니다. 미션 1·2 는 수업에서, 미션 3 과 과제는 방과후에 진행합니다.

## 오늘의 데이터

**freMTPL2freq** 는 프랑스 자동차 책임보험의 실제 계약 기록입니다. 계약 한 건이 한 행이고 678,013행 × 12컬럼입니다. 내려받은 직후에 적용하는 규칙은 첫날에 적용한 규칙 2개와 같습니다.

| 컬럼 | 뜻 | 내려받은 직후에 적용하는 규칙 |
|---|---|---|
| `ClaimNb` | 보험 기간 중 사고(청구) 건수 (건) | 상한 4건으로 상한 처리 — 4건을 초과하는 계약이 소수 있어, 그 몇 건이 전체 추정을 좌우하지 않게 합니다 |
| `Exposure` | 그 계약이 보장을 받은 기간 (년) | 상한 1.0년으로 상한 처리 — 같은 이유입니다 |
| `VehPower` | 차량 출력 등급 (4 부터 15 까지의 정수) | 그대로 사용 — 마력이 아니라 보험사가 매긴 등급이라 6 은 「출력 등급 6」입니다 |

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Japanese_car_accident.jpg" width="640" height="360"/>
<p align="center"><em>▲ 교차로 접촉 사고 — 위 표에 적힌 freMTPL2freq 의 한 행 한 행은 이런 현장의 기록이다. 계약 수가 적어 추정값의 변동이 컸던 세그먼트의 요율을 정하는 것이 마지막 날의 목표 (출처: Wikipedia, Japanese_car_accident.jpg)</em></p>

미션 1·2 는 지역 × 브랜드 **242개 세그먼트**를 쓰고, 여기서 다루는 값은 연간 사고율 λ(단위: 건/년)입니다. 미션 3 과 과제는 지역 × 브랜드 × 차량 출력 등급 **2,323개 세그먼트**를 쓰고, 여기서 다루는 값은 사고 경험률 p(단위: 비율)입니다.

**연간 사고율 λ** 는 그 세그먼트의 사고 건수를 계약이 보장을 받은 기간(년), 곧 노출로 나눈 값이고, **사고 경험률 p** 는 사고를 1번 이상 겪은 계약 수를 그 세그먼트의 계약 수로 나눈 값입니다.

## 이 노트북을 쓰는 법

**환경** — ▶ 실행은 Google Colab 연결 안내를 따르고, AI 튜터는 상단 「Claude 도우미」를 설치한 뒤에 사용합니다. **채점은 Colab 이 연결되면 자동으로 준비됩니다. 코드 패널 위쪽에 「채점 준비됨」이 표시된 뒤에 미션을 시작하고, 미션 셀을 실행했는데 채점 결과가 출력되지 않으면 그 「채점 준비됨」을 누르세요.**

노트북을 내려받아 열었다면 맨 위의 접힌 `[채점]` 셀을 한 번 ▶ 실행한 뒤에 시작합니다. 채점이 끝나 🔓 가 표시되면 「🔓 완성본 풀이 보기」를 열어 내 코드와 비교합니다.

**맨 위 두 셀을 먼저 실행하세요.** 설치 셀이 표집기 `pymc` 를 내려받는 과정과 사후 표본을 얻는 계산이 몇 분까지 걸리므로, 진행 표시가 없어도 실행을 중단하지 마세요.

미션마다 코드 셀이 2개입니다. 앞의 **준비 셀** 은 그대로 실행하고, 뒤의 **PROMPT 셀** 에 시킬 일을 내 말로 적으면 표시선 아래에 AI 가 코드를 작성합니다. 실행하면 채점 결과가 출력됩니다.

**출력 이름 규칙** — 「만들 것」에 굵게 적힌 이름을 글자 그대로 사용해 「이름 = 값 단위」 한 행으로 출력하세요. 변수 이름은 자유이며, 채점 기준은 이 출력 행과 그림입니다. 값은 자릿수가 아니라 모형이 내는 범위로 판정하므로 마지막 자릿수가 달라도 통과합니다.

**주고받기 규칙 2가지** — AI 튜터와 주고받다가 막혔을 때는 이 2가지만 지킵니다.

| 이럴 때 | 이렇게 합니다 |
|---|---|
| 같은 말을 두 번째 되풀이하게 될 때 | 말을 바꾸지 말고 `PROMPT` 에 **빠진 조건을 하나** 추가합니다 |
| 조건을 추가해도 안 될 때 | 앞 대화를 이어서 쓰지 말고, 요구를 한 덩어리로 다시 적어 새 메시지로 보냅니다 |

In [ ]:
# 실습 준비 - 패키지 설치
!pip install -q "numpy>=1.26" "pandas>=2.0" "scipy>=1.11" "matplotlib>=3.7" "scikit-learn>=1.3" "pymc==6.3.2" "arviz==1.3.0"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pymc as pm
import arviz as az
import warnings; warnings.filterwarnings("ignore")

# 한글 폰트 — Colab 이면 나눔고딕을 설치한 뒤 그 ttf 를 폰트 목록에 **직접** 등록한다.
#  설치는 디스크에 파일만 만든다. 이미 떠 있는 폰트 목록은 그 사실을 모르므로 addfont 로
#  넣어 줘야 이름이 보인다(넣지 않으면 DejaVu Sans 로 밀려 한글이 전부 □ 로 나온다).
import glob, shutil, subprocess
if shutil.which("apt-get"):
    subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"], check=False)
for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):   # 설치 여부와 무관하게 한 번
    try: fm.fontManager.addfont(_p)
    except Exception: pass
try: fm._get_font.cache_clear()
except Exception: pass
installed = {f.name for f in fm.fontManager.ttflist}
FONT = next((n for n in ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"] if n in installed), "DejaVu Sans")
plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = FONT in {"Pretendard", "DejaVu Sans"}   # 나머지 폰트에는 U+2212 가 없다
%config InlineBackend.print_figure_kwargs = {"bbox_inches": None}   # 그림마다 폭이 깎이지 않게
plt.rcParams["figure.dpi"] = plt.rcParams["savefig.dpi"] = 150      # 폭 FIG_W x 150dpi = 1500px 고정
plt.rcParams["figure.autolayout"] = True                            # 여백은 tight_layout 이 알아서
plt.rcParams.update({"font.size": 11, "axes.titlesize": 15, "axes.labelsize": 12,
                     "legend.fontsize": 11, "figure.titlesize": 17, "font.weight": "regular",
                     "text.color": "black", "axes.labelcolor": "black", "axes.titlecolor": "black",
                     "axes.edgecolor": "#757575", "xtick.color": "#757575", "ytick.color": "#757575",
                     "xtick.labelcolor": "black", "ytick.labelcolor": "black",
                     # 위 테두리·오른쪽 테두리·격자를 걷어내 데이터만 남긴다
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False})

# 오늘 그림들이 함께 쓰는 색 — 구간=하늘색, 무 풀링 점=회색, 기준선=남색
COL = {"남색": "#051C2C", "파랑": "#2251FF", "하늘색": "#00A9F4",
       "회색": "#757575", "연회색": "#B3B3B3", "참값": "#757575"}
FIG_W = 10.0     # 캔버스 폭은 모두 이 값으로 고정하고 높이만 그림마다 조절한다

print("사용 폰트:", FONT, "| PyMC", pm.__version__, "· ArviZ", az.__version__,
      "| 채점 셀은 LMS 가 자동으로 준비합니다")

In [ ]:
#@title [채점] 이 셀을 먼저 실행하세요 (한 번이면 됩니다)
#   미션의 「PROMPT 셀」을 실행하면 그 셀의 출력과 그림을 붙잡아 바로 아래에 채점을 찍습니다.
#   통과하면 그 자리에서 완성본 풀이가 열리고, 아니면 항목별 힌트와 튜터에 보낼 질문이 나옵니다.
#   읽지 않고 실행만 해도 됩니다 — 기대값과 풀이는 이 셀에만 있고 「PROMPT 셀」에는 없습니다.
import base64
import glob
import io
import json
import re
import shutil
import subprocess
import sys

import matplotlib.font_manager as fm      # 아래 폰트 자급이 쓴다
import matplotlib.pyplot as plt


def _font_bootstrap():
    """한글 폰트 자급 — 맨 위 설치·폰트 셀을 건너뛰어도 그래프의 한글이 깨지지 않게.

    LMS 는 이 셀만 자동 실행하므로 폰트 준비도 여기서 한 번 더 한다. 설치·폰트 셀과 같은
    규칙이라 둘 다 돌아도 결과가 같고, apt-get 이 없는 자리(로컬)에서는 곧바로 지나간다.
    설치한 ttf 는 `addfont` 로 폰트 목록에 직접 넣는다 — 넣지 않으면 목록이 옛 상태라
    한글이 □ 로 깨진다(2026-09-09 Colab 실사고).
    """
    try:
        cands = ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"]
        installed = {f.name for f in fm.fontManager.ttflist}
        if not any(n in installed for n in cands) and shutil.which("apt-get"):
            subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"],
                           check=False, timeout=120)
        # 설치는 파일만 만든다 — 이미 떠 있는 폰트 목록에 addfont 로 넣어야 이름이 보인다.
        # (설치를 건너뛴 자리도 나눔 ttf 가 이미 있으면 이 코드만으로 잡힌다.)
        for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):
            try: fm.fontManager.addfont(_p)
            except Exception: pass
        try: fm._get_font.cache_clear()
        except Exception: pass
        installed = {f.name for f in fm.fontManager.ttflist}
        _font = next((n for n in cands if n in installed), "DejaVu Sans")
        plt.rcParams["font.family"] = _font
        plt.rcParams["axes.unicode_minus"] = _font in {"Pretendard", "DejaVu Sans"}
    except Exception:
        print("한글 폰트를 못 찾아 그래프 글자가 깨질 수 있습니다 —"
              " 맨 위 설치·폰트 셀을 실행하세요")


_font_bootstrap()


def _dec_json(s):                          # base64(JSON) → 값. 리스트는 원래 튜플이라 되돌립니다
    def _t(v):
        return ({k: _t(x) for k, x in v.items()} if isinstance(v, dict)
                else tuple(_t(x) for x in v) if isinstance(v, list) else v)
    return _t(json.loads(base64.b64decode(s).decode("utf-8")))


# 채점 기준 — base64(JSON), 통과 판정에만 쓴다
_ANS = _dec_json(
    "eyJtMSI6eyJ2YWx1ZXMiOnsi64W47LacIDYuNjLrhYQg7IS46re466i87Yq47J2YIOu2gOu2hCDtkoDrp4Eg6rCSICjqsbQv64WE"
    "KSI6WzAuMDk1LDAuMTE1LCLqsbQv64WEIl0sIuq1rOqwhOydtCDsmYTsoIQg7ZKA66eBIOqwkuydhCDtj6ztlajtlZwg7IS46re4"
    "66i87Yq4IOyImCAo6rCcKSI6WzExLjUsMTIuNSwi6rCcIl19LCJwbG90Ijp7ImtpbmQiOiJpbnRlcnZhbCIsIm5fc3BhbnMiOlsx"
    "MiwyNl0sIm5fcG9pbnRzIjpbMTIsMzBdLCJ4X3JhbmdlIjpbMC4wNjYsMC4xNDVdLCJ2bGluZV94IjpbMC4wOTgsMC4xMDRdLCLs"
    "tpXrnbzrsqgiOnRydWV9fSwibTIiOnsidmFsdWVzIjp7IuqzoOy5nCDrkqQg6rCA7J6lIO2BsCBSLWhhdCI6WzAuOTk1LDEuMDEs"
    "IiJdLCLqs6DsuZwg65KkIOqwgOyepSDsnpHsnYAg7Jyg7Zqo7ZGc67O47IiYICjrsowpIjpbMTUwMCwxNTAwMCwi67KMIl19LCJw"
    "bG90Ijp7ImtpbmQiOiJzY2F0dGVyIiwibl9wb2ludHMiOlsyNDIsNTIwXSwiaGxpbmVfeSI6WzEuMDA1LDEuMDE1XSwi7LaV6528"
    "67KoIjp0cnVlfX0sIm0zIjp7InZhbHVlcyI6eyLqs4Tslb0gNjPqsbQg7IS46re466i87Yq47J2YIOu2gOu2hCDtkoDrp4Eg6rCS"
    "ICjruYTsnKgpIjpbMC4wNDUsMC4wNTYsIuu5hOycqCJdLCLqt7gg7IS46re466i87Yq47J2YIDk1JSDsi6DsmqnqtazqsIQg6ri4"
    "7J20ICglcCkiOlszLjgsNS40LCIlcCJdfSwicGxvdCI6eyJraW5kIjoic2NhdHRlciIsIm5fcG9pbnRzIjpbMjAwMCwyNTAwXSwi"
    "eHNjYWxlIjoibG9nIiwiaGxpbmVfeSI6WzAuOSwxLjFdLCLstpXrnbzrsqgiOnRydWV9fSwi6rO87KCcIjp7InZhbHVlcyI6eyLs"
    "oIDsnqXtlZwg7ZGc7J2YIOy7rOufvCDsiJggKOqwnCkiOls3LjUsOC41LCLqsJwiXSwi7KCE7LK0IOqzhOyVveydmCDsgqzqs6Ag"
    "6rK97ZeY66WgICjruYTsnKgpIjpbMC4wNTAxLDAuMDUwNCwi67mE7JyoIl19LCJwbG90Ijp7ImtpbmQiOiJsaW5lIiwiaGxpbmVf"
    "eSI6WzAuNDUsMS4xNV0sIuy2leudvOuyqCI6dHJ1ZX19fQ==")
_REVEAL = {                        # 완성본 풀이 — base64, 통과할 때만 디코드해 인쇄합니다
    "m1":
        "8J+UkyDrr7jshZggMSDsmYTshLHrs7gg4oCUIEFJIOy9lOuTnCDtkZzsi5zshKAg7JWE656Y7JeQIOuTpOyWtOyYrCDsvZTrk5zr"
        "ipQg7J20IOygleuPhOyeheuLiOuLpAoK4pagIOydtCDsvZTrk5zrpbwg67aA66W4IO2UhOuhrO2UhO2KuCDsmIgKCiAgICDsnbTr"
        "r7gg66eM65Ok7Ja07KC4IOyeiOuKlCBDTEFJTVMsIEVYUE8sIFJBVywgTEFNX1BPT0wsIEk2NjIsIFRISU4xMiwgSiDrpbwg6re4"
        "64yA66GcIOyTsOqzoCDri6Tsi5wg66eM65Ok7KeA64qUIOunkOyVhCDspJguCiAgICBweW1jIOuhnCAyNDLqsJwg7IS46re466i8"
        "7Yq47J2YIOqzhOy4tSDtj6zslYTshqHsnYQg66eM65Ok7Ja0IOykmCDigJQg66eoIOychOyXkCDrqqjrk6Ag7IS46re466i87Yq4"
        "7JeQIOqzte2GteyduCDroZzqt7gg7IKs6rOg7JyoIG11IO2VmOuCmCwg6rCA7Jq0642w7JeQIOyEuOq3uOuovO2KuOuniOuLpOyd"
        "mCDtjrjssKjrpbwg7IS46re466i87Yq4IOqwhCDtkZzspIDtjrjssKggdGF1IOyXkCB6IOulvCDqs7HtlZwKICAgIHRhdSAqIHog"
        "7ZiV7YOc66GcIOuRkOqzoCwg66eoIOyVhOuemOyXkOyEnCDshLjqt7jrqLztirggaiDsnZgg7IKs6rOgIOqxtOyImOulvCDtj4nq"
        "t6AgZXhwKG11ICsgdGhldGFbal0pICogRVhQT1tqXSDsnbgg7Y+s7JWE7Iah7Jy866GcIOuRkOuptCDrj7wuCiAgICDssrTsnbgg"
        "NOqwnOyXkCDrsJjrs7UgMTAwMO2ajCwg7Iuc65Oc64qUIDQ5IOuhnCDqs6DsoJXtlbQg7KSYLiDsgqztm4Qg7ZGc67O47JeQ7ISc"
        "IOyEuOq3uOuovO2KuOuniOuLpOydmCDsgqzqs6DsnKjsnYQgZXhwKG11ICsgdGhldGEpIOuhnCDrs4DtmZjtlZwg65KkCiAgICBJ"
        "NjYyIOuyiCDshLjqt7jrqLztirjsnZgg7IKs7ZuE7Y+J6reg7J2EICLrhbjstpwgNi42MuuFhCDshLjqt7jrqLztirjsnZgg67aA"
        "67aEIO2SgOungSDqsJIgPSAo7Iir7J6QKSDqsbQv64WEIiDsnLzroZwsIOydtOumhOqzvCDri6jsnITrpbwg7ZWo6ruYIOyGjOyI"
        "mOygkAogICAg7JWE656YIDTsnpDrpqzquYzsp4AgcHJpbnQg7ZW0IOykmC4gVEhJTjEyIOyXkCDri7TquLQgMTLqsJwg7IS46re4"
        "66i87Yq466eI64ukIOyCrO2bhCDtkZzrs7jsnZggNSUg67aE7JyE7IiY7JmAIDk1JSDrtoTsnITsiJjrpbwg6rWs7ZW07IScIOq3"
        "uCDqtazqsITsnbQKICAgIExBTV9QT09MIOydhCDtj6ztlajtlZjripQg7IS46re466i87Yq46rCAIOuqhyDqsJzsnbjsp4Ag7KeR"
        "6rOE7ZW0ICLqtazqsITsnbQg7JmE7KCEIO2SgOungSDqsJLsnYQg7Y+s7ZWo7ZWcIOyEuOq3uOuovO2KuCDsiJggPSAo7Iir7J6Q"
        "KSDqsJwiIOuhnCDtlZwg7ZaJIOuNlCDstpzroKXtlbQg7KSYLgogICAg64yA7KGw7Jqp7Jy866GcICLqsJnsnYAg7IS46re466i8"
        "7Yq47J2YIOustCDtkoDrp4Eg6rCSID0gKOyIq+yekCkg6rG0L+uFhCIg6rO8ICLsmYTsoIQg7ZKA66eBIOqwkiA9ICjsiKvsnpAp"
        "IOqxtC/rhYQiIOuPhCDshJzroZwg64uk66W4IO2WieyXkCDstpzroKXtlZjqs6AuCiAgICDqt7jrnpjtlITripQg7ZWcIOyepeun"
        "jCDqt7jroKQg7KSYIOKAlCDshLjroZzroZwgMTLqsJwg7IS46re466i87Yq466W8IOuwsOy5mO2VmOqzoCDshLjqt7jrqLztirjr"
        "p4jri6Qg6re4IDkwJSDqtazqsITsnYQg6rCA66GcIOunieuMgOuhnCDqt7jrprAg65KkLCDqsJnsnYAKICAgIOyEuOq3uOuovO2K"
        "uOydmCBSQVcg6rCS7J2EIOygkOycvOuhnCDtlZjrgpjslKkg7ZGc7Iuc7ZWY6rOgLCBMQU1fUE9PTCDqsJIg7JyE7LmY7JeQIOyE"
        "uOuhnOyEoOydhCDquIvqs6AsIOqwgOuhnOy2leqzvCDshLjroZzstpXsl5Ag652867Ko7J2EIOuLrOyVhCDspJguCgogICAgd2l0"
        "aCBwbS5Nb2RlbCgpIGFzIGhpZXI6CiAgICAgICAgbXUgPSBwbS5Ob3JtYWwoIm11IiwgbnAubG9nKDAuMTApLCAxLjApICAgICAg"
        "ICAgICAgICAgIyDrp6gg7JyEIOKAlCDqs7XthrUg66Gc6re4IOyCrOqzoOycqCDtlZjrgpgKICAgICAgICB0YXUgPSBwbS5IYWxm"
        "Tm9ybWFsKCJ0YXUiLCAxLjApICAgICAgICAgICAgICAgICAgICAgICAjIOyEuOq3uOuovO2KuCDqsIQg7ZGc7KSA7Y647LCoCiAg"
        "ICAgICAgeiA9IHBtLk5vcm1hbCgieiIsIDAuMCwgMS4wLCBzaGFwZT1KKSAgICAgICAgICAgICAgICAgIyDshLjqt7jrqLztirjr"
        "p4jri6TsnZgg7Y647LCoICh0YXUg66W8IOqzse2VtCDslIHri4jri6QpCiAgICAgICAgdGhldGEgPSBwbS5EZXRlcm1pbmlzdGlj"
        "KCJ0aGV0YSIsIHRhdSAqIHopCiAgICAgICAgcG0uUG9pc3Nvbigib2JzIiwgbXU9cG0ubWF0aC5leHAobXUgKyB0aGV0YSkgKiBF"
        "WFBPLCBvYnNlcnZlZD1DTEFJTVMpCiAgICAgICAgcG9zdGVyaW9yID0gcG0uc2FtcGxlKGRyYXdzPTEwMDAsIHR1bmU9MTAwMCwg"
        "Y2hhaW5zPTQsIHJhbmRvbV9zZWVkPTQ5LCBwcm9ncmVzc2Jhcj1GYWxzZSkKCiAgICBsb2dfdmFscyA9IHBvc3Rlcmlvci5wb3N0"
        "ZXJpb3JbIm11Il0udmFsdWVzWy4uLiwgTm9uZV0gKyBwb3N0ZXJpb3IucG9zdGVyaW9yWyJ0aGV0YSJdLnZhbHVlcwogICAgc2Vn"
        "X3ZhbHMgPSBucC5leHAobG9nX3ZhbHMpLnJlc2hhcGUoLTEsIEopICAgICAgICAgICAgICAgICAgICAgICAgICAjIO2RnOuzuCB4"
        "IOyEuOq3uOuovO2KuAogICAgcGFydGlhbF9wb29sID0gc2VnX3ZhbHMubWVhbihheGlzPTApCiAgICBjaV9sbywgY2lfaGkgPSBu"
        "cC5wZXJjZW50aWxlKHNlZ192YWxzWzosIFRISU4xMl0sIFs1LCA5NV0sIGF4aXM9MCkKICAgIGNvdmVyZWRfc2VncyA9IGludCgo"
        "KGNpX2xvIDw9IExBTV9QT09MKSAmIChjaV9oaSA+PSBMQU1fUE9PTCkpLnN1bSgpKQogICAgcHJpbnQoZiLrhbjstpwgNi42MuuF"
        "hCDshLjqt7jrqLztirjsnZgg67aA67aEIO2SgOungSDqsJIgPSB7cGFydGlhbF9wb29sW0k2NjJdOi40Zn0g6rG0L+uFhCIpCiAg"
        "ICBwcmludChmIuq1rOqwhOydtCDsmYTsoIQg7ZKA66eBIOqwkuydhCDtj6ztlajtlZwg7IS46re466i87Yq4IOyImCA9IHtjb3Zl"
        "cmVkX3NlZ3N9IOqwnCIpCiAgICBwcmludChmIuqwmeydgCDshLjqt7jrqLztirjsnZgg66y0IO2SgOungSDqsJIgPSB7UkFXW0k2"
        "NjJdOi40Zn0g6rG0L+uFhCIpICAgICAjIOuMgOyhsOyaqSDigJQg7LGE7KCQ7J20IOuztOuKlCDtlonsnbQg7JWE64uZ64uI64uk"
        "CiAgICBwcmludChmIuyZhOyghCDtkoDrp4Eg6rCSID0ge0xBTV9QT09MOi40Zn0g6rG0L+uFhCIpICAgICAgICAgICAgICAgICAg"
        "ICAjIOuMgOyhsOyaqSDigJQg7LGE7KCQ7J20IOuztOuKlCDtlonsnbQg7JWE64uZ64uI64ukCgogICAgeXBvcyA9IG5wLmFyYW5n"
        "ZSgxMikKICAgIGZpZywgYXggPSBwbHQuc3VicGxvdHMoZmlnc2l6ZT0oOCwgMy44KSkKICAgIGF4LmhsaW5lcyh5cG9zLCBjaV9s"
        "bywgY2lfaGksIGx3PTQsIGNvbG9yPSIjMDBBOUY0IikgICAgICAgICAgICAjIOyEuOq3uOuovO2KuOuniOuLpOydmCA5MCUg7Iug"
        "7Jqp6rWs6rCECiAgICBheC5wbG90KFJBV1tUSElOMTJdLCB5cG9zLCAibyIsIGNvbG9yPSIjNzU3NTc1IikgICAgICAgICAgICMg"
        "6rCZ7J2AIOyEuOq3uOuovO2KuOydmCDrrLQg7ZKA66eBIOqwkgogICAgYXguYXh2bGluZShMQU1fUE9PTCwgbHM9Ii0tIiwgY29s"
        "b3I9IiMwNTFDMkMiKSAgICAgICAgICAgICAjIOyZhOyghCDtkoDrp4Eg6rCSIOychOy5mOydmCDshLjroZzshKAKICAgIGF4LnNl"
        "dF95dGlja3MoeXBvcykKICAgIGF4LnNldF95dGlja2xhYmVscyhbZiJ7RVhQT1tqXTouMWZ964WEIiBmb3IgaiBpbiBUSElOMTJd"
        "KQogICAgYXguc2V0X3hsYWJlbCgi7Jew6rCEIOyCrOqzoOycqCAo6rG0L+uFhCkiKQogICAgYXguc2V0X3lsYWJlbCgi64W47Lac"
        "7J20IOyggeydgCAxMuqwnCDshLjqt7jrqLztirgiKQogICAgcGx0LnNob3coKQoK7ZW07ISdCgotIOuFuOy2nCA2LjYy64WE7JeQ"
        "IOyCrOqzoCAy6rG07J20642YIOyEuOq3uOuovO2KuOydmCDrrLQg7ZKA66eBIOqwkuydgCAwLjMwMjEg6rG0L+uFhOydtOyXiOuK"
        "lOuNsCwg67aA67aEIO2SgOungeydgCDqt7gg7IS46re466i87Yq466W8IDAuMTA0MCDqsbQv64WE7Jy866GcIOydtOuPmeyLnO2C"
        "teuLiOuLpC4gNi4264WE7J2EIOq0gOy4oe2VtCAy6rG07J2EIOuzuCDqsoPrp4zsnLzroZwg44CM7J20IOyEuOq3uOuovO2KuOuK"
        "lCDsoITssrTsnZggM+uwsOOAjeudvOqzoCDsmpTsnKjtkZzsl5Ag7KCB7J2EIOyImOuKlCDsl4bri6TripQg65y77J6F64uI64uk"
        "LgotIOq0gOy4oeydtCDsoIHsnYAgMTLqsJwg7IS46re466i87Yq464qUIDEy6rCcIOuqqOuRkCA5MCUg7Iug7Jqp6rWs6rCE7J20"
        "IOyZhOyghCDtkoDrp4Eg6rCS7J2EIO2PrO2VqO2VqeuLiOuLpC4g7J20IOq1rOqwhOydgCDri6Trpbgg7IS46re466i87Yq47J2Y"
        "IOq4sOuhneq5jOyngCDtlajqu5gg7IKs7Jqp7ZWcIOyCrO2bhOq1rOqwhOydtOudvCDsnbTrr7gg7JmE7KCEIO2SgOungSDqsJIg"
        "7Kq97Jy866GcIOydtOuPme2VtCDsnojsnLzrr4DroZwsIDEy6rCc6rCAIOuqqOuRkCDtj6ztlajsnLzroZwg64KY7Jik64qUIOqy"
        "g+ydgCDqt7gg7J2064+Z7J2YIOqysOqzvOydtOyngCDqs4Tslb0g7KCE7LK07JmAIOqyrOykgCDqsoDsoJXsnZgg6rKw6rO86rCA"
        "IOyVhOuLmeuLiOuLpC4KLSDrsJjrjIDsqr3rj4Qg7ZWo6ruYIOuztOyEuOyalC4g64W47Lac7J20IOqwgOyepSDrp47snYAg7IS4"
        "6re466i87Yq4KDM1LDk0OeuFhCnripQg66y0IO2SgOungeuPhCDrtoDrtoQg7ZKA66eB64+EIDAuMDg5MCDqsbQv64WE7Jy866Gc"
        "IOqwmeydgCDqsJLsnoXri4jri6QuIOuFuOy2nCDsgqzrtoTsnITroZwg64KY64igIOuztOuptCDqtIDsuKHsnbQg7KCB7J2AIOyq"
        "veydgCDqsJLsnbQg7KSR7JWZ6rCSIDQ0LjUlIOydtOuPme2VmOqzoCDqtIDsuKHsnbQg66eO7J2AIOyqveydgCAwLjczJSDsnbTr"
        "j5ntlanri4jri6QuCi0g7IKs6rOg6rCAIO2VnCDqsbTrj4Qg7JeG642YIDnqsJwg7IS46re466i87Yq464+EIOqwkuydtCAwIOyd"
        "tCDslYTri4jrnbwgMC4wOTEg67aA7YSwIDAuMDk3IOyCrOydtOyXkCDsnITsuZjtlanri4jri6QuIOuEt+ynuCDrgqDquYzsp4Ag"
        "44CMMCDsnYQg7JqU7Jyo7ZGc7JeQIOyWtOuWu+qyjCDsoIHripTqsIDjgI3roZwg64Ko642YIOusuOygnOqwgCwg66qo7ZiV7J2E"
        "IOuwlOq+uOyekCDrlLDroZwg7LKY66as7ZWY7KeAIOyViuyVhOuPhCDtlbTshozrkJjsl4jsirXri4jri6QuCi0g7IS46re466i8"
        "7Yq4IOqwhCDtkZzspIDtjrjssKggdGF1IOuKlCDsgqztm4Ttj4nqt6AgMC4yMTA3LCA5MCUg7Iug7Jqp6rWs6rCEIFswLjE4NTYs"
        "IDAuMjM4OF0g7J6F64uI64ukLiDsnbQg7ZGc7KSA7Y647LCo6rCAIOyWvOuniOuCmCDsnpHsnYDsp4DqsIAg6rOnIOOAjOuLpOul"
        "uCDshLjqt7jrqLztirjsnZgg7KCV67O066W8IOyWvOuniOuCmCDqsJXtlZjqsowg7IKs7Jqp7ZWY64qU6rCA44CN66W8IOygle2V"
        "qeuLiOuLpC4KCuy9lOuTnOqwgCDri7XsnYQg66eM65Oc64qUIOuwqeyLnSDigJQg7IS46re466i87Yq466eI64uk7J2YIO2OuOyw"
        "qOulvCDshLjqt7jrqLztirgg6rCEIO2RnOykgO2OuOywqCB0YXUg7ZWY64KY66GcIOustuyWtCAyNDLqsJwg7IS46re466i87Yq4"
        "66W8IO2VqOq7mCDstpTsoJXtlanri4jri6Qo6re4656Y7IScIOuFuOy2nOydtCDsoIHsnYAg7IS46re466i87Yq47J287IiY66Gd"
        "IOqwkuydtCDsmYTsoIQg7ZKA66eBIOqwkiDsqr3snLzroZwg642UIOunjuydtCDsnbTrj5ntlanri4jri6QpLg==",
    "m2":
        "8J+UkyDrr7jshZggMiDsmYTshLHrs7gg4oCUIOyghOyehCDri7Tri7nsnpDsnZgg7L2U65Oc64qUIOq3uOuMgOuhnCDrkZDqs6Ag"
        "7ZGc7KeR6riw66W8IOyngOygle2VmOyngCDslYrsnYAg7Iuk7ZaJ7J2EIO2VmOuCmCDrjZQg66eM65Ot64uI64ukCgrilqAg7J20"
        "IOy9lOuTnOulvCDrtoDrpbgg7ZSE66Gs7ZSE7Yq4IOyYiAoKICAgIOyVhOuemCDsoITsnoQg64u064u57J6QIOy9lOuTnOuKlCDt"
        "lZwg7ZaJ64+EIOqzoOy5mOyngCDslYrqs6Ag6re464yA66GcIO2VnCDrsogg7Iuk7ZaJ7ZW0IOqysOqzvOulvCDrgqjqsqgg65GQ"
        "6rOgLCBzdGVwIOyduOyekOuhnCDtkZzsp5HquLDrpbwg7KeA7KCV7ZWcIOy9lOuTnCDtlonsnbQKICAgIOyXhuuKlCDsi6Ttlons"
        "nYQg7ZWY64KYIOuNlCDrp4zrk6TslrQg7KSYLiDqs6DsuZwg7Iuk7ZaJ7J2AIOyytOyduCA06rCc7JeQIOuwmOuztSA1MDAw7ZqM"
        "LCDsm4zrsI3sl4Xrj4QgNTAwMCDsnLzroZwg7ZWY6rOgIOyLnOuTnOuKlCA0OSDroZwg6rOg7KCV7ZW0IOykmC4g66qo7ZiV7J2Y"
        "CiAgICDrgpjrqLjsp4Ag7L2U65OcIO2WieydgCDqt7jrjIDroZwg65GQ6rOgLCDsnbTrr7gg66eM65Ok7Ja07KC4IOyeiOuKlCBD"
        "TEFJTVMsIEVYUE8sIEog66W8IOuLpOyLnCDrp4zrk6Tsp4DripQg66eQ7JWEIOykmC4KICAgIOqzoOy5nCDsi6Ttlonsl5DshJwg"
        "YXJ2aXog66GcIOuqqOyImCDsoITssrTsnZggUi1oYXQg6rO8IOycoO2aqO2RnOuzuOyImOulvCDqs4TsgrDtlbQg7KSYLiDqsIDs"
        "nqUg7YGwIFItaGF0IOydhAogICAgIuqzoOy5nCDrkqQg6rCA7J6lIO2BsCBSLWhhdCA9ICjsiKvsnpApIiDroZwg7IaM7IiY7KCQ"
        "IOyVhOuemCA07J6Q66as6rmM7KeALCDqsIDsnqUg7J6R7J2AIOycoO2aqO2RnOuzuOyImOulvAogICAgIuqzoOy5nCDrkqQg6rCA"
        "7J6lIOyekeydgCDsnKDtmqjtkZzrs7jsiJggPSAo7Iir7J6QKSDrsowiIOuhnCDri6jsnITrpbwg67aZ7JesIHByaW50IO2VtCDs"
        "pJguCiAgICDqsJnsnYAg7ZGc67O47JeQ7IScIOqzte2GtSDtj4nqt6AgbXUg7J2YIOyCrO2bhO2Pieq3oOqzvCDqt7gg6rCS7J2E"
        "IOyngOyImOuhnCDrs7Xsm5DtlZwg7Jew6rCEIOyCrOqzoOycqOuPhCDtlZwg7ZaJIOuNlCBwcmludCDtlbQg7KSYLgogICAg6re4"
        "656Y7ZSE64qUIO2VnCDsnqXrp4wg6re466CkIOykmC4g6rOg7LmcIOyLpO2WieydmCDshLjqt7jrqLztirgg7Y647LCoIHRoZXRh"
        "IOydmCBSLWhhdCAyNDLqsJzrpbwg6rCA66Gc7LaVIOyEuOq3uOuovO2KuCDrsojtmLjsl5Ag64yA6rOgIOygkOycvOuhnAogICAg"
        "7LCN6rOgLCAxLjAxIOychOy5mOyXkCDqsIDroZzshKDsnYQg7ZWY64KYIOq3uOydgCDri6TsnYwg6rCA66Gc7LaV6rO8IOyEuOuh"
        "nOy2leyXkCDrnbzrsqjsnYQg64us7JWEIOykmC4KCiAgICB3aXRoIHBtLk1vZGVsKCkgYXMgbGVnYWN5OiAgICAgICAgICAgICAg"
        "ICAgICAgIyDqs6DsuZjquLAg7KCEIOKAlCDsoITsnoQg64u064u57J6QIOy9lOuTnCDqt7jrjIDroZwKICAgICAgICBtdSA9IHBt"
        "Lk5vcm1hbCgibXUiLCBucC5sb2coMC4xMCksIDEuMCkKICAgICAgICB0YXUgPSBwbS5IYWxmTm9ybWFsKCJ0YXUiLCAxLjApCiAg"
        "ICAgICAgeiA9IHBtLk5vcm1hbCgieiIsIDAuMCwgMS4wLCBzaGFwZT1KKQogICAgICAgIHRoZXRhID0gcG0uRGV0ZXJtaW5pc3Rp"
        "YygidGhldGEiLCB0YXUgKiB6KQogICAgICAgIHBtLlBvaXNzb24oIm9icyIsIG11PXBtLm1hdGguZXhwKG11ICsgdGhldGEpICog"
        "RVhQTywgb2JzZXJ2ZWQ9Q0xBSU1TKQogICAgICAgIHRyYWNlX2JlZm9yZSA9IHBtLnNhbXBsZShkcmF3cz0xMDAwLCB0dW5lPTEw"
        "MDAsIGNoYWlucz00LCBzdGVwPXBtLk1ldHJvcG9saXMoKSwKICAgICAgICAgICAgICAgICAgICAgICAgIHJhbmRvbV9zZWVkPTQ5"
        "LCBwcm9ncmVzc2Jhcj1GYWxzZSkKCiAgICB3aXRoIHBtLk1vZGVsKCkgYXMgZml4ZWQ6ICAgICAgICAgICAgICAgICAgICAjIOqz"
        "oOy5nCDrkqQg4oCUIHN0ZXA9IOyXhuydtCDquLDrs7gg7ZGc7KeR6riw66GcIOuRkOqzoCDrsJjrs7Ug7IiY66W8IOuKmOumsCDs"
        "g4gg7Iuk7ZaJCiAgICAgICAgbXUgPSBwbS5Ob3JtYWwoIm11IiwgbnAubG9nKDAuMTApLCAxLjApCiAgICAgICAgdGF1ID0gcG0u"
        "SGFsZk5vcm1hbCgidGF1IiwgMS4wKQogICAgICAgIHogPSBwbS5Ob3JtYWwoInoiLCAwLjAsIDEuMCwgc2hhcGU9SikKICAgICAg"
        "ICB0aGV0YSA9IHBtLkRldGVybWluaXN0aWMoInRoZXRhIiwgdGF1ICogeikKICAgICAgICBwbS5Qb2lzc29uKCJvYnMiLCBtdT1w"
        "bS5tYXRoLmV4cChtdSArIHRoZXRhKSAqIEVYUE8sIG9ic2VydmVkPUNMQUlNUykKICAgICAgICB0cmFjZV9hZnRlciA9IHBtLnNh"
        "bXBsZShkcmF3cz01MDAwLCB0dW5lPTUwMDAsIGNoYWlucz00LAogICAgICAgICAgICAgICAgICAgICAgICAgcmFuZG9tX3NlZWQ9"
        "NDksIHByb2dyZXNzYmFyPUZhbHNlKQoKICAgIHN1bW1hcnlfYWZ0ZXIgPSBhei5zdW1tYXJ5KHRyYWNlX2FmdGVyKSAgICAgICAg"
        "ICAgICAgICAgICAgICAjIOuqqOyImOuniOuLpCDtlZwg7ZaJIOKAlCByX2hhdCDCtyBlc3NfYnVsayDsl7TsnbQg7J6I7Iq164uI"
        "64ukCiAgICBtYXhfcmhhdCA9IGZsb2F0KHN1bW1hcnlfYWZ0ZXJbInJfaGF0Il0ubWF4KCkpICAgICAgICAgICAgICAgIyDrqqjs"
        "iJgg7KCE7LK07JeQ7IScIOqwgOyepSDtgbAg6rCSIO2VmOuCmAogICAgbWluX2VzcyA9IGZsb2F0KHN1bW1hcnlfYWZ0ZXJbImVz"
        "c19idWxrIl0ubWluKCkpICAgICAgICAgICAgICMg6rCZ7J2AIO2RnOuzuOyXkOyEnCDqsIDsnqUg7J6R7J2AIOqwkiDtlZjrgpgK"
        "ICAgIHByaW50KGYi6rOg7LmcIOuSpCDqsIDsnqUg7YGwIFItaGF0ID0ge21heF9yaGF0Oi40Zn0iKQogICAgcHJpbnQoZiLqs6Ds"
        "uZwg65KkIOqwgOyepSDsnpHsnYAg7Jyg7Zqo7ZGc67O47IiYID0ge21pbl9lc3M6LC4wZn0g67KMIikKICAgIG11X21lYW4gPSBm"
        "bG9hdCh0cmFjZV9hZnRlci5wb3N0ZXJpb3JbIm11Il0udmFsdWVzLm1lYW4oKSkgICAjIOydtOuhoOydtCDsnKDtmqjtkZzrs7js"
        "iJgg65WM66y47JeQIOuvuOukhCDrkZQg6rO17Ya1IO2Pieq3oAogICAgbXVfbGFtID0gZmxvYXQobnAuZXhwKHRyYWNlX2FmdGVy"
        "LnBvc3RlcmlvclsibXUiXS52YWx1ZXMpLm1lYW4oKSkKICAgIHByaW50KGYi6rOg7LmcIOuSpCBtdSDsnZgg7IKs7ZuE7Y+J6reg"
        "ID0ge211X21lYW46LjRmfSAo67O17JuQ7ZWY66m0IHttdV9sYW06LjRmfSDqsbQv64WEKSIpCgogICAgcmhhdF9zZWcgPSBhei5z"
        "dW1tYXJ5KHRyYWNlX2FmdGVyLCB2YXJfbmFtZXM9WyJ0aGV0YSJdKVsicl9oYXQiXSAgICMg7IS46re466i87Yq466eI64ukIO2V"
        "mOuCmOyUqSAyNDLqsJwKICAgIGZpZywgYXggPSBwbHQuc3VicGxvdHMoZmlnc2l6ZT0oOCwgMy42KSkKICAgIGF4LnBsb3QobnAu"
        "YXJhbmdlKGxlbihyaGF0X3NlZykpLCByaGF0X3NlZywgIm8iLCBtcz00LCBjb2xvcj0iIzc1NzU3NSIpCiAgICBheC5heGhsaW5l"
        "KDEuMDEsIGxzPSItLSIsIGNvbG9yPSIjMDUxQzJDIikgICAgICAgICAgICAjIOydtOuhoOyXkOyEnCDsoJXtlZwg6riw7KSA6rCS"
        "CiAgICBheC5zZXRfeGxhYmVsKCLshLjqt7jrqLztirgg67KI7Zi4IikKICAgIGF4LnNldF95bGFiZWwoIuqzoOy5nCDrkqQg7IS4"
        "6re466i87Yq467OEIFItaGF0IikKICAgIHBsdC5zaG93KCkKCu2VtOyEpAoKLSDqsJLsnYAg7LKY7J2M67aA7YSwIO2DgOuLue2V"
        "tCDrs7TsmIDsirXri4jri6QuIOyghOyehCDri7Tri7nsnpAg7L2U65Oc7J2YIOyCrO2bhO2Pieq3oCAwLjEwNDEg7J2AIOqzoOy5"
        "nCDrkqTsnZggMC4xMDM4IOqzvCDshozsiJjsoJAg7JWE656YIDPsnpDrpqzquYzsp4Ag6rCZ7Iq164uI64ukLiDti4DrprAg6rKD"
        "7J2AIOqwkuydtCDslYTri4jrnbwg6re4IOqwkuydhCDsi6DrorDtlaAg7IiYIOyeiOuKlOyngCDsuKHsoJXtlZjsp4Ag7JWK7J2A"
        "IOqyg+yeheuLiOuLpC4KLSDtlajsoJXsnYAg65GY66GcIOuCmOuJqeuLiOuLpC4g7ZWY64KY64qUIO2RnOynkeq4sOulvCDsp4Hs"
        "oJEg7KeA7KCV7ZW0IOyytOyduOydtCDqsJnsnYAg7IOB7YOc66W8IOqxsOydmCDrspfslrTrgpjsp4Ag66q77ZWcIOqyg+ydtOqz"
        "oCwg64uk66W4IO2VmOuCmOuKlCDqt7gg7IKs7Iuk7J20IO2ZlOuptOyXkCBSLWhhdCAxLjA5IMK3IOycoO2aqO2RnOuzuOyImCAz"
        "NCDroZwg7Lac66Cl65CY7Ja0IOyeiOyXiOuKlOuNsOuPhCDsnb3sp4Ag7JWK6rOgIOyalOycqO2RnOyXkCDsgqzsmqntlZwg6rKD"
        "7J6F64uI64ukLiDslZ7snZgg6rKD7J2AIOy9lOuTnOydmCDqsrDtlajsnbTqs6Ag65Kk7J2YIOqyg+ydgCDsoIjssKjsnZgg6rKw"
        "7ZWo7J6F64uI64ukLgotIOqzoOy5nCDrkqTsl5DripQg6rCA7J6lIO2BsCBSLWhhdCDsnbQgMS4wMDExLCDqsIDsnqUg7J6R7J2A"
        "IOycoO2aqO2RnOuzuOyImOqwgCA0LDUzN+uyjOuhnCDquLDspIDqsJIgMuqwnOulvCDrqqjrkZAg7Ya16rO87ZWp64uI64ukLiDr"
        "sJjrs7Ug7IiY66eMIOuKmOumrOuKlCDrsKnrspXsnLzroZzripQg7Jes6riw6rmM7KeAIOyYpOyngCDrqrvtlanri4jri6QuIO2R"
        "nOynkeq4sOulvCDqt7jrjIDroZwg65GUIOyxhCDrsJjrs7Ug7IiY66W8IDIw67Cw66GcIOuKmOugpOuPhCDsnKDtmqjtkZzrs7js"
        "iJjripQgNjc267KM7JeQIOq3uOyzkCwg7ZGc7KeR6riw66W8IOuwlOq+vCDsi6TtlonsnZggN+u2hOydmCAxIOyImOykgOyeheuL"
        "iOuLpC4g6riw7KSA6rCS7J2EIO2GteqzvO2VnCDsnbQg7ZGc67O47JeQ7IScIOydtOuhoOydtCDsnKDtmqjtkZzrs7jsiJggMzMw"
        "67KMIOuVjOusuOyXkCDrr7jrpIQg65GUIOqzte2GtSDtj4nqt6AgbXUg64+EIO2VqOq7mCDsnb3sirXri4jri6QuIOyCrO2bhO2P"
        "ieq3oOydhCDrs7Xsm5DtlZjrqbQgMC4wOTYg6rG0L+uFhOyeheuLiOuLpC4KLSDqs6DsuZwg65KkIDI0MuqwnCDshLjqt7jrqLzt"
        "irjsnZggUi1oYXQg7J2AIOyghOu2gCAxLjAwMTEg7J207ZWY6528IOq4sOykgOyEoCDslYTrnpjsl5Ag66qo7J6F64uI64ukLiDq"
        "s6DsuZjquLAg7KCEIOqwkuydgCDsoITsnoQg64u064u57J6QIOy9lOuTnOqwgCDstpzroKXtlZwg7ZGc7JeQIOydtOuvuCDsnojq"
        "s6AsIOq3uCDsi6Ttlonsl5DshJzripQg7IS46re466i87Yq467OEIFItaGF0IOydtCAxLjAwMDcg67aA7YSwIDEuMDUyNyDquYzs"
        "p4Ag7Z2p7Ja07KC4IDQ06rCcIOyEuOq3uOuovO2KuOqwgCDquLDspIDqsJIgMS4wMSDsnYQg7LSI6rO87ZaI7Iq164uI64ukLgot"
        "IFItaGF0IOydgCDrqqjsiJjrp4jri6Qg65Sw66GcIOqzhOyCsOuQqeuLiOuLpC4gMjQy6rCcIOyEuOq3uOuovO2KuOuptCDqs7Xt"
        "hrUg7Y+J6regIO2VmOuCmCDCtyDshLjqt7jrqLztirgg6rCEIO2RnOykgO2OuOywqCDtlZjrgpggwrcg7IS46re466i87Yq466eI"
        "64ukIO2VmOuCmOyUqeydtOudvCDjgIzqsIDsnqUg7YGwIOqwkiDtlZjrgpjjgI3roZwg7KSE7JesIOq4sOykgOqwkuqzvCDruYTq"
        "tZDtlZjripQg6rKD7J20IOyalOycqO2RnCDslZ7sl5DshJwg7ZWgIOydvOyeheuLiOuLpC4KCuy9lOuTnOqwgCDri7XsnYQg66eM"
        "65Oc64qUIOuwqeyLnSDigJQg6rCZ7J2AIOuqqO2YleydhCDtkZzsp5HquLDsmYAg67CY67O1IOyImOunjCDrsJTqv5Qg65GQIOuy"
        "iCDsi6TtlontlZwg65KkLCDsgqztm4Qg7JqU7JW97J20IOyVhOuLiOudvCBSLWhhdCDqs7wg7Jyg7Zqo7ZGc67O47IiY66W8IOuo"
        "vOyggCDsnb3slrQg7JqU7Jyo7ZGc7JeQIOyCrOyaqe2VoOyngOulvCDsoJXtlanri4jri6Qu",
    "m3":
        "8J+UkyDrr7jshZggMyDsmYTshLHrs7gg4oCUIOqzhOyVvSA2M+qxtOyduCDshLjqt7jrqLztirjsnZgg6rWs6rCE7J20IOydtOug"
        "h+qyjCDsp6fslYTsp5Hri4jri6QKCuKWoCDsnbQg7L2U65Oc66W8IOu2gOuluCDtlITroaztlITtirgg7JiICgogICAg7J2066+4"
        "IOunjOuTpOyWtOyguCDsnojripQgTjIzLCBLMjMsIFJBVzIzLCBQX1BPT0wsIEk2MywgU0U2MywgTE9fRDQ1LCBISV9ENDUsIEoy"
        "MyDsnYQg6re464yA66GcIOyTsOqzoCDri6Tsi5wg66eM65Ok7KeA64qUCiAgICDrp5DslYQg7KSYLiBweW1jIOuhnCAyLDMyM+qw"
        "nCDshLjqt7jrqLztirjsnZgg6rOE7Li1IOydtO2VreydhCDrp4zrk6TslrQg7KSYIOKAlCDrp6gg7JyE7JeQIOyghOyytCDroZzs"
        "p5MgbXUg7ZWY64KY66W8IOykkeyLrCAtMi45NCDroZwg65GQ6rOgLCDqsIDsmrTrjbDsl5Ag7IS46re466i87Yq466eI64uk7J2Y"
        "IO2OuOywqOulvCDshLjqt7jrqLztirgg6rCEIO2RnOykgO2OuOywqCB0YXUg7JeQIHog66W8IOqzse2VnAogICAgdGF1ICogeiDt"
        "mJXtg5zroZwg65GQ6rOgLCDrp6gg7JWE656Y7JeQ7IScIOyEuOq3uOuovO2KuCBqIOydmCDsgqzqs6Ag6rKq7J2AIOqzhOyVvSDs"
        "iJggSzIzW2pdIOulvCDsi5ztlokg7IiYIE4yM1tqXSDCtyDtmZXrpaAgc2lnbW9pZChtdSArIHRoZXRhW2pdKQogICAg7J24IOyd"
        "tO2VreycvOuhnCDrkZDrqbQg64+8LiDssrTsnbggNOqwnOyXkCDrsJjrs7UgMjAwMO2ajCwg7JuM67CN7JeFIDIwMDDtmowsIOyL"
        "nOuTnOuKlCA0OSDroZwg6rOg7KCV7ZW0IOykmC4gSTYzIOuyiCDshLjqt7jrqLztirjsnZgg7IKs7ZuE7Y+J6reg7J2ECiAgICAi"
        "6rOE7JW9IDYz6rG0IOyEuOq3uOuovO2KuOydmCDrtoDrtoQg7ZKA66eBIOqwkiA9ICjsiKvsnpApIOu5hOycqCIg66GcIOyGjOyI"
        "mOygkCDslYTrnpggNOyekOumrOq5jOyngCwg6re4IOyEuOq3uOuovO2KuOydmCA5NSUg7Iug7Jqp6rWs6rCEIOq4uOydtOulvCDt"
        "jbzshLztirjtj6zsnbjtirjroZwg67CU6r+UCiAgICAi6re4IOyEuOq3uOuovO2KuOydmCA5NSUg7Iug7Jqp6rWs6rCEIOq4uOyd"
        "tCA9ICjsiKvsnpApICVwIiDroZwgcHJpbnQg7ZWY6rOgLCDssqvrgqDsnZggwrEyIO2RnOykgOyYpOywqCDqtazqsIQg6ri47J20"
        "64+EIExPX0Q0NSDsmYAgSElfRDQ1IOuhnCDqs4TsgrDtlbQKICAgIOqwmeydgCDri6jsnITroZwg64uk66W4IO2WieyXkCDstpzr"
        "oKXtlbQg7KSYLiDqt7jrnpjtlITripQg7ZWcIOyepSDigJQg6rCA66Gc7LaV7JeQIOyEuOq3uOuovO2KuOydmCDqs4Tslb0g7IiY"
        "66W8IOuhnOq3uCDriIjquIjsnLzroZwsIOyEuOuhnOy2leyXkCDshLjqt7jrqLztirjrp4jri6TsnZgg7IOB64yAIOq1rOqwhCDq"
        "uLjsnbTrpbwKICAgIOygkOycvOuhnCDssI3qs6Ag7IOB64yAIOq1rOqwhCDquLjsnbQgMSDsnITsuZjsl5Ag6rCA66Gc7ISg7J2E"
        "IOq3uOydgCDrkqQg65GQIOy2leyXkCDrnbzrsqjsnYQg64us7JWEIOykmC4KCiAgICB3aXRoIHBtLk1vZGVsKCkgYXMgaGllcl9i"
        "aW5vbToKICAgICAgICBtdSA9IHBtLk5vcm1hbCgibXUiLCAtMi45NCwgMS4wKSAgICAgICAgICAgICAgICAgICMg7IKs6rOgIOqy"
        "ve2XmOuloCAwLjA1MDIg66W8IOuhnOynk+ycvOuhnCDrs4DtmZjtlZwg6rCSCiAgICAgICAgdGF1ID0gcG0uSGFsZk5vcm1hbCgi"
        "dGF1IiwgMS4wKQogICAgICAgIHogPSBwbS5Ob3JtYWwoInoiLCAwLjAsIDEuMCwgc2hhcGU9SjIzKQogICAgICAgIHRoZXRhID0g"
        "cG0uRGV0ZXJtaW5pc3RpYygidGhldGEiLCB0YXUgKiB6KQogICAgICAgIHBtLkJpbm9taWFsKCJvYnMiLCBuPU4yMywgcD1wbS5t"
        "YXRoLnNpZ21vaWQobXUgKyB0aGV0YSksIG9ic2VydmVkPUsyMykKICAgICAgICBwb3N0ZXJpb3IgPSBwbS5zYW1wbGUoZHJhd3M9"
        "MjAwMCwgdHVuZT0yMDAwLCBjaGFpbnM9NCwgcmFuZG9tX3NlZWQ9NDksIHByb2dyZXNzYmFyPUZhbHNlKQoKICAgIGxvZ192YWxz"
        "ID0gcG9zdGVyaW9yLnBvc3RlcmlvclsibXUiXS52YWx1ZXNbLi4uLCBOb25lXSArIHBvc3Rlcmlvci5wb3N0ZXJpb3JbInRoZXRh"
        "Il0udmFsdWVzCiAgICBzZWdfdmFscyA9ICgxIC8gKDEgKyBucC5leHAoLWxvZ192YWxzKSkpLnJlc2hhcGUoLTEsIEoyMykKICAg"
        "IGNpX2xvLCBjaV9oaSA9IG5wLnBlcmNlbnRpbGUoc2VnX3ZhbHNbOiwgSTYzXSwgWzIuNSwgOTcuNV0pCiAgICBwcmludChmIuqz"
        "hOyVvSA2M+qxtCDshLjqt7jrqLztirjsnZgg67aA67aEIO2SgOungSDqsJIgPSB7c2VnX3ZhbHNbOiwgSTYzXS5tZWFuKCk6LjRm"
        "fSDruYTsnKgiKQogICAgcHJpbnQoZiLqt7gg7IS46re466i87Yq47J2YIDk1JSDsi6DsmqnqtazqsIQg6ri47J20ID0geyhjaV9o"
        "aSAtIGNpX2xvKSAqIDEwMDouMmZ9ICVwIikKICAgIHByaW50KGYi7LKr64KgIMKxMiDtkZzspIDsmKTssKgg6rWs6rCEIOq4uOyd"
        "tCA9IHsoSElfRDQ1IC0gTE9fRDQ1KSAqIDEwMDouMmZ9ICVwIikgICAjIOuMgOyhsOyaqSDtlokKCiAgICByZWxfc3BhbiA9ICgo"
        "bnAucGVyY2VudGlsZShzZWdfdmFscywgOTcuNSwgYXhpcz0wKSAtIG5wLnBlcmNlbnRpbGUoc2VnX3ZhbHMsIDIuNSwgYXhpcz0w"
        "KSkKICAgICAgICAgICAgICAvIHNlZ192YWxzLm1lYW4oYXhpcz0wKSkgICAgICAgICAgICAgICAgICAgICAgIyDshLjqt7jrqLzt"
        "irjrp4jri6Qg6re4IOyEuOq3uOuovO2KuOydmCDqsJLsnLzroZwg64KY64iV64uI64ukCiAgICBmaWcsIGF4ID0gcGx0LnN1YnBs"
        "b3RzKGZpZ3NpemU9KDgsIDMuNikpCiAgICBheC5zY2F0dGVyKE4yMywgcmVsX3NwYW4sIHM9OCwgY29sb3I9IiMwMEE5RjQiKSAg"
        "ICAgICAgICAgICAgICMg7KCQIO2VmOuCmOqwgCDshLjqt7jrqLztirgg7ZWY64KYCiAgICBheC5heGhsaW5lKDEuMCwgbHM9Ijoi"
        "LCBjb2xvcj0iIzA1MUMyQyIpICAgICAgICAgICAgICAgICAgICAjIOq1rOqwhCDquLjsnbTqsIAg6rCS6rO8IOqwmeyVhOyngOuK"
        "lCDshKAKICAgIGF4LnNldF94c2NhbGUoImxvZyIpCiAgICBheC5zZXRfeGxhYmVsKCLshLjqt7jrqLztirjsnZgg6rOE7JW9IOyI"
        "mCAo6rG0LCDroZzqt7gg64iI6riIKSIpCiAgICBheC5zZXRfeWxhYmVsKCLsg4HrjIAg6rWs6rCEIOq4uOydtCAo6rWs6rCEIOq4"
        "uOydtCDDtyDstpTsoJXqsJIpIikKICAgIHBsdC5zaG93KCkKCu2VtOyEnQoKLSDssqvrgqAg6re4IOyEuOq3uOuovO2KuOydmCDC"
        "sTIg7ZGc7KSA7Jik7LCoIOq1rOqwhOydgCDquLjsnbQgMTIuMjklcCDsmIDsirXri4jri6QuIOqwkiDsnpDssrTqsIAgMC4wNjM1"
        "IOyduOuNsCDqtazqsIQg6ri47J206rCAIDEyJXAg66m0IOyalOycqO2RnOyXkCDsoIHsnYQg7IiYIOyXhuuKlCDshLjqt7jrqLzt"
        "irjsmIDsirXri4jri6QuIOyYpOuKmCDqsJnsnYAg7IS46re466i87Yq46rCAIOyVvSAwLjA1MCDsl5AgWzAuMDMxLCAwLjA3N10g"
        "66GcLCDqtazqsIQg6ri47J20IDQuNTglcCDroZwg7Ken7JWE7KeR64uI64ukLiDtkZzsp5HsnbTrnbwg7Iuk7ZaJ66eI64ukIOyG"
        "jOyImOygkCDslYTrnpggMuyekOumrOqwgCDsobDquIjslKkg64us65287KeR64uI64ukLgotIOq1rOqwhCDquLjsnbTqsIAg7KSE"
        "7Ja065OgIOqwkuydgCDrjIDqsIAg7JeG7J20IOyWu+ydgCDqsoPsnbQg7JWE64uZ64uI64ukLiDri6Trpbgg7IS46re466i87Yq4"
        "7J2YIOygleuztOulvCDtlajqu5gg7IKs7Jqp7ZaI7Jy864uIIOq3uCDshLjqt7jrqLztirjrp4zsnZgg7Yq57J207ISx7J2AIOq3"
        "uOunjO2BvCDsp4Dsm4zsoYzsirXri4jri6QuIOqzhOyVvSDsiJjqsIAg66eO7J2AIOyEuOq3uOuovO2KuOydvOyImOuhnSDri6Tr"
        "pbgg7IS46re466i87Yq47J2YIOygleuztOulvCDsoIHqsowg7IKs7Jqp7ZWY6rOgLCA2M+qxtOyduCDshLjqt7jrqLztirjripQg"
        "66eO7J20IOyCrOyaqe2VqeuLiOuLpC4KLSDrrLQg7ZKA66eBIOqwkuydtCAwIOydtOuNmCA3NTbqsJwg7IS46re466i87Yq464qU"
        "IDAuMDM0IOu2gO2EsCAwLjA0OCDsgqzsnbTsl5Ag7JyE7LmY7ZWp64uI64ukLiAwIOuztOuLpOuKlCDsnITsnbTqs6Ag7JmE7KCE"
        "IO2SgOungSDqsJIgMC4wNTAyIOuztOuLpOuKlCDslYTrnpjsnoXri4jri6QuIOyCrOqzoOulvCDtlZwg6rG064+EIOqyquyngCDs"
        "lYrsnYAg7IS46re466i87Yq4652864+EIOyalOycqOydtCAwIOydvCDsiJjripQg7JeG64uk64qUIOyLpOustOydmCDsg4Hsi53s"
        "nbQsIOuqqO2YlSDslYjsl5DshJwg7J6Q64+Z7Jy866GcIOyngOy8nOynkeuLiOuLpC4KLSDsg4HrjIAg6rWs6rCEIOq4uOydtOuh"
        "nCDrs7TrqbQgMiwzMjPqsJwg7IS46re466i87Yq4IOqwgOyatOuNsCAyLDI0MOqwnCDslYjtjI7snbQgMSDrr7jrp4zsnoXri4jr"
        "i6QuIOyyq+uCoOydmCDCsTIg7ZGc7KSA7Jik7LCoIOq1rOqwhOycvOuhnOuKlCA0MTDqsJwg7IS46re466i87Yq467+Q7J207JeI"
        "7Iq164uI64ukLiDrrLTsgqzqs6AgNzU26rCcIOyEuOq3uOuovO2KuOuKlCDqt7gg6rWs6rCE7Jy866GcIOqzhOyCsO2VmOuptCDq"
        "uLjsnbTqsIAgMCDsnbTrnbwg44CM7Ken64uk44CNIOyqveyXkCDtj6ztlajrkJjripTrjbAg6rCS64+EIDAg7J206528IOyalOyc"
        "qO2RnOyXkCDsoIHsnYQg7IiY6rCAIOyXhuyXiOyKteuLiOuLpC4g6rCZ7J2AIOuNsOydtO2EsOyXkOyEnCDqt7jrjIDroZwg7KCB"
        "7J2EIOyImCDsnojripQg7IS46re466i87Yq46rCAIDXrsLAg64SY6rKMIOymneqwgO2WiOyKteuLiOuLpC4KCuy9lOuTnOqwgCDr"
        "i7XsnYQg66eM65Oc64qUIOuwqeyLnSDigJQg7IS46re466i87Yq466eI64uk7J2YIO2ZleuloOydhCDroZzsp5Mg7Iqk7LyA7J28"
        "7JeQ7IScIOyEuOq3uOuovO2KuCDqsIQg7ZGc7KSA7Y647LCoIHRhdSDtlZjrgpjroZwg66y27Ja0IO2VqOq7mCDstpTsoJXtlZjq"
        "s6AsIOqwkuydtCDslYTri4jrnbwg6re4IOqwkuyXkCDtlajqu5gg7KCB64qUIOq1rOqwhOydmCDquLjsnbTroZwg7IS46re466i8"
        "7Yq466W8IOq1rOu2hO2VqeuLiOuLpC4=",
    "과제":
        "8J+UkyDqs7zsoJwg4oCUIOyxhOygkOyekOqwgCDsnbQg7ZGc7JeQ7IScIOuztOuKlCDqsoMKCuKWoCDsnbQg7L2U65Oc66W8IOu2"
        "gOuluCDtlITroaztlITtirgg7JiICgogICAg7ZWcIOyYiCDigJQg67KU7JyEIOyEoOyWuOydgCDsiqTsvZTtlIQgMiwzMjPqsJwg"
        "7IS46re466i87Yq4IOyghOu2gCwg67Cp67KV7J2AIOqzhOy4tSDrqqjtmJXsnZgg67aA67aEIO2SgOungSDqsJIsIOq4sOykgOyd"
        "gCDsg4HrjIAg6rWs6rCEIOq4uOydtCAxLjAg7Jy866GcIOygle2WiOyWtC4KICAgIOydtOuvuCDrp4zrk6TslrTsoLgg7J6I64qU"
        "IFNFR19OQU1FLCBOMjMsIFJBVzIzLCBQX0hJRVIsIExPX0hJRVIsIEhJX0hJRVIsIFBfUE9PTCDsnYQg6re464yA66GcIOyTsOqz"
        "oCDri6Tsi5wKICAgIOygge2Vqe2VmOyngOuKlCDrp5DslYQg7KSYLiDshLjqt7jrqLztirjrp4jri6Qg7IOB64yAIOq1rOqwhCDq"
        "uLjsnbTrpbwgKEhJX0hJRVIgLSBMT19ISUVSKSAvIFBfSElFUiDroZwg6rWs7ZWcIOuSpCwg6re4IOqwkuydtCDrgrQg6riw7KSA"
        "6rCS67O064ukCiAgICDsnpHsnLzrqbQgIuyekOuPmeyKueyduCIg7JWE64uI66m0ICLsgqzrnozsi6zsgqwiIOuhnCDsoIHsnYAg"
        "7Je07J2EIOy2lOqwgO2VtOyEnCDshLjqt7jrqLztirggwrcg6rOE7JW97IiYIMK3IOustO2SgOungSDCtyDsoIHsnYTqsJIgwrcg"
        "7ZWY7ZWcIMK3CiAgICDsg4HtlZwgwrcg7IOB64yAIOq1rOqwhCDquLjsnbQgwrcg7YyQ7KCVIDjqsJwg7Je07J2EIOqwgOynhCDt"
        "kZzrpbwg66eM65Ok6rOgIOyalOycqO2RnF/shLjqt7jrqLztirjrs4TsmpTsnKhfdjIuY3N2IOuhnCDsoIDsnqXtlbQg7KSYLiDs"
        "oIDsnqXtlaAg65WMCiAgICBpbmRleD1GYWxzZSDroZwg7ZW07IScIO2WiSDrsojtmLjripQg7YyM7J287JeQIOyTsOyngCDrp5Ds"
        "lYQg7KSYLiDsoIDsnqXtlZwg7YyM7J287J2ECiAgICDri6Tsi5wg7J297Ja07IScICLsoIDsnqXtlZwg7ZGc7J2YIOy7rOufvCDs"
        "iJggPSAo7Iir7J6QKSDqsJwiIOyZgCAi7KCE7LK0IOqzhOyVveydmCDsgqzqs6Ag6rK97ZeY66WgID0gKOyIq+yekCkg67mE7Jyo"
        "IiDsnYQg7ZWcIO2WieyUqQogICAgcHJpbnQg7ZW0IOyjvOqzoCwg6rK97ZeY66Wg7J2AIOyGjOyImOygkCDslYTrnpggNuyekOum"
        "rOuhnCDstpzroKXtlbQg7KSYLgogICAg7YyQ7KCVIOyXtOydmCDsnpDrj5nsirnsnbgg7IS46re466i87Yq4IOyImOyZgCDsgqzr"
        "nozsi6zsgqwg7IS46re466i87Yq4IOyImOuPhCDqsIHqsIEg7ZWcIO2WieyUqSBwcmludCDtlbQg7KSYLiDqt7jrnpjtlITripQg"
        "7ZWcIOyepSDigJQg6rCA66Gc7LaV7JeQIOyEuOq3uOuovO2KuOydmCDqs4Tslb0g7IiY66W8IOuhnOq3uCDriIjquIjsnLzroZws"
        "CiAgICDshLjroZzstpXsl5Ag7IS46re466i87Yq466eI64uk7J2YIOyDgeuMgCDqtazqsIQg6ri47J2066W8IOygkOycvOuhnCDs"
        "sI3qs6Ag64K0IOq4sOykgOqwkiDsnITsuZjsl5Ag6rCA66Gc7ISg7J2EIO2VmOuCmCDqt7jsnYAg65KkIOuRkCDstpXsl5Ag6528"
        "67Ko7J2EIOuLrOyVhCDspJguCgrssYTsoJDsnpDqsIAg67O064qUIOqygwoKLSDtkZzsnZgg7Je0IDjqsJzqsIAg6re464yA66Gc"
        "IOyeiOqzoCwg7YyQ7KCVIOyXtOydmCDqsJLsnbQg7J6Q64+Z7Iq57J246rO8IOyCrOuejOyLrOyCrCDrkZjrv5DsnbTrqbAsIOuR"
        "kCDtjJDsoJXsnZgg7IS46re466i87Yq4IOyImOulvCDrjZTtlZwg6rCS7J20IO2RnOydmCDtlokg7IiY7JmAIOqwmeydgOqwgC4g"
        "7KCA7J6l7ZWcIO2MjOydvOydhCDri6Tsi5wg7J297JeI7J2EIOuVjCDtlokg7IiY7JmAIOyXtCDsiJjqsIAg6rCZ7J2A6rCALgot"
        "IOumrO2PrO2KuOydmCDqsrDroaAg7ZWcIOusuOyepeyXkCDqsJLqs7wg64uo7JyE7JmAIOqysOyglSDrj5nsgqzqsIAg7ZWo6ruY"
        "IOyeiOuKlOqwgC4g6re86rGwIOuRkCDtla3rqqkg6rCA7Jq0642wIO2VmOuCmOqwgCBSLWhhdCDqs7wg7Jyg7Zqo7ZGc67O47IiY"
        "7J246rCALgotIOuylOychCDshKDslrggM+qwnCDtla3rqqnsl5Ag44CM7Jmc44CN6rCAIOu2meyWtCDsnojripTqsIAuIOyKpOy9"
        "lO2UhMK367Cp67KVwrfquLDspIAgM+qwnCDrqqjrkZAg64uk66W4IOyEoO2DneydtCDqsIDriqXtlojri6TripQg6rKD7J2EIOyV"
        "jOqzoCDqs6Drpbgg6re86rGw6rCAIOyeiOuKlOqwgC4KCuqysOygleuniOuLpCDsg53quLDripQg7ZWo7KCVCgrCtyDtlokg67KI"
        "7Zi466W8IO2VqOq7mCDsoIDsnqXtlZjripQg6rKDIOKAlCBgdG9fY3N2YCDripQg6riw67O46rCS7J20IO2WiSDrsojtmLjrpbwg"
        "7KCA7J6l7ZWp64uI64ukLiDqt7jrjIDroZwg65GQ66m0IOuLpOyLnCDsnb3snYAg7Je07J20IDnqsJzqsIAg65CY7Ja0IOqwkiAx"
        "67KI7J20IOKdjCDsnoXri4jri6QuCsK3IOuRkCDqsIDsp4Ag6rCS7J2EIO2VqOq7mCDsoIHripQg6rKDIOKAlCDtlZwg7ZGc7JeQ"
        "IOu5hOycqOqzvCDqsbQv64WE7J2EIO2VqOq7mCDsoIHsnLzrqbQg6rCZ7J2AIOyXtOydtCAy67Cw66GcIOydve2emeuLiOuLpC4g"
        "7JqU7Jyo7ZGc7J2YIO2VnCDsl7Tsl5DripQg7ZWcIOqwgOyngCDqsJLrp4wg7KCB7Iq164uI64ukLgrCtyDsiJjroLTsnYQg7YyQ"
        "7KCV7ZWY7KeAIOyViuuKlCDqsoMg4oCUIO2RnOyXkCDsoIHsnYAg6rCS7J20IFItaGF0IOqzvCDsnKDtmqjtkZzrs7jsiJjsnZgg"
        "6riw7KSA6rCS7J2EIO2GteqzvO2VnCDtkZzrs7jsl5DshJwg64KY7JioIOqyg+ydtCDslYTri4jrqbQsIOq3uCDsiJjripQg7LK0"
        "7J247J20IOyVhOyngSDsiJjroLTtlZjsp4Ag7JWK7J2AIOyDge2DnOyXkOyEnCDrgpjsmKgg6rCS7J28IOu/kOyeheuLiOuLpC4K"
        "wrcg66y07IKs6rOgIOyEuOq3uOuovO2KuOulvCDqt7jrjIDroZwg65GQ64qUIOqygyDigJQg7IKs6rOg6rCAIO2VnCDqsbTrj4Qg"
        "7JeG642YIOyEuOq3uOuovO2KuOydmCDqsJLsnYAg67Cp67KV7JeQIOuUsOudvCAwLjAyMDQg67aA7YSwIDAuMDQ4MCDquYzsp4Ag"
        "64us65287KeR64uI64ukLiDslrTripAg67Cp67KV7J2EIOqzqOuekOuKlOyngCDrsJ3tnojsp4Ag7JWK7Jy866m0IOq3uCDssKjs"
        "nbTqsIAg7Ya17Ke466GcIOqwkOy2sOynkeuLiOuLpC4Kwrcg7IKs7KCE67aE7Y+s7J2YIOy2nOyymOulvCDsoIHsp4Ag7JWK64qU"
        "IOqygyDigJQg7JW9IDMwMOqxtOydgCDsgqzrnozsnbQg7KeA7KCV7ZWcIOyImOqwgCDslYTri4jrnbwg642w7J207YSw66GcIOy2"
        "lOygle2VnCDsiJjsnoXri4jri6QuIOuLpOydjCDrtoTquLDsl5Ag64uk7IucIOyLpO2Wie2VmOuptCDri6zrnbzsp4Tri6TripQg"
        "66y47J6l7J20IOumrO2PrO2KuOyXkCDsnojslrTslbwg7ZWp64uI64ukLgoK7L2U65Oc6rCAIOuLteydhCDrp4zrk5zripQg67Cp"
        "7IudIOKAlCDqsJIg7ZWY64KY6rCAIOyVhOuLiOudvCDqsJLCt+q1rOqwhMK37IOB64yAIOq1rOqwhCDquLjsnbTCt+2MkOygleyd"
        "hCDtlZwg7ZaJ7JeQIOustuyWtCDshLjqt7jrqLztirjrp4jri6Qg7KCB6rOgLCDsgqzrnozsnbQg67O8IOyEuOq3uOuovO2KuOyZ"
        "gCDqt7jrjIDroZwg64KY6rCIIOyEuOq3uOuovO2KuOulvCDquLDspIDqsJIg7ZWY64KY66GcIOq1rOu2hO2VqeuLiOuLpC4=",
}
#  힌트는 (한 문장 힌트, 튜터에게 물을 것) 짝입니다 — 채점이 ❌ 인 항목에만 나옵니다
#  가운데가 하나 더 있는 항목은 2단 계단이라, 두 번째 ❌ 에서 다음 단이 나옵니다
# 항목별 힌트 — base64(JSON)
_HINT = _dec_json(
    "eyJtMSI6IHsi64W47LacIDYuNjLrhYQg7IS46re466i87Yq47J2YIOu2gOu2hCDtkoDrp4Eg6rCSICjqsbQv64WEKSI6IFsi66qo"
    "7ZiV7J2AIOydtOuhoOydmCDjgIwyLiDrtoDrtoQg7ZKA66eB6rO8IOqzhOy4tSDrqqjtmJXjgI3snbQg66eM65OgIDPsuLUg6re4"
    "64yA66Gc7J6F64uI64ukIOKAlCDrp6gg7JyE7JeQIOuqqOuToCDshLjqt7jrqLztirjsl5Ag6rO17Ya17J24IO2Pieq3oCDtlZjr"
    "gpgsIOqwgOyatOuNsOyXkCDshLjqt7jrqLztirjrp4jri6TsnZgg7Y647LCo66W8IOyEuOq3uOuovO2KuCDqsIQg7ZGc7KSA7Y64"
    "7LCoIHRhdSDsl5Ag7ZGc7KSA7KCV6recIO2OuOywqOulvCDqs7HtlZwg7ZiV7YOc66GcLCDrp6gg7JWE656Y7JeQIO2PrOyVhOyG"
    "oS4g6re4IOyEuOq3uOuovO2KuOydmCDsgqztm4Ttj4nqt6DsnYQg64uo7JyEIOqxtC/rhYTqs7wg7ZWo6ruYIOyGjOyImOygkCDs"
    "lYTrnpggNOyekOumrOuhnCDstpzroKXtlZjrnbzqs6Ag7KCB7Jy87IS47JqULiDsnbTrpoTsnYAg44CM64W47LacIDYuNjLrhYQg"
    "7IS46re466i87Yq47J2YIOu2gOu2hCDtkoDrp4Eg6rCS44CNIOq3uOuMgOuhnCDsk7Drnbzqs6Ag7ZWo6ruYIOyggeycvOyEuOya"
    "lCIsICLshLjqt7jrqLztirgg7ZWY64KY7J2YIOyCrO2bhO2Pieq3oOydhCDsgqztm4Qg7ZGc67O47JeQ7IScIOyWtOuWu+qyjCDq"
    "s4TsgrDtlZjripTsp4AiXSwgIuq1rOqwhOydtCDsmYTsoIQg7ZKA66eBIOqwkuydhCDtj6ztlajtlZwg7IS46re466i87Yq4IOyI"
    "mCAo6rCcKSI6IFsi6rSA7Lih7J20IOyggeydgCAxMuqwnCDshLjqt7jrqLztirjrp4jri6Qg7IKs7ZuEIO2RnOuzuOydmCA1JSDr"
    "toTsnITsiJjsmYAgOTUlIOu2hOychOyImOulvCDqtaztlZwg65KkLCDqt7gg6rWs6rCE7J20IOyZhOyghCDtkoDrp4Eg6rCS7J2E"
    "IO2PrO2VqO2VmOuKlCDshLjqt7jrqLztirjqsIAg66qHIOqwnOyduOyngCDsp5Hqs4TtlbQg64uo7JyEIOqwnOyZgCDtlajqu5gg"
    "7Lac66Cl7ZWY65286rOgIOyggeycvOyEuOyalC4g66y0IO2SgOungSDqsJLqs7wg7JmE7KCEIO2SgOungSDqsJLsnYAg6rCB6rCB"
    "IOuLpOuluCDtlonsl5Ag7Lac66Cl7ZWp64uI64ukIiwgIuq1rOqwhOydtCDslrTrlqQg6rCS7J2EIO2PrO2VqO2VmOuKlOyngCDt"
    "jJDsoJXtlZjripQg7KGw6rG0Il0sICJwbG90IjogWyLqtIDsuKHsnbQg7KCB7J2AIDEy6rCcIOyEuOq3uOuovO2KuOydmCA5MCUg"
    "7Iug7Jqp6rWs6rCE7J2EIOqwgOuhnCDrp4nrjIAgMTLqsJzroZwsIOqwmeydgCDshLjqt7jrqLztirjsnZgg66y0IO2SgOungSDq"
    "sJLsnYQg7KCQIDEy6rCc66GcLCDsmYTsoIQg7ZKA66eBIOqwkuydhCDshLjroZzshKAg7ZWY64KY66GcIOKAlCDrkZAg7LaVIOud"
    "vOuyqOq5jOyngCA06rCc66W8IFBST01QVCDsl5Ag64ukIOyggeyXiOuKlOyngCDtmZXsnbjtlZjshLjsmpQiLCAi6re466a87JeQ"
    "7IScIOyWtOuKkCDsmpTqtazqsIAg67mg7KGM64qU7KeAIl0sICJlcnJvciI6IFsiQ0xBSU1TIMK3IEVYUE8gwrcgSiDripQg7J20"
    "66+4IOunjOuTpOyWtOyguCDsnojsirXri4jri6Qg4oCUIO2RnOulvCDri6Tsi5wg66eM65Ok7KeAIOunkOqzoCDqt7gg7J2066aE"
    "7J2EIOq3uOuMgOuhnCDsk7Drnbzqs6Ag7KCB7Jy87IS47JqULiDsgqztm4Qg7ZGc67O47J2EIOyEuOq3uOuovO2KuCDstpXsnLzr"
    "oZwg7KCE6rCc7ZWgIOuVjCDrqqjslpHsnbQg7Ja06riL64KY66m0IOyXrOq4sOyXkOyEnCDqsbjrpr3ri4jri6QiLCAi7J20IOyY"
    "pOulmOqwgCDslrTripAg7L2U65OcIO2WieyXkOyEnCDsmZwg64Ks64qU7KeAIl19LCAibTIiOiB7IuqzoOy5nCDrkqQg6rCA7J6l"
    "IO2BsCBSLWhhdCI6IFsi7Lac66Cl65CcIOuRkCDsiJjrpbwg7J2066Gg7JeQ7IScIOygle2VnCDquLDspIDqsJIgMuqwnOyZgCDr"
    "uYTqtZDtlbQg67O07IS47JqUIiwgInBtLnNhbXBsZSDsl5Ag7KCE64us7ZWcIOyduOyekCDqsIDsmrTrjbAg7ZGc7KeR6riw66W8"
    "IOyngOygle2VnCBzdGVwIOydtCDsnojsirXri4jri6Qg4oCUIOq3uCDsvZTrk5wg7ZaJ7J2EIOyCreygnO2VmOqzoCDrsJjrs7Ug"
    "NSwwMDDtmowgwrcg7JuM67CN7JeFIDUsMDAw7ZqM66GcIOuKmOugpCDrs7TshLjsmpQiLCAi7Ja064qQIOyduOyekOqwgCDssrTs"
    "nbjsnZgg67CY67O1IO2an+yImOulvCDsoJXtlZjripTsp4AiXSwgIuqzoOy5nCDrkqQg6rCA7J6lIOyekeydgCDsnKDtmqjtkZzr"
    "s7jsiJggKOuyjCkiOiBbIuycoO2aqO2RnOuzuOyImOuKlCDssrTsnbjsnbQg6rCZ7J2AIOyDge2DnOyXkCDrgqjsnLzrqbQg67CY"
    "67O1IO2an+yImOuztOuLpCDtm6jslKwg7J6R7JWE7KeR64uI64ukIiwgIu2RnOynkeq4sOulvCDrsJTqvrjqs6Ag67CY67O1IOyI"
    "mOulvCA1LDAwMO2ajOuhnCDripjrprAg65KkIGFydml6IOuhnCDri6Tsi5wg6rOE7IKw7ZW0LCDrqqjsiJgg7KCE7LK07J2YIOy1"
    "nOyGn+qwkuydhCDri6jsnIQg67KM6rO8IO2VqOq7mCDstpzroKXtlZjrnbzqs6Ag7KCB7Jy87IS47JqUIiwgIuuwmOuztSDtmp/s"
    "iJjsmYAg7Jyg7Zqo7ZGc67O47IiY6rCAIOyZnCDri6Trpbgg7IiY7J247KeAIl0sICJwbG90IjogWyLqs6DsuZwg7Iuk7ZaJ7J2Y"
    "IOyEuOq3uOuovO2KuCDtjrjssKggdGhldGEg7J2YIFItaGF0IDI0MuqwnOulvCDsoJDsnLzroZwg7ZGc7Iuc7ZWY6rOgLCDquLDs"
    "pIDqsJIgMS4wMSDsnITsuZjsl5Ag6rCA66Gc7ISg7J2EIO2VmOuCmCDquIvqs6AsIOuRkCDstpXsl5Ag652867Ko7J2EIOuLteuL"
    "iOuLpCIsICLsgrDsoJDrj4Tsl5DshJwg7Ja064qQIOyalOq1rOqwgCDruaDsoYzripTsp4AiXSwgImVycm9yIjogWyJDTEFJTVMg"
    "wrcgRVhQTyDCtyBKIOuKlCDsnbTrr7gg66eM65Ok7Ja07KC4IOyeiOyKteuLiOuLpCDigJQg7ZGc66W8IOuLpOyLnCDrp4zrk6Ts"
    "p4Ag66eQ6rOgIOq3uOuMgOuhnCDsk7Drnbzqs6Ag7KCB7Jy87IS47JqUIiwgIuydtCDsmKTrpZjqsIAg7Ja064qQIOy9lOuTnCDt"
    "lonsl5DshJwg7JmcIOuCrOuKlOyngCJdfSwgIm0zIjogeyLqs4Tslb0gNjPqsbQg7IS46re466i87Yq47J2YIOu2gOu2hCDtkoDr"
    "p4Eg6rCSICjruYTsnKgpIjogWyLrr7jshZggMSDsnZggM+y4teyXkOyEnCDrp6gg7JWE656Y6rCAIOydtO2VreycvOuhnCDrsJTr"
    "gIzqs6Ag66eoIOychCDsuLXsnZgg7KSR7Ius64+EIOuhnOynkyDsiqTsvIDsnbzsnZggLTIuOTQg66GcIOuwlOuAneuLiOuLpCDi"
    "gJQg7J207ZWt7J2YIOyLnO2WiSDsiJgg7J247J6Q7JeQIOq3uCDshLjqt7jrqLztirjsnZgg6rOE7JW9IOyImOulvCwg6rSA7Lih"
    "IOyduOyekOyXkCDsgqzqs6Drpbwg6rKq7J2AIOqzhOyVvSDsiJjrpbwg7KeA7KCV7ZWY65286rOgIOyggeycvOyEuOyalC4g7IKs"
    "7ZuEIO2RnOuzuOydgCDroZzsp5Mg7Iqk7LyA7J287JeQIOyeiOycvOuvgOuhnCDtmZXrpaDroZwg65CY64+M66awIOuSpCDqt7gg"
    "7IS46re466i87Yq47J2YIO2Pieq3oOydhCDri6jsnIQg67mE7Jyo6rO8IO2VqOq7mCDshozsiJjsoJAg7JWE656YIDTsnpDrpqzr"
    "oZwg7Lac66Cl7ZWY652864qUIOqyg+q5jOyngCDsoIHsirXri4jri6QiLCAi6re4IOyEuOq3uOuovO2KuOydmCDsgqztm4Ttj4nq"
    "t6DsnYQg7Ja065akIOyKpOy8gOydvOyXkOyEnCDsgrDstpztlbTslbwg7ZWY64qU7KeAIl0sICLqt7gg7IS46re466i87Yq47J2Y"
    "IDk1JSDsi6DsmqnqtazqsIQg6ri47J20ICglcCkiOiBbIuqwmeydgCDtkZzrs7jsl5DshJwg6re4IOyEuOq3uOuovO2KuOydmCAy"
    "LjUlIOu2hOychOyImOyZgCA5Ny41JSDrtoTsnITsiJjrpbwg6rWs7ZW0IOywqOulvCDqs4TsgrDtlZjqs6AsIOu5hOycqOydtCDs"
    "lYTri4jrnbwg7Y287IS87Yq47Y+s7J247Yq466GcIOyGjOyImOygkCDslYTrnpggMuyekOumrOq5jOyngCDstpzroKXtlZjrnbzq"
    "s6Ag7KCB7Jy87IS47JqUIiwgIuq1rOqwhOydmCDquLjsnbTrpbwg7Ja065akIOuLqOychOuhnCDsoIHslrTslbwg7ZWY64qU7KeA"
    "Il0sICJwbG90IjogWyLshLjqt7jrqLztirjrpbwg7KCQ7Jy866GcIO2VmOuCmOyUqSDtkZzsi5ztlZjqs6AsIOqwgOuhnOy2leyd"
    "gCDqs4Tslb0g7IiYKOuhnOq3uCDriIjquIgpLCDshLjroZzstpXsnYAg7IOB64yAIOq1rOqwhCDquLjsnbQo6re4IOyEuOq3uOuo"
    "vO2KuOydmCDqtazqsIQg6ri47J2066W8IOq3uCDshLjqt7jrqLztirjsnZgg6rCS7Jy866GcIOuCmOuIiCDqsJIp7J6F64uI64uk"
    "IOKAlCAyLDMyM+qwnOulvCDqsJnsnYAg7IiYIO2VmOuCmOuhnCDrgpjriITsp4Ag7JWK7Iq164uI64ukLiDqsIDroZzshKAg7ZWY"
    "64KY7J2YIOychOy5mOyZgCDrkZAg7LaVIOydtOumhOq5jOyngCBQUk9NUFQg7JeQIOuLpCDsoIHsl4jripTsp4Ag7ZmV7J247ZWY"
    "7IS47JqUIiwgIuyCsOygkOuPhOyXkCDrrLTsl4fsnYQg6rCA66Gc7ISg7Jy866GcIOuwsOy5mO2VtOyVvCDtlZjripTsp4AiXSwg"
    "ImVycm9yIjogWyJOMjMgwrcgSzIzIMK3IFJBVzIzIMK3IFBfUE9PTCDCtyBKMjMgwrcgSTYzIOuKlCDsnbTrr7gg66eM65Ok7Ja0"
    "7KC4IOyeiOyKteuLiOuLpCDigJQg64uk7IucIOunjOuTpOyngCDrp5Dqs6Ag6re4IOydtOumhOydhCDqt7jrjIDroZwg7JOw6528"
    "6rOgIOyggeycvOyEuOyalC4g7IKs7ZuEIO2RnOuzuOydhCDshLjqt7jrqLztirgg7LaV7Jy866GcIOyghOqwnO2VoCDrlYwg66qo"
    "7JaR7J20IOyWtOq4i+uCmOuptCDsl6zquLDsl5DshJwg6rG466a964uI64ukIiwgIuydtCDsmKTrpZjqsIAg7Ja064qQIOy9lOuT"
    "nCDtlonsl5DshJwg7JmcIOuCrOuKlOyngCJdfSwgIuqzvOygnCI6IHsi7KCA7J6l7ZWcIO2RnOydmCDsu6zrn7wg7IiYICjqsJwp"
    "IjogWyLri6Tsi5wg7J297J2AIOyXtOydtCA56rCc66m0IO2WiSDrsojtmLjqsIAg7ZWo6ruYIOyggOyepeuQnCDqsoPsnoXri4jr"
    "i6Qg4oCUIHRvX2NzdiDsl5AgaW5kZXg9RmFsc2Ug66W8IOyngOygle2VmOudvOqzoCDsoIHsnLzshLjsmpQuIOyXtCDsnbTrpoTs"
    "nYAg44CM66eM65OkIOqyg+OAjeyXkCDsoIHtnowgOOqwnOyeheuLiOuLpCIsICLtkZzsnZgg7Je07J20IOustOyXhyDrrLTsl4fs"
    "nbTslrTslbwg7ZWY64qU7KeAIl0sICLsoITssrQg6rOE7JW97J2YIOyCrOqzoCDqsr3tl5jrpaAgKOu5hOycqCkiOiBbIuyEuOq3"
    "uOuovO2KuOuhnCDrgpjriITquLAg7KCELCDqs4Tslb0g7KCE7LK07JeQ7IScIOyCrOqzoOulvCDqsqrsnYAg6rOE7JW97J2YIOu5"
    "hOycqOyeheuLiOuLpCDigJQg7IOB7ZWcIOyymOumrCDrkZAg6rec7LmZ7J2EIO2VqOq7mCDsoIHsmqntlojripTsp4Ag7J20IOqw"
    "kiDtlZjrgpjroZwg7ZmV7J247ZWp64uI64ukIiwgIuydtCDqsJLsnZgg67aE7J6Q7JmAIOu2hOuqqOqwgCDrrLTsl4fsnbjsp4Ai"
    "XSwgInBsb3QiOiBbIuyEuOq3uOuovO2KuOuniOuLpOydmCDsg4HrjIAg6rWs6rCEIOq4uOydtOulvCDqt7jrprzsnLzroZwg67Cw"
    "7LmY7ZWY6rOgIOuCtOqwgCDqs6Drpbgg6riw7KSA6rCS7J2EIOqwgOuhnOyEoOycvOuhnCDtlZjrgpgg7LaU6rCA7ZWcIOuSpCwg"
    "65GQIOy2leyXkCDrnbzrsqjsnYQg64u164uI64ukIiwgIuq3uOumvOyXkOyEnCDslrTripAg7JqU6rWs6rCAIOu5oOyhjOuKlOyn"
    "gCJdLCAiZXJyb3IiOiBbIuuvuOyFmCAzIOydmCDsgqztm4Qg7ZGc67O46rO8IOykgOu5hCDshYDsnZgg6rCS7J2AIOydtOuvuCDr"
    "p4zrk6TslrTsoLgg7J6I7Iq164uI64ukIOKAlCDri6Tsi5wg7KCB7ZWp7ZWY7KeAIOunkOqzoCDqt7jrjIDroZwg7JOw65286rOg"
    "IOyggeycvOyEuOyalCIsICLsnbQg7Jik66WY6rCAIOyWtOuKkCDsvZTrk5wg7ZaJ7JeQ7IScIOyZnCDrgqzripTsp4AiXX19")
_ASK = {"m1": "full", "m2": "blank",       # 뒤로 갈수록 걷힙니다 — 미션 3 은 뼈대만 남습니다
        "m3": "self", "과제": "bare"}
_ASK_HEAD = {"full": "◼ 튜터에 그대로 보낼 질문 (코드는 고치지 말라고 명시합니다)",
             "blank": "◼ 튜터에 보낼 질문 (빈 곳은 내가 채웁니다)",
             "bare": "◼ 튜터에 보낼 질문의 뼈대 (문장으로 만드는 것은 내 몫입니다)",
             "self": "◼ 스스로 확인할 항목의 뼈대 (이 미션은 힌트 없이 내가 명세합니다)"}
_ASK_FORM = {"full": "내 PROMPT 는 「{prompt}」 였는데 「{name}」 채점이 ❌ 였습니다."
                     " AI 코드 표시선 아래 코드를 보고 「{ask}」 에 3~5문장으로 답해 주세요. 코드는 고치지 마세요.",
             "blank": "내 PROMPT 는 ____ 였는데 「{name}」 채점이 ❌ 였습니다. AI 코드 표시선 아래 코드에서"
                      " 「{ask}」 부터 짚고, 어느 코드 행을 무엇으로 고칠지 한 문장으로 답해 주세요.",
             "bare": "무엇을 : {ask} / 어디를 : {name} / 고치지 말 것 : 코드",
             "self": "무엇을 : {ask} / 어디를 : {name} / 다시 볼 곳 : AI 코드 표시선 아래 코드"}

_FIGS = []                                 # 「PROMPT 셀」이 만든 그림을 모으는 자리
_STATE = {"tag": None, "buf": None, "out": None, "ids0": set()}
_TRY = {}                                  # 미션마다 ❌ 가 몇 번째인지 — 2단 힌트가 이 수를 봅니다
_orig_figure = getattr(plt.figure, "_orig", plt.figure)   # 이 셀을 다시 실행해도 겹겹이 싸이지 않게


def _figure(*a, **k):
    f = _orig_figure(*a, **k)
    _FIGS.append(f)
    return f


_figure._orig = _orig_figure
plt.figure = _figure                       # subplots · pandas · seaborn 도 이 경로를 거칩니다


def _get(name):
    """이름이 살아 있으면 그 값, 없으면 None — 아직 만들지 않은 이름에도 채점이 죽지 않게"""
    return globals().get(name)


def _open_figs():
    """지금 열려 있는 그림들 — 래퍼를 거치지 않고 읽습니다"""
    try:
        return [plt._pylab_helpers.Gcf.figs[n].canvas.figure for n in plt.get_fignums()]
    except Exception:
        return []


def _nums(text):
    """출력 텍스트에서 숫자만 뽑습니다 (천 단위 쉼표 제거, 지수 표기 포함)"""
    picked = []
    for seg in re.findall(r"-?\d[\d,]*\.?\d*(?:[eE][-+]?\d+)?", text or ""):
        try:
            picked.append(float(seg.replace(",", "")))
        except ValueError:
            continue
    return picked


def _value_lines(txt, units):
    """값 채점이 볼 행만 고릅니다 — 단위가 붙은 행, 또는 「이름 = 값」 꼴의 행.

    표를 통째로 띄우면 그 안에 정답이 우연히 섞여 있어, 「값을 골라 단위와 함께 출력한다」 는
    계약을 지키지 않고도 ✅ 가 열립니다. 그래서 단위를 달아 고른 행이 하나라도 있으면 그
    행들만 보고, 그런 행이 아예 없을 때만 출력 전체를 봅니다.
    """
    picked = []
    for line in (txt or "").splitlines():
        if not line.strip():
            continue
        if "=" in line or any(unit and unit in line for unit in units):
            picked.append(line)
    return picked


def _label_lines(val_lines, label):
    """라벨이 든 값줄만 골라, 그 줄에서 라벨 뒤(「=」·「:」 뒤) 토막을 돌려줍니다.

    값줄의 숫자를 한 주머니에 담아 띠와 대조하면 **옆 칸의 값**이 이 칸을 헛통과시킵니다 —
    부호 칸 둘이 같은 띠일 때 한쪽에 찍힌 1 이 다른 칸까지 열어 주던 자리입니다. 그래서
    라벨이 든 줄이 있으면 그 줄의 라벨 뒤만 봅니다(없으면 부르는 쪽이 값줄 전체로 물러납니다).
    라벨 찾기는 정확히 같은 문자열 → 못 찾으면 공백·괄호를 걷어낸 꼴, 두 단입니다.
    """
    def _tighten(txt):                     # 공백과 괄호(안의 말까지) 를 걷어낸 글자 · 원래 자리
        keep, depth = [], 0
        for i, ch in enumerate(txt or ""):
            if ch in "([":
                depth += 1
            elif ch in ")]":
                depth = max(0, depth - 1)
            elif depth == 0 and not ch.isspace():
                keep.append(i)
        return "".join((txt or "")[i] for i in keep), keep

    tight_label = _tighten(label)[0]
    tails = []
    for line in val_lines:
        end = line.find(label) + len(label) if label and label in line else -1
        if end < 0 and tight_label:                       # 2단 — 공백·괄호를 걷어내고 다시 봅니다
            tight_line, keep = _tighten(line)
            at = tight_line.find(tight_label)
            end = (keep[at + len(tight_label) - 1] + 1) if at >= 0 else -1
        if end < 0:
            continue
        tail = line[end:]
        cut_at = [p for p in (tail.find("="), tail.find(":")) if p >= 0]
        tails.append(tail[min(cut_at) + 1:] if cut_at else tail)
    return tails


def _axes(fig):
    """Figure · Axes 어느 쪽을 받아도 축 목록으로 폅니다"""
    if fig is None:
        return []
    axs = getattr(fig, "axes", None)
    if isinstance(axs, list):
        return axs
    return [fig] if hasattr(fig, "get_xlabel") else []


def _seg_lines(ax):
    """`vlines` · `hlines` 로 그은 선 — Line2D 가 아니라 **LineCollection** 이라 `ax.lines` 에 없습니다.

    토막 하나를 `(가로값, 세로값)` 한 쌍으로 펴서 돌려줍니다. 이 길을 열어 두지 않으면
    `plt.vlines(...)` 로 화면에 분명히 그어 놓은 수직선이 「수직선이 아예 없다」 로 채점돼,
    학생이 고칠 곳 없는 결함을 고치러 가게 됩니다.
    """
    picked = []
    for c in getattr(ax, "collections", []):
        get_segs = getattr(c, "get_segments", None)      # LineCollection 에만 있는 메서드입니다
        if get_segs is None:                              # 산점도(PathCollection)·채움 영역(PolyCollection)은 선이 아닙니다
            continue
        try:
            segs = get_segs()
        except Exception:
            continue
        for seg in segs:
            xy = np.asarray(seg, dtype=float).reshape(-1, 2)
            if xy.shape[0] >= 2:
                picked.append((xy[:, 0], xy[:, 1]))
    return picked


def _lines(ax):
    """축의 선을 셋으로 갈라 돌려줍니다 — 곡선(점 10개 이상) · 가로 기준선 · 세로 기준선.

    세로축 값이 처음부터 끝까지 같은 선은 점이 몇 개든 **가로 기준선** 으로 셉니다. `axhline` 으로 긋든
    같은 값을 열 번 늘어놓아 긋든 그림이 말하는 것은 「상수 하나」 로 같은데, 뒤엣것을
    곡선으로 집계하면 곡선 개수 기준이 그림 그리는 방법 차이로 달라진다.
    """
    cur, hline, vline = [], [], []
    cands = []
    for ln in ax.lines:                     # 선 하나가 깨져도 나머지는 살린다 — `axvline(x=날짜)` 처럼 x 가
        try:                                #  Timestamp 인 세로선은 날짜를 숫자로 바꿔 받는다
            xd = np.asarray(ln.get_xdata(), dtype=float).reshape(-1)
            yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
        except (TypeError, ValueError):
            try:
                import matplotlib.dates as _mdates
                xd = np.asarray(_mdates.date2num(ln.get_xdata()), dtype=float).reshape(-1)
                yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
            except Exception:
                continue
        cands.append((xd, yd))
    cands += _seg_lines(ax)                     # `plt.hlines` · `plt.vlines` 로 그은 선도 여기서 합류합니다
    for xd, yd in cands:
        if xd.size >= 2 and yd.size == xd.size and np.allclose(yd, yd[0]):
            hline.append(float(yd[0]))
        elif xd.size >= 10:
            cur.append((xd, yd))
        elif xd.size == 2 and np.isclose(xd[0], xd[1]):
            vline.append(float(xd[0]))
    return cur, hline, vline


def _marker_lines(ax):
    """마커를 찍은 Line2D — `plot(x, y, "o")` 든 `plot(x, y, marker="o")` 든 점으로 셉니다.

    선으로 이으면서 점도 함께 찍은 선(`linestyle` 이 기본 실선)까지 세는 것은, 점추정을 그렇게
    찍은 그림이 「점추정 마커가 0개다」 로 ❌ 나던 자리이기 때문입니다. 점 계수는 linestyle 과
    무관하고, 마커를 달지 않은 선만 점에서 뺍니다.
    """
    picked = []
    for ln in ax.lines:
        marker = str(ln.get_marker()).strip()
        if marker in ("", "None", "none"):     # 마커 없는 선은 점이 아닙니다 — 선 모양은 보지 않습니다
            continue
        if marker in ("|", "_"):               # errorbar 의 capsize 캡은 점이 아닙니다 — 구간 끝을 그은 짧은 선입니다
            continue
        try:                                # 날짜 x 인 선 하나가 나머지 점까지 죽이지 않게 갈라 받는다
            xd = np.asarray(ln.get_xdata(), dtype=float).reshape(-1)
            yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
        except (TypeError, ValueError):
            try:
                import matplotlib.dates as _mdates
                xd = np.asarray(_mdates.date2num(ln.get_xdata()), dtype=float).reshape(-1)
                yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
            except Exception:
                continue
        if xd.size and xd.size == yd.size:
            picked.append((ln, xd, yd))
    return picked


def _diagonal(ax, skip=(), xy=None):
    """y = x 인 기준선이 점이 놓인 범위를 가로지르는가 — 선이 있기만 하면 통과가 아닙니다"""
    drop_ids = {id(ln) for ln in skip}
    cands = [(np.asarray(ln.get_xdata(), dtype=float).reshape(-1),
           np.asarray(ln.get_ydata(), dtype=float).reshape(-1))
          for ln in ax.lines if id(ln) not in drop_ids]
    cands += _seg_lines(ax)                     # LineCollection 으로 그은 y = x 기준선도 셉니다
    for xd, yd in cands:
        if xd.size < 2 or xd.size != yd.size:
            continue
        if not (np.isfinite(xd).all() and np.isfinite(yd).all()):
            continue
        width = float(xd.max() - xd.min())
        if width <= 0:                                    # 수직선은 대각선이 아닙니다
            continue
        if not np.allclose(yd, xd, rtol=1e-3, atol=0.02 * width):
            continue                                   # 세로가 가로를 따라가지 않으면 넘어갑니다
        if xy is not None and getattr(xy, "size", 0):
            pt_span = float(xy[:, 0].max() - xy[:, 0].min())
            overlap = float(min(xd.max(), xy[:, 0].max()) - max(xd.min(), xy[:, 0].min()))
            if pt_span > 0 and overlap < 0.4 * pt_span:          # 점이 놓인 구간을 거의 덮지 못하는 토막 선
                continue
        return True
    return False


def _near(val, ref, factor=1.5):
    """실측값이 기준 기준의 배수 안에 드는가 — 눈금·표본 차이는 넘어갑니다"""
    try:
        val = float(val)
    except (TypeError, ValueError):
        return False
    if ref == 0:
        return abs(val) < 1e-9
    return val * ref > 0 and abs(ref) / factor <= abs(val) <= abs(ref) * factor


def _pt_sets(ax):
    """축에 찍힌 점 — 산점도(PathCollection)와 마커를 단 선을 한자리에서 모읍니다"""
    marker = _marker_lines(ax)
    pt_sets = [np.asarray(c.get_offsets(), dtype=float).reshape(-1, 2)
          for c in ax.collections if not hasattr(c, "get_segments")]   # 선 컬렉션(LineCollection)은 점으로 세지 않습니다
    pt_sets += [np.column_stack([xd, yd]) for _, xd, yd in marker]
    return marker, pt_sets


def _spans(ax):
    """가로로 누운 구간 막대 — 세그먼트마다의 구간을 그린 토막을 `(왼끝, 오른끝)` 으로 셉니다.

    `hlines`(LineCollection) · `plot([lo, hi], [y, y])`(Line2D) · `errorbar(xerr=…)` ·
    `barh`(Rectangle) 어느 길로 그렸어도 같은 자리에서 셉니다 — 그리는 방법을 하나로
    못 박으면 AI 가 고른 다른 길이 「막대가 없다」 로 읽혀 학생이 애먼 곳을 고칩니다.
    세로로 선 토막(세로 기준선)은 구간이 아니라 여기서 빠집니다.
    """
    picked = []
    cands = [(np.asarray(ln.get_xdata(), dtype=float).reshape(-1),
           np.asarray(ln.get_ydata(), dtype=float).reshape(-1)) for ln in ax.lines]
    cands += _seg_lines(ax)
    for xd, yd in cands:
        if xd.size != 2 or yd.size != 2:
            continue
        if not (np.isfinite(xd).all() and np.isfinite(yd).all()):
            continue
        if not np.isclose(yd[0], yd[1]) or float(xd.max() - xd.min()) <= 0:
            continue                                   # 가로로 누워 있고 길이가 있는 토막만
        picked.append((float(xd.min()), float(xd.max())))
    for p in getattr(ax, "patches", []):               # `barh(left=lo, width=hi-lo)` 로 그린 구간
        try:
            x0, bar_w, heights = float(p.get_x()), float(p.get_width()), float(p.get_height())
        except (AttributeError, TypeError, ValueError):
            continue
        if bar_w > 0 and heights > 0 and np.isfinite(x0):
            picked.append((x0, x0 + bar_w))
    return picked


def _plot_gaps(ax, spec):
    """축 하나를 기준과 대조해 어긋난 곳을 문장 목록으로 돌려줍니다 (빈 목록이면 통과)"""
    gaps = []
    if spec.get("kind") == "line":
        cur, hline, vline = _lines(ax)
        want = spec.get("n_series", len(cur))
        if len(cur) != want:
            gaps.append(f"곡선이 {len(cur)}개다 (기준 {want}개 —"
                        " 상수 하나인 가로선은 곡선으로 세지 않습니다)")
        if cur:
            xd, yd = max(cur, key=lambda t: t[0].size)
            lo, hi = spec.get("x_range", (xd.min(), xd.max()))
            slack = 0.2 * (hi - lo)
            if not (xd.min() <= lo + slack and xd.max() >= hi - slack):
                gaps.append(f"가로축이 {xd.min():.3g}~{xd.max():.3g} 만 포함한다 (기준 {lo:g}~{hi:g})")
            if "peak_x" in spec:
                p0, p1 = spec["peak_x"]
                peak = float(xd[yd.argmax()])         # 로그우도면 봉우리가 그 자리에 섭니다
                trough = float(xd[yd.argmin()])         # 부호를 뒤집어 그렸으면 골짜기가 그 자리입니다
                if not (p0 <= peak <= p1 or p0 <= trough <= p1):
                    gaps.append(f"곡선의 최댓값 위치가 {peak:.4f}, 최솟값 위치가 {trough:.4f} 다"
                                f" (기준 {p0}~{p1} — 둘 중 하나가 이 구간 안에 있어야 합니다)")
        if "hline_y" in spec:
            h0, h1 = spec["hline_y"]
            if not any(h0 <= v <= h1 for v in hline):
                gaps.append(f"기준 구간({h0:g} ~ {h1:g}) 안에 가로선이 없다"
                            + (f" (지금 {hline[0]:.4g})" if hline else " (가로선이 아예 없다)"))
        if "vline_x" in spec:
            v0, v1 = spec["vline_x"]
            if not any(v0 <= v <= v1 for v in vline):
                gaps.append(f"기준 구간({v0:g} ~ {v1:g}) 안에 세로선이 없다"
                            + (f" (지금 {vline[0]:.4f})" if vline else " (세로선이 아예 없다)"))
    elif spec.get("kind") == "bar":
        heights = [float(p.get_height()) for p in ax.patches]
        if "n" in spec and len(heights) != spec["n"]:
            gaps.append(f"막대가 {len(heights)}개다 (기준 {spec['n']}개)")
        if heights and "ymax" in spec and not (spec["ymax"][0] <= max(heights) <= spec["ymax"][1]):
            gaps.append(f"가장 높은 막대가 {max(heights):.4f} 다 (기준 {spec['ymax']})")
    elif spec.get("kind") == "interval":
        spans = _spans(ax)
        if "n_spans" in spec:
            n_lo, n_hi = spec["n_spans"]
            if not (n_lo <= len(spans) <= n_hi):
                gaps.append(f"가로 구간 막대가 {len(spans)}개다 (기준 {n_lo}개 — 세그먼트 하나에 막대"
                            f" 하나, 강조로 덧그린 막대는 {n_hi - n_lo}개까지 넘어갑니다)")
        if spans:
            widths = [right - left for left, right in spans]
            left_end, right_end = min(left for left, _ in spans), max(right for _, right in spans)
            if "span_max" in spec:
                lo, hi = spec["span_max"]
                if not (lo <= max(widths) <= hi):
                    gaps.append(f"가장 긴 막대의 길이가 {max(widths):.4f} 다 (기준 {lo:g}~{hi:g})")
            if "x_range" in spec:
                lo, hi = spec["x_range"]
                if not (_near(left_end, lo) and _near(right_end, hi)):
                    gaps.append(f"막대가 차지하는 가로축 값이 {left_end:.4g}~{right_end:.4g} 다"
                                f" (기준 {lo:g}~{hi:g})")
        if "n_points" in spec:
            _, pt_sets = _pt_sets(ax)
            n_pts = int(sum(p.shape[0] for p in pt_sets))
            n_lo, n_hi = spec["n_points"]
            if not (n_lo <= n_pts <= n_hi):
                gaps.append(f"점이 {n_pts}개다 (기준 {n_lo}개 — 세그먼트 하나에 점 하나)")
        if "vline_x" in spec:
            _, _, vline = _lines(ax)
            v0, v1 = spec["vline_x"]
            if not any(v0 <= v <= v1 for v in vline):
                gaps.append(f"기준 구간({v0:g} ~ {v1:g}) 안에 세로선이 없다"
                            + (f" (지금 {vline[0]:.4f})" if vline else " (세로선이 아예 없다)"))
    elif spec.get("kind") == "scatter":
        markers, pt_sets = _pt_sets(ax)                          # plot(x, y, "o") 로 찍은 점도 셉니다
        n_pts = int(sum(p.shape[0] for p in pt_sets))
        if "n_points" in spec:
            n_spec = spec["n_points"]
            n_lo, n_hi = n_spec if isinstance(n_spec, tuple) else (n_spec, n_spec)
            if not (n_lo <= n_pts <= n_hi):
                gaps.append(f"점이 {n_pts}개다 (기준 {n_lo}개 — 세그먼트 하나에 점 하나,"
                            f" 강조로 덧찍는 점은 {n_hi - n_lo}개까지 넘어갑니다)")
        xy = np.vstack([p for p in pt_sets if p.size]) if any(p.size for p in pt_sets) else None
        if xy is not None:
            xy = xy[np.isfinite(xy).all(axis=1)]
        if xy is not None and xy.size:
            xs, ys = xy[:, 0], xy[:, 1]
            if "x_range" in spec:
                lo, hi = spec["x_range"]
                if not (_near(xs.min(), lo) and _near(xs.max(), hi)):
                    gaps.append(f"가로축 값이 {xs.min():.3g}~{xs.max():.3g} 다 (기준 {lo:g}~{hi:g})")
            if "y_range" in spec:
                lo, hi = spec["y_range"]
                if not (_near(ys.min(), lo) and _near(ys.max(), hi)):
                    gaps.append(f"세로축 값이 {ys.min():.4g}~{ys.max():.4g} 다 (기준 {lo:g}~{hi:g})")
            if "ymax" in spec and not (spec["ymax"][0] <= ys.max() <= spec["ymax"][1]):
                gaps.append(f"세로축 최댓값이 {ys.max():.4g} 다"
                            f" (기준 {spec['ymax'][0]:g}~{spec['ymax'][1]:g})")
            if "gap_min" in spec:
                spread = float(np.abs(ys - xs).max())
                if spread < spec["gap_min"]:
                    gaps.append(f"두 축의 값이 가장 크게 벌어진 곳도 {spread:.4f} 뿐이다"
                                f" (기준 {spec['gap_min']:g} 이상)")
        if "hline_y" in spec:
            _, hline, _ = _lines(ax)
            h0, h1 = spec["hline_y"]
            if not any(h0 <= v <= h1 for v in hline):
                gaps.append(f"기준 구간({h0:g} ~ {h1:g}) 안에 가로선이 없다"
                            + (f" (지금 {hline[0]:.4g})" if hline else " (가로선이 아예 없다)"))
        if spec.get("xscale") and ax.get_xscale() != spec["xscale"]:
            gaps.append(f"가로축이 {ax.get_xscale()} 눈금이다 (기준 {spec['xscale']} 눈금)")
        if spec.get("diag") and not _diagonal(ax, skip=[ln for ln, _, _ in markers], xy=xy):
            gaps.append("두 축의 값이 같은 점을 잇는 대각선(y = x)이 점이 놓인 구간에 없다")
    if spec.get("축라벨") and not (ax.get_xlabel().strip() and ax.get_ylabel().strip()):
        gaps.append("가로축·세로축 라벨 중 빈 것이 있다")
    return gaps


_ref_cache = {}                                # 참조 그림이 표집을 한 번만 하도록 담아 두는 자리


def _quiet_posterior(y_, E_, seed0=49):
    """참조 그림용 사후 표본 — 표집기가 출력하는 안내 문구를 잠시 끄고 한 번만 실행합니다"""
    import logging
    import warnings as _w
    try:
        import pymc as pm
    except Exception:
        return None                        # 설치 셀을 아직 안 돌렸으면 참조 그림은 건너뜁니다
    silence = [logging.getLogger(n) for n in ("pymc", "pymc.sampling", "pytensor")]
    prev = [g.level for g in silence]
    for g in silence:
        g.setLevel(logging.ERROR)
    try:
        with _w.catch_warnings():
            _w.simplefilter("ignore")
            J_ = int(len(y_))
            with pm.Model():
                mu = pm.Normal("mu", np.log(0.10), 1.0)
                tau = pm.HalfNormal("tau", 1.0)
                z = pm.Normal("z", 0.0, 1.0, shape=J_)
                theta = pm.Deterministic("theta", tau * z)
                pm.Poisson("obs", mu=pm.math.exp(mu + theta) * E_, observed=y_)
                posterior = pm.sample(draws=1000, tune=1000, chains=4, random_seed=seed0,
                                progressbar=False)
        return np.exp(posterior.posterior["mu"].values[..., None]
                      + posterior.posterior["theta"].values).reshape(-1, J_)
    except Exception:
        return None
    finally:
        for g, lv in zip(silence, prev):
            g.setLevel(lv)


def _ref_plot_m1():
    """미션 1 의 참조 그림 — 관측이 적은 12개 세그먼트의 90% 구간 · 무 풀링 점 · 완전 풀링 세로선"""
    y_, E_, thin, ref = _get("CLAIMS"), _get("EXPO"), _get("THIN12"), _get("LAM_POOL")
    if y_ is None or E_ is None or thin is None or ref is None:
        return None
    span = _ref_cache.get("m1")
    if span is None:
        cell_val = _quiet_posterior(y_, E_)
        if cell_val is None:
            return None
        sampled = cell_val[:, np.asarray(thin)]
        span = (np.percentile(sampled, 5, axis=0), np.percentile(sampled, 95, axis=0))
        _ref_cache["m1"] = span
    left, right = span
    ypos = np.arange(len(left))
    f = _orig_figure(figsize=(FIG_W * 0.8, 3.8))
    a = f.add_subplot(111)
    a.hlines(ypos, left, right, color=COL["하늘색"], lw=4)
    a.plot(np.asarray(y_)[np.asarray(thin)] / np.asarray(E_)[np.asarray(thin)], ypos,
           "o", color=COL["회색"])
    a.axvline(float(ref), ls="--", color=COL["남색"])
    a.set_xlabel("연간 사고율 (건/년)")
    a.set_ylabel("노출이 적은 12개 세그먼트")
    return f


def _ref_plot_m2():
    """미션 2 의 참조 그림은 두지 않습니다 — 사후 표본을 두 벌 더 돌려야 해서, 어긋난 곳은
    채점 문장이 짚습니다"""
    return None


def _rel_span_approx(n_, k_, ref_p, M=300.9):
    """참조 그림용 근사 — 계층 사후분포를 다시 실행하지 않고 공식으로 바로 구하는 해(closed-form)로 구간 길이와 상대 구간 길이를 계산합니다"""
    post_p = (np.asarray(k_, dtype=float) + M * float(ref_p)) / (np.asarray(n_, dtype=float) + M)
    sd_val = np.sqrt(post_p * (1 - post_p) / (np.asarray(n_, dtype=float) + M))
    width = 2 * 1.96 * sd_val
    return post_p, width, width / post_p


def _ref_plot_m3():
    """미션 3 의 참조 그림 — 세그먼트의 계약 수(로그) 대 상대 구간 길이, 상대 구간 길이 1 자리에 가로선.

    참조 그림은 내 그림과 비교하는 용도라 계층 사후분포를 다시 실행하지 않고 공식으로 바로 구하는 해(closed-form)로 그립니다
    (채점에 쓰는 값이 아닙니다).
    """
    n_, k_, ref_p = _get("N23"), _get("K23"), _get("P_POOL")
    if n_ is None or k_ is None or ref_p is None:
        return None
    _, _, rel_span = _rel_span_approx(n_, k_, ref_p)
    f = _orig_figure(figsize=(FIG_W * 0.8, 3.6))
    a = f.add_subplot(111)
    a.scatter(n_, rel_span, s=8, color=COL["연회색"])
    a.axhline(1.0, color=COL["남색"], ls=":")
    a.set_xscale("log")
    a.set_xlabel("세그먼트의 계약 수 (건, 로그 눈금)")
    a.set_ylabel("상대 구간 길이 (구간 길이 ÷ 추정값)")
    return f


def _ref_plot_task():
    """과제의 참조 그림 — 상대 구간 길이 1 을 선으로 두고 세그먼트를 두 부류로 구분한 한 예"""
    n_, k_, ref_p = _get("N23"), _get("K23"), _get("P_POOL")
    if n_ is None or k_ is None or ref_p is None:
        return None
    _, _, rel_span = _rel_span_approx(n_, k_, ref_p)
    auto = rel_span < 1.0
    f = _orig_figure(figsize=(FIG_W * 0.8, 3.6))
    a = f.add_subplot(111)
    for pick, color, name in ((auto, COL["하늘색"], "자동 승인"), (~auto, COL["파랑"], "사람 심사")):
        a.scatter(np.asarray(n_)[pick], rel_span[pick], s=8, color=color, label=name)
    a.axhline(1.0, color=COL["남색"], ls=":")
    a.set_xscale("log")
    a.set_xlabel("세그먼트의 계약 수 (건, 로그 눈금)")
    a.set_ylabel("상대 구간 길이 (구간 길이 ÷ 추정값)")
    a.legend(frameon=False)
    return f
_REF = {"m1": _ref_plot_m1, "m2": _ref_plot_m2,   # 미션마다 참조 그림을 그리는 함수 하나
        "m3": _ref_plot_m3, "과제": _ref_plot_task}


def _show(fig):
    """채점 훅 안에서 그림을 화면에 올립니다 (인라인 백엔드는 이 시점에 plt.show() 가 듣지 않습니다)"""
    try:
        from IPython.display import display
        display(fig)
    except Exception:
        plt.show()


def _ref_show(fig, key):
    """내 그림과 참조 그림을 나란히 — 내 그림이 없으면 참조 그림도 띄우지 않습니다"""
    if not _axes(fig) or key not in _REF:
        return
    ref = None
    try:
        ref = _REF[key]()
        if ref is None:
            return
        panels = [(fig, "내 그림"), (ref, "참조 그림")]
        f = _orig_figure(figsize=(FIG_W, 3.4))
        for i, (g, title) in enumerate(panels):
            a = f.add_subplot(1, len(panels), i + 1)
            buf = io.BytesIO()
            (g if hasattr(g, "savefig") else g.figure).savefig(buf, format="png", dpi=110,
                                                               bbox_inches="tight")
            buf.seek(0)
            a.imshow(plt.imread(buf))
            a.axis("off")
            a.set_title(title, fontsize=11)
        _show(f)
        plt.close(f)
    except Exception:
        print("ⓘ 참조 그림을 나란히 놓지 못했습니다 — 위 채점 결과는 그대로 유효합니다")
    finally:
        if ref is not None:
            plt.close(ref)


def _one_line(e, n=120):
    """예외를 한 문장으로 — 트레이스백 대신 학생이 읽을 문장 하나만 남깁니다"""
    txt = str(e).replace("\n", " ").strip()
    return (txt[:n] + "…") if len(txt) > n else txt



def _prompt1(n=70):
    """지금 셀의 지시 첫 행 — 튜터에 보낼 질문에 그대로 싣습니다"""
    head_lines = [l.strip() for l in str(_get("PROMPT") or "").split("\n") if l.strip()]
    return (head_lines[0][:n] + "…") if head_lines else "____"


_PROMPT_MIN = 30
_PROMPT_PLACEHOLDERS = ("", "목표", "쓸 변수", "출력", "목표 —", "쓸 변수 —", "출력 —", "TODO",
                    "여기에 내 말로 적습니다")


def _prompt_missing():
    """PROMPT 입력란이 아직 내 말로 채워지지 않았으면 사유 한 문장, 채워졌으면 None"""
    try:
        txt = str(_ip.user_ns.get("PROMPT", _get("PROMPT")) or "").strip()
    except Exception:
        txt = str(_get("PROMPT") or "").strip()
    if not txt:
        return "PROMPT 입력란이 비어 있습니다"
    if len(txt) < _PROMPT_MIN:
        return f"PROMPT 입력란이 {len(txt)}자뿐입니다 (최소 {_PROMPT_MIN}자)"
    stripped = txt.replace("—", "").replace("·", "").strip()
    if stripped in _PROMPT_PLACEHOLDERS:
        return "PROMPT 입력란이 자리표시자 그대로입니다"
    return None


def _reveal_show(txt):
    """완성본 코드와 해설 — **평문 그대로** 찍는다.

    LMS 렌더러 계약: 「🔓 정답이 열렸습니다 …」 행부터 채점 표를 닫는 `─` 구분선 행까지를
    출력 패널이 「🔓 완성본 풀이 보기」 상자로 접는다. 접힘의 표지가 이 **평문 마커**라
    HTML `details` 로 싣지 않는다 — 마커도 이 함수도 바꾸지 않는다.
    """
    print(txt)


def judge(key, text="", figs=(), error=None):
    """출력 텍스트와 그림을 기준과 대조해 채점 표를 찍습니다 — 통과하면 완성본 풀이를 엽니다"""
    spec = _ANS.get(key)
    if spec is None:                       # 기준이 아직 없는 미션은 조용히 넘어갑니다
        return None
    head_label = ("미션 " + key[1:]) if key[:1] == "m" and key[1:].isdigit() else key
    failed, passed = [], True
    print("\n" + "─" * 54)
    print(f"◼ {head_label} 채점 결과")
    reason = _prompt_missing()                        # 값·그래프보다 먼저 — 내 말로 시킨 자리가 있는가
    _locked = bool(reason)
    if reason and key != "m2":
        print(f"❌ {reason} — 짧게 시켜도 됩니다. 다만 무엇을 어떤 단위로 출력할지까지 적으면 한 번에 통과할 확률이 높습니다")
        print("   PROMPT 입력란을 채우기 전에는 값·그래프 채점도, 완성본 풀이도 열지 않습니다")
        print("─" * 54)
        return False
    elif reason:                                   # 함정 미션만은 전임자 코드의 진단이 먼저다
        print(f"ⓘ {reason} — 아래 채점 결과는 전임 담당자 코드의 진단입니다."
              " 고칠 지시를 PROMPT 에 쓰세요")
    if error is not None:                   # 오류가 나면 값·그래프 채점은 뜻이 없습니다
        print(f"❌ 실행 오류 — {type(error).__name__}: {_one_line(error)}")
        hint, question = _HINT.get(key, {}).get("error", ("", "이 오류가 어느 코드 행에서 왜 났는지"))
        if hint:
            print(f"   힌트 : {hint}")
        form = _ASK.get(key, "full")
        print(_ASK_HEAD[form])
        print("  " + _ASK_FORM[form].format(name="실행 오류", prompt=_prompt1(), ask=question))
        print("─" * 54)
        return False
    txt = text or ""
    if not txt.strip():
        print("❌ 출력이 없습니다 — 값을 print 하도록 프롬프트의 출력 형식에 적으세요")
    units = [str(ref[2]) for ref in spec.get("values", {}).values() if len(ref) > 2]
    val_lines = _value_lines(txt, units)                    # 단위·「=」 가 붙은 행이 있으면 그 행만 봅니다
    nums = _nums("\n".join(val_lines)) if val_lines else _nums(txt)
    table_flood = not val_lines and len(nums) > 30        # 골라낸 자리 없이 숫자만 잔뜩 = 표를 통째로 띄운 모양
    if table_flood:
        print(f"❌ 값 — 단위도 「이름 = 값」 도 없이 숫자가 {len(nums)}개 나왔습니다 — 표를"
              " 통째로 띄우면 어느 숫자가 답인지 채점이 고르지 못합니다."
              " 값은 「이름 = 숫자 단위」 한 행으로 출력하세요")
        nums = []
    for label, ref in spec.get("values", {}).items():
        lo, hi = float(ref[0]), float(ref[1])
        unit = ref[2] if len(ref) > 2 else ""
        tail = _label_lines(val_lines, label) if nums else []   # 이 라벨이 든 행의 「=」 뒤만 — 옆 칸과 안 섞이게
        fell_back = bool(nums) and not tail             # 라벨이 든 행이 없을 때만 값줄 전체로 물러납니다
        #  라벨이 든 행을 찾았으면 **라벨 뒤 첫 숫자 하나**만 본다 — 같은 행의 대조용
        #  값이 이 칸을 헛통과시키던 자리다(미션 1 실측, 2026-09-19).
        hits = [v for v in (_nums("\n".join(tail))[:1] if tail else nums) if lo <= v <= hi]
        if hits:
            print(f"✅ {label} — 출력에서 {hits[0]:,.6g} 를 찾았습니다")
            if unit and unit not in txt:
                print(f"   ⓘ 단위 「{unit}」 도 값과 함께 출력하면 보고서에 그대로 옮길 수 있습니다")
        elif table_flood:
            passed = False
            failed.append((label, label))
            print(f"❌ {label} — 「이름 = 숫자 {unit}」 한 행으로 골라 출력하면 채점이 읽습니다")
        else:
            passed = False
            failed.append((label, label))
            _prev_fail = _TRY.get(key, 0)              # 결정 16 — 첫 ❌ 에는 기준 범위를 감춘다
            print(f"❌ {label} — 기준에 닿는 숫자를 출력에서 찾지 못했습니다"
                  + (f" (기준 범위 {lo:g} ~ {hi:g})" if _prev_fail else ""))
            if fell_back:                            # 라벨 행을 못 찾아 값줄 전체를 본 자리
                print(f"   ⓘ 라벨 「{label}」 이 든 행을 찾지 못해 값줄 전체에서 찾았습니다"
                      " — 「이름 = 숫자 단위」 한 행으로 출력하면 그 행만 보고 채점합니다")
    plot_spec = spec.get("plot")
    if plot_spec:
        axs = [a for f in figs for a in _axes(f)]
        if not axs:
            passed = False
            failed.append(("그래프", "plot"))
            print("❌ 그래프 — 그림을 찾지 못했습니다 (그래프 한 장도 같은 프롬프트에 함께 시킵니다)")
        else:
            gaps = None
            for ax in axs:                 # 패널이 여럿이면 기준에 맞는 축이 하나만 있어도 통과입니다
                try:
                    this_gaps = _plot_gaps(ax, plot_spec)
                except Exception as e:
                    this_gaps = [f"그림에서 데이터를 뽑다 걸렸습니다 ({type(e).__name__})"]
                if not this_gaps:
                    gaps = []
                    break
                if gaps is None or len(this_gaps) < len(gaps):
                    gaps = this_gaps
            if gaps:
                passed = False
                failed.append(("그래프", "plot"))
                print("❌ 그래프 — " + " · ".join(gaps))
            else:
                print("✅ 그래프 — 기준대로 그렸습니다")
            _ref_show(figs[0] if figs else None, key)
    tbl = _HINT.get(key, {})
    attempt = _TRY[key] = _TRY.get(key, 0) + (1 if failed else 0)
    for name, hkey in failed:
        tiers = tbl.get(hkey, ("",))
        tier = tiers[:-1] if len(tiers) > 2 else tiers[:1]   # 2단 힌트는 마지막 원소가 튜터에 물을 것입니다
        hint = tier[min(attempt, len(tier)) - 1] if tier else ""
        if hint:
            print(f"   힌트({name}) : {hint}")
    if failed:
        form = _ASK.get(key, "full")
        print(_ASK_HEAD[form])
        for name, hkey in failed:
            question = (tbl.get(hkey, ("", ""))[-1] if len(tbl.get(hkey, ("",))) > 1 else "") \
                or "이 항목이 왜 ❌ 로 남는지"
            print("  " + _ASK_FORM[form].format(name=name, prompt=_prompt1(), ask=question))
    elif _locked:                                  # 값·그림은 닿았지만 내 말로 시킨 자리가 아직 비었습니다
        print("ⓘ 값과 그림은 기준에 닿았습니다 — PROMPT 에 고칠 지시를 내 말로 쓰고"
              " 다시 실행하면 완성본 풀이가 열립니다")
    else:
        print("🔓 정답이 열렸습니다 — 아래 「🔓 완성본 풀이 보기」를 펼쳐 내 코드와 견줘 보세요")
        _reveal_show(base64.b64decode(_REVEAL[key]).decode("utf-8"))
    print("─" * 54)
    return passed and not _locked


class _Tee:
    """「PROMPT 셀」의 출력을 화면과 버퍼 양쪽으로 흘려보냅니다"""

    def __init__(self, orig, buf):
        self._o, self._b = orig, buf

    def write(self, s):
        self._b.write(s)
        return self._o.write(s)

    def flush(self):
        self._o.flush()

    def __getattr__(self, n):
        return getattr(self._o, n)


def _tag(src):
    """첫 행이 `# 미션 N — 프롬프트`·`# 과제 — 프롬프트` 인 셀만 채점 대상입니다"""
    head_line = (src or "").split("\n")[0]
    m = re.match(r"\s*#\s*(미션\s*(\d+)|과제)\b", head_line)
    if not m or "프롬프트" not in head_line:                           # 「준비 셀」(`# 미션 N · 준비`)은 대상이 아닙니다
        return None
    return "m" + m.group(2) if m.group(2) else "과제"


def _pre(info=None):
    """「PROMPT 셀」이면 출력 버퍼를 걸고 그림 목록을 비웁니다"""
    try:
        tag = _tag(getattr(info, "raw_cell", ""))
        _STATE["tag"] = tag
        if tag is None or tag not in _ANS:
            return
        _FIGS.clear()
        _STATE["ids0"] = {id(f) for f in _open_figs()}
        _STATE["buf"], _STATE["out"] = io.StringIO(), sys.stdout
        sys.stdout = _Tee(sys.stdout, _STATE["buf"])
    except Exception:
        pass


def _post(result=None):
    """「PROMPT 셀」이 끝나면 출력을 되돌리고 그 셀의 출력·그림으로 채점합니다"""
    tag, buf, out = _STATE.get("tag"), _STATE.get("buf"), _STATE.get("out")
    _STATE["tag"], _STATE["buf"], _STATE["out"] = None, None, None
    if out is not None:
        sys.stdout = out
    if tag is None or tag not in _ANS:
        return
    seen = {id(f) for f in _FIGS}
    figs_all = list(_FIGS) + [f for f in _open_figs()
                         if id(f) not in seen and id(f) not in _STATE.get("ids0", set())]
    txt = buf.getvalue() if buf is not None else ""
    last_val = getattr(result, "result", None)       # 셀 마지막 표현식의 값 — print 없이 표로 띄운 값도 읽습니다
    if last_val is not None:
        try:
            txt = txt + "\n" + str(last_val)
        except Exception:
            pass
    try:
        judge(tag, txt, figs_all, error=getattr(result, "error_in_exec", None))
    except Exception as e:
        print(f"(채점을 마치지 못했습니다 — {type(e).__name__}: {_one_line(e)})")


try:
    _ip = get_ipython()
    for _ev, _fn in (("pre_run_cell", _pre), ("post_run_cell", _post)):
        for _old in list(_ip.events.callbacks[_ev]):
            if getattr(_old, "__name__", "") == _fn.__name__:
                _ip.events.unregister(_ev, _old)     # 이 셀을 다시 실행해도 채점은 한 번만 찍힙니다
        _ip.events.register(_ev, _fn)
    print("채점 준비 완료 — 미션의 「PROMPT 셀」을 실행하면 실행 결과(출력)에 채점 결과가 찍힙니다")
except Exception:
    print("채점 준비 완료 (IPython 밖이라 훅 없이 judge() 만 올렸습니다)")

## 관측이 적은 세그먼트의 추정값이 수축하는 방향

### 미션 1 — 242개 세그먼트의 무 풀링·완전 풀링·부분 풀링 비교

#### 할 일 — 미션 1 (25분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다. 데이터를 내려받으므로 1분 안팎 걸립니다.
2. 아래 만들 것대로 「PROMPT 셀」의 PROMPT 에 시킵니다.
3. 셀을 클릭한 채 AI 튜터에게 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶ (LMS 밖이면 PROMPT 를 AI 에 전달하고 받은 코드를 AI 코드 표시선 아래에 직접 입력합니다).
4. 실행 결과(출력)에 출력되는 채점 결과를 봅니다. ❌ 면 힌트대로 PROMPT 를 고쳐 3번부터 다시 합니다.

**이 미션의 질문** — 노출이 6.62년인 세그먼트의 요율을 그 세그먼트의 기록만으로 추정한 값과, 다른 세그먼트의 기록까지 함께 사용해 추정한 값은 각각 얼마입니까?

**쓸 수 있는 것** — 「준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 | 다룸 |
|---|---|---|
| `CLAIMS` | 세그먼트별 사고 건수, 단위 건 | 고정 |
| `EXPO` | 세그먼트별 노출, 단위 년 | 고정 |
| `RAW` | 세그먼트별 무 풀링 값, 단위 건/년 | 고정 |
| `LAM_POOL` | 완전 풀링 값 하나, 단위 건/년 | 고정 |
| `J` | 세그먼트 수 242 | 고정 |
| `I662` | 노출 6.62년 세그먼트의 번호 | 고정 |
| `THIN12` | 노출이 적은 12개 세그먼트의 번호 | 고정 |

#### 만들 것

「PROMPT 셀」 한 곳에서 아래가 다 나오게 합니다. 굵은 이름은 글자 그대로 사용합니다.

1. **노출 6.62년 세그먼트의 부분 풀링 값** — 단위 건/년 · 소수점 아래 4자리 · 계층 포아송의 사후 표본에서 그 세그먼트의 평균
   - 사후 표본은 로그 스케일에 있으므로 지수를 취해 원래 단위로 변환한 뒤에 평균을 계산합니다
2. **구간이 완전 풀링 값을 포함한 세그먼트 수** — 단위 개 · 정수 · `THIN12` 의 12개 세그먼트 가운데 90% 신용구간이 `LAM_POOL` 을 포함한 세그먼트
   - 여기서 세는 구간은 다른 세그먼트의 기록까지 함께 사용한 계층 모형의 사후구간이라, 이미 완전 풀링 값(계약 678,013건 전체로 계산한 값) 쪽으로 이동해 있습니다. 노출이 적은 12개 세그먼트에서 대부분 포함으로 나오는 것은 그 이동이 일어난 결과이지, 그 세그먼트를 계약 전체와 견준 검정의 결과가 아닙니다. 포함한 세그먼트가 몇 개인지가 2번의 값입니다
   - 90% 신용구간은 사후 표본의 5% 분위수부터 95% 분위수까지입니다(이론에서 다룬 95% 신용구간과 자르는 규칙이 같습니다)
3. **대조용 2행** — `RAW[I662]` 를 「같은 세그먼트의 무 풀링 값 = 값 건/년」 한 행으로, `LAM_POOL` 을 「완전 풀링 값 = 값 건/년」 한 행으로 **서로 다른 행에** 출력합니다
   - 채점 대상은 아닙니다. 다만 1번과 같은 행에 적으면 어느 숫자가 1번의 답인지 채점에서 구분되지 않습니다
4. **그래프 한 장** — 세로로 `THIN12` 의 12개 세그먼트를 배치하고 가로축이 연간 사고율인 구간 그림
   - 가로 막대 : 세그먼트마다 90% 신용구간 하나씩, 모두 12개
   - 점 : 같은 세그먼트의 `RAW` 값 하나씩, 모두 12개
   - 세로선 : `LAM_POOL` 값 위치에 그림을 위아래로 가로지르는 선 하나
   - 가로축 이름 : 「연간 사고율 (건/년)」
   - 세로축 이름 : 「노출이 적은 12개 세그먼트」

**조건** — 체인 4개 · 반복 1,000회 · 워밍업 1,000회 · 시드 49 로 고정합니다. 시드를 고정하지 않으면 실행할 때마다 소수점 아래 4자리가 달라집니다.

**출력 형식** — 값은 「이름 = 값 단위」 형태로 한 행씩 `print` 합니다. 그래프는 한 장입니다.

**통과 기준** — 1번과 2번이 각각 굵은 이름 그대로 한 행씩 출력되고 그 값이 모형이 내는 범위에 들며, 4번 그래프가 막대 12개·점 12개·세로선 1개와 가로축·세로축 이름을 갖추면 통과입니다. 채점은 이름 뒤의 첫 숫자까지 확인합니다. 이름을 정확히 적었는데 통과가 나오지 않으면 철자가 아니라 값을 다시 확인하세요.

#### 프롬프트에 넣을 것(힌트)

- **고정할 이름** — 위 표의 이름 7개는 이미 있으므로 다시 만들지 말라고 적습니다.
- **모형 3층** — 맨 위에 모든 세그먼트에 공통인 평균 하나, 가운데에 세그먼트마다의 편차를 세그먼트 간 표준편차에 표준정규 편차를 곱한 형태로, 맨 아래에 포아송. 표준정규 편차는 평균 0 에 표준편차 1 인 정규분포를 따르는 값이고, 여기에 세그먼트 간 표준편차 $\tau$ 를 곱한 값은 이론의 「2. 부분 풀링과 계층 모형」에 적은 세그먼트 편차 $\theta_j \sim N(0,\ \tau^2)$ 과 같습니다. 포아송의 평균에 그 세그먼트의 노출이 곱해진다는 것까지 적습니다.
- **출력** — 값 2개의 이름과 단위, 그리고 소수점 아래 몇 자리까지 출력할지.
- **그래프** — 가로 막대·점·세로선이 각각 무엇이고 몇 개인지 · 가로축과 세로축의 이름.

**값을 적기 전에** — `az.summary` 로 `r_hat` 최댓값과 `ess_bulk` 최솟값을 한 번 보고, 실행 로그에 발산 경고가 남았는지도 확인합니다. 위 조건인 반복 1,000회로도 두 기준값을 통과합니다. 통과하지 못하면 반복과 워밍업을 2배로 올려 다시 실행하고, 그렇게 했다는 것을 적어 둡니다.

**막히면** — AI 튜터에게 「사후 표본에서 세그먼트 하나의 평균과 5% · 95% 분위수를 어떻게 구하나요」 처럼 질문하세요.

#### 실행 전에 — 부분 풀링 값의 예측

세그먼트 하나의 값을 미리 예측하고, 고른 이유도 한 문장 적어 두세요.

> **[예측] 노출 6.62년에 사고 2건인 세그먼트의 무 풀링 값은 0.30 건/년입니다. 여기에 부분 풀링을 쓰면 값이 어디에 위치할까요?**
>
> 1. 0.30 그대로 — 그 세그먼트의 기록이 그 값이므로
> 2. 약 0.20 — 계약 전체의 값과 그 세그먼트의 값을 절반씩 반영한 위치
> 3. 약 0.10 — 계약 전체의 값에 거의 근접한다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

<details class="lms-extra"><summary>배경 — 사후 표본에서 얻는 값과 구간</summary>

이론의 「2. 부분 풀링과 계층 모형」에서 만든 3층 모형을 242개 세그먼트의 실제 기록에 그대로 적용합니다. 이 모형은 켤레가 성립하지 않아 사후분포를 식으로 적을 수 없으므로, 이론에서 다룬 MCMC 로 표집기가 사후분포에서 표본을 얻고 값과 구간은 그 표본에서 계산합니다.

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Generic_forest_plot.png" width="640" height="413"/>
<p align="center"><em>▲ 위 4번에서 요구한 그림의 본보기 — 행마다 한 대상의 값을 네모로, 그 값의 구간을 가로선으로, 여럿을 하나로 합친 값을 세로 점선으로 그렸다. 관측 수가 적은 위쪽 3개 행은 가로선이 길고, 관측이 많은 아래 2개 행은 짧다 (출처: Wikipedia, Generic_forest_plot.png)</em></p>

막대·점·세로선이 각각 무엇인지 이 그림에서 확인하세요. 다만 이 그림의 가로선은 연구마다 계산한 **신뢰구간**이고 오늘 그릴 막대는 사후 표본에서 계산한 **신용구간**이라, 측정하는 규칙이 서로 다릅니다(이론의 「7. 요율표에 적는 값: 신용구간과 상대 구간 길이」).

이 그림의 네모는 가로선 한가운데에 위치하지만, 오늘 그릴 점은 막대와 다른 값인 무 풀링 값이라 막대 밖에 위치할 수 있습니다. 점이 막대 가운데에서 떨어져 있는 거리가 그 세그먼트에서 일어난 **수축의 크기**입니다.

</details>

<details class="lms-extra"><summary>주의 — AI 튜터에게 맡길 일과 내가 확인할 일</summary>

왼쪽은 맡겨도 되는 일이고, 오른쪽은 **틀려도 오류가 나지 않아** 사람이 확인해야 하는 일입니다.

| AI 튜터에게 맡길 일 | 내가 반드시 확인할 일 |
|---|---|
| 모형을 만들고 표집기를 호출하는 코드 | 포아송의 평균에 그 세그먼트의 **노출**이 곱해져 있는가 |
| 사후 표본을 세그먼트 방향으로 전개해 평균·분위수 계산 | 값이 **건/년** 단위로 타당한가, 무 풀링 값과 완전 풀링 값 사이에 위치하는가 |
| 구간 막대와 점과 세로선을 한 장에 배치 | 막대가 12개인가, 세로선이 완전 풀링 값 위치에 있는가 |
| `az.summary` 로 진단값 계산 | `r_hat` 최댓값이 1.01 이하이고 `ess_bulk` 와 `ess_tail` 최솟값이 각각 400 이상인가, 실행 로그의 발산 경고가 0건인가 |

</details>

In [ ]:
# 미션 1 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 1 에 쓸 변수를 직접 만듭니다
from sklearn.datasets import fetch_openml

# freMTPL2freq — 프랑스 자동차보험 계약별 사고 기록 (678,013 x 12). OpenML 에서 바로 내려받습니다
df = fetch_openml(data_id=41214, as_frame=True).frame
df["IDpol"] = df["IDpol"].astype(int)
for c in ["Area", "VehBrand", "VehGas", "Region"]:
    df[c] = df[c].astype(str).str.strip("'").astype("category")

# 첫날 고정한 전처리 2개 규칙 — 5일 내내 이대로 씁니다
df["ClaimNb"] = df["ClaimNb"].astype(int).clip(upper=4)   # 사고 5건 이상은 극소수라 상한 4 로 상한 처리
df["Exposure"] = df["Exposure"].clip(upper=1.0)           # 유지 기간은 1년 상한

# 지역 x 브랜드 242개 세그먼트 — 오늘 앞 두 미션이 쓰는 세그먼트이고, 여기서 다루는 값은 연간 사고율입니다
SEGS242 = (df.groupby(["Region", "VehBrand"], observed=True)
              .agg(n=("IDpol", "size"), claims=("ClaimNb", "sum"), expo=("Exposure", "sum"))
              .reset_index().sort_values(["Region", "VehBrand"]).reset_index(drop=True))

CLAIMS = SEGS242["claims"].to_numpy(dtype=float)    # 쓸 변수 1 — 세그먼트별 사고 건수 (단위 건)
EXPO = SEGS242["expo"].to_numpy(dtype=float)        # 쓸 변수 2 — 세그먼트별 노출 (단위 년)
RAW = CLAIMS / EXPO                                  # 쓸 변수 3 — 세그먼트별 무 풀링 값 (건/년)
LAM_POOL = float(CLAIMS.sum() / EXPO.sum())          # 쓸 변수 4 — 완전 풀링 값 하나 (건/년)
J = int(len(CLAIMS))                                 # 쓸 변수 5 — 세그먼트 수
I662 = int(np.argmin(np.abs(EXPO - 6.62)))           # 쓸 변수 6 — 노출 6.62년 세그먼트의 번호
THIN12 = np.argsort(EXPO)[:12]                       # 쓸 변수 7 — 노출이 적은 12개 세그먼트의 번호

print(f"계약 {len(df):,} 건 · 노출 합 {df['Exposure'].sum():,.2f} 년 · 사고 합 {int(df['ClaimNb'].sum()):,} 건")
print(f"세그먼트 {J} 개 · 세그먼트의 계약 수 최소 {SEGS242['n'].min():,} / 중앙 {SEGS242['n'].median():,.0f}"
      f" / 최대 {SEGS242['n'].max():,} 건 · 사고가 한 건도 없는 세그먼트 {int((CLAIMS == 0).sum())} 개")
print(f"완전 풀링 {LAM_POOL:.6f} 건/년 · 노출 6.62년 세그먼트의 무 풀링 {RAW[I662]:.4f} 건/년"
      f" · 노출이 가장 많은 세그먼트({EXPO.max():,.0f} 년)의 무 풀링 {RAW[int(EXPO.argmax())]:.4f} 건/년")
print("쓸 수 있는 이름 : CLAIMS (세그먼트별 사고 건수, 건) · EXPO (세그먼트별 노출, 년) · RAW (무 풀링, 건/년)"
      " · LAM_POOL (완전 풀링, 건/년) · J (세그먼트 수) · I662 (노출 6.62년 세그먼트 번호) · THIN12 (관측이 적은 12개 세그먼트 번호)")

In [ ]:
# 미션 1 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

> **[문제] 미션 1 의 AI 코드 표시선 아래 코드에서 포아송의 평균을 `exp(mu + theta) * EXPO` 로 적은 것이 결과에 어떻게 작용하나요?**
>
> 1. 세그먼트별 사고 건수를 노출로 미리 나눠 사고율로 바꿔 둔 것이라, 모형이 집계하는 값의 단위는 건/년이다
> 2. 모형이 집계하는 것은 사고 건수 그대로이고, 노출을 곱한 항이 「그 세그먼트를 얼마나 오래 관측했는가」를 평균에 반영한다 — 그래서 exp(mu + theta) 하나가 연간 사고율이 된다
> 3. 노출은 가중치라서 곱하든 더하든 같은 값이 되고, 세그먼트의 순서만 달라진다

## 사후 표본의 수렴 진단 — R-hat 과 유효표본수

### 미션 2 — 사후 표본의 수렴 진단 (함정 미션)

#### 할 일 — 미션 2 (25분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다. 이 미션은 사후 표본을 두 번 얻습니다. 2번과 4번에서 각각 표집을 기다리고, 한 번에 1분에서 2분 정도입니다.
2. 「PROMPT 셀」을 **먼저 그대로 ▶** 해 전임 담당자 코드의 채점 결과(❌)를 봅니다. PROMPT 가 비어 있어도 이 채점은 출력됩니다.
3. 아래 만들 것대로 「PROMPT 셀」의 PROMPT 에 시킵니다.
4. 미션 1 과 같은 순서로 AI 튜터에게 코드를 받아 ▶ 하고, 채점 결과가 ❌ 면 힌트대로 PROMPT 를 고쳐 다시 합니다.

**이 미션의 질문** — 사후평균은 미션 1 과 같은 값인데 채점은 통과로 판정하지 않습니다. 이 사후 표본을 요율표에 사용해도 됩니까?

**쓸 수 있는 것** — 「준비 셀」이 만들어 둔 이름입니다. 마지막 `prev` 하나만 예외입니다.

| 이름 | 무엇 | 다룸 |
|---|---|---|
| `CLAIMS` | 세그먼트별 사고 건수, 단위 건 | 고정 |
| `EXPO` | 세그먼트별 노출, 단위 년 | 고정 |
| `J` | 세그먼트 수 242 | 고정 |
| `I662` | 노출 6.62년 세그먼트의 번호 | 고정 |
| `THRESH_RHAT` | 이론에서 정한 R-hat 기준값 1.01 | 고정 |
| `THRESH_ESS` | 이론에서 정한 유효표본수 기준값 400, 단위 개 | 고정 |
| `prev` | 고치기 전 사후 표본 | 「PROMPT 셀」의 전임 담당자 코드가 만듭니다 · 지우지 않습니다 |

#### 만들 것 — 전임 담당자 코드는 그대로 두고 실행을 하나 더 만듭니다

전임 담당자 코드에는 결함이 둘 있습니다. 하나는 표집기를 직접 지정한 코드의 결함이고, 다른 하나는 그렇게 얻은 표본의 진단값을 읽지 않은 절차의 결함입니다. 오늘 바로잡는 것은 둘 다이고, 출력 이름의 「고친 뒤」는 새 실행의 결과입니다.

「여기부터 AI 코드」 행 아래의 전임 담당자 코드는 미션 1 과 같은 242개 세그먼트 계층 포아송입니다. 오류 없이 실행되지만 그 값을 신뢰할 수 있는지 측정하지 않았고 그림도 없습니다.

**전임 담당자 코드는 한 행도 지우거나 수정하지 않고 그대로 둡니다. 그 아래에 같은 모형을 한 번 더 적합하는 실행을 새로 만들고, 새 실행에서는 `step` 인자로 표집기를 명시하지 않습니다.** 두 실행에서 달라지는 것은 표집기와 반복 수뿐입니다. 굵은 이름은 글자 그대로 사용합니다.

1. **고친 뒤 가장 큰 R-hat** — 소수점 아래 4자리
   - 새로 만드는 실행은 **체인 4개 · 반복 5,000회 · 워밍업 5,000회 · 시드 49** 로 지정합니다
   - 워밍업은 본 표본을 얻기 전에 버리는 반복이고, `pm.sample` 의 `tune` 인자가 그 횟수입니다. 이론에서 번인이라고 부른 그 구간입니다
   - 새 실행에서 모수 전체의 R-hat 을 계산해 가장 큰 값 하나를 출력합니다
2. **고친 뒤 가장 작은 유효표본수** — 단위 개 · 정수
   - 같은 표본에서 모수 전체의 유효표본수를 계산해 가장 작은 값 하나를 출력합니다
3. **대조용 한 행** — 같은 표본에서 공통 평균 `mu` 의 사후평균과, 그 값을 지수로 복원한 연간 사고율을 한 행으로 출력합니다
   - 채점 대상은 아닙니다. 이론이 유효표본수 때문에 읽지 않고 미뤄 둔 값이 이 값입니다
4. **그래프 한 장** — 가로축은 세그먼트 번호, 세로축은 고친 뒤 그 세그먼트의 R-hat 인 산점도
   - 점 : 세그먼트 하나에 점 하나, 모두 242개 — 세그먼트 편차 `theta` 의 R-hat 만 고르면 242개입니다
   - 가로선 : 기준값 1.01 위치에 그림을 좌우로 가로지르는 선 하나
   - 가로축 이름 : 「세그먼트 번호」
   - 세로축 이름 : 「고친 뒤 세그먼트별 R-hat」

**출력 형식** — 값은 「이름 = 값 단위」 형태로 한 행씩 `print` 합니다. 그래프는 한 장입니다.

**통과 기준** — 1번과 2번이 각각 굵은 이름 그대로 한 행씩 출력되고 그 값이 모형이 내는 범위에 들며, 4번 그래프가 점 242개·가로선 1개와 가로축·세로축 이름을 갖추면 통과입니다.

#### 프롬프트에 넣을 것(힌트)

- **새 실행에서 달라질 인자** — 전임 담당자 코드의 `pm.sample` 에 전달한 인자를 하나씩 읽어 보세요. 체인이 이동하는 방식을 정하는 인자와 반복 수를 정하는 인자가 그 안에 있습니다.
- **바꾸지 않을 범위** — 모형 코드와 `CLAIMS` · `EXPO` · `J` 는 그대로 두라고 적습니다. 두 실행이 같은 모형이어야 고치기 전과 고친 뒤를 비교할 수 있습니다.
- **출력** — 값 2개의 이름과 단위, 소수점 아래 몇 자리까지 출력할지, 그리고 모수 **전체**를 계산한 뒤 최댓값·최솟값 하나로 줄인다는 것.
- **그래프** — 세그먼트 편차의 R-hat 을 점으로 배치한다는 것 · 가로선의 위치 · 가로축과 세로축의 이름.

**막히면** — AI 튜터에게 「제안이 기각되어 체인이 같은 상태에 남으면 유효표본수가 왜 반복 수보다 작아지나요」 처럼 질문하세요.

#### 새 실행을 만들기 전에 — 출력된 진단값 확인

전임 담당자 코드가 출력한 표에는 사후평균 옆에 두 열이 더 있습니다. 그 두 값을 이론에서 정한 기준값 2개와 비교해 각각 얼마나 차이가 나는지 한 문장 적어 두세요. 적을 때에는 그 값이 어느 모수에서 나온 값인지 이름도 함께 적습니다. 이론에서도 세그먼트별 값 242개와 공통 평균 하나를 나누어 판정했습니다.

> **[예측] 「AI 코드 표시선」 아래 코드는 오류 없이 돌아가고 값도 타당해 보이는데 채점은 통과로 판정하지 않습니다. 틀린 곳은 어디일까요?**
>
> 1. 표집기를 `pm.Metropolis()` 로 직접 지정해 체인이 반복마다 거의 이동하지 않는다
> 2. 세그먼트 간 표준편차의 사전분포 `HalfNormal(1.0)` 이 너무 넓어 세그먼트 값이 흩어진다
> 3. 노출을 곱하지 않고 더해서 값의 단위가 어긋난다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

<details class="lms-extra"><summary>배경 — 반복 횟수와 유효표본수가 다른 수인 이유</summary>

이론의 「5. 수렴 진단: R-hat 과 유효표본수」에서 기준값 2개를 정했습니다.

R-hat 은 1.01 이하, 유효표본수는 400 이상이 통과입니다. 오늘은 그 판정을 내가 작성하지 않은 코드에 적용합니다.

바로 앞 반복에서 제안이 기각되어 같은 상태에 남으면 그 반복에서 새로 얻는 정보가 없으므로, 반복 횟수와 표본의 정보량은 다른 수입니다. **유효표본수** 는 그 표본이 서로 독립인 표본 몇 개와 같은 정보량인지를 나타낸 값이라 체인이 같은 상태에 남으면 반복 횟수보다 훨씬 작게 계산되고, **R-hat** 은 체인 여럿이 같은 분포로 수렴했는지를 측정하므로 체인이 시작 위치 근처를 벗어나지 못하면 1 에서 멀어집니다.

<img src="https://en.wikipedia.org/wiki/Special:FilePath/MH_Rosenbrock_movie_ND.gif" width="560" height="378"/>
<p align="center"><em>▲ 메트로폴리스 체인 3개(노랑·분홍·하늘)가 한 반복씩 이동한 경로 — 화면 제목의 minimum 은 이 예제가 목표로 삼은 위치일 뿐이고, 우리는 같은 방식으로 사후분포에서 표본을 얻는다. 이동 경로가 앞으로 나아가지 못하고 한 위치에 겹친 구간이, 제안한 값이 기각되어 체인이 같은 상태에 남은 반복이다 (출처: Wikimedia Commons, MH Rosenbrock movie ND.gif)</em></p>

전임 담당자 코드에서는 유효표본수가 두 자리로 출력됩니다. 먼저 확인할 것은 반복 횟수가 아니라 표집기입니다.

</details>

<details class="lms-extra"><summary>주의 — AI 튜터에게 맡길 일과 내가 확인할 일</summary>

| AI 튜터에게 맡길 일 | 내가 반드시 확인할 일 |
|---|---|
| 표집기를 호출하는 코드 행을 바꿔 다시 실행하는 코드 | 두 실행이 **같은 모형**인가 — 바뀐 것이 표집기와 반복 수뿐인가 |
| arviz 로 R-hat·유효표본수를 계산해 최댓값·최솟값으로 줄이기 | **모수 전체**를 계산하는가, 한두 개만 계산하고 있지 않은가, 실행 로그의 발산 경고가 0건인가 |
| 점 242개와 기준값 가로선을 한 장에 배치 | 가로선이 1.01 위치인가, 점이 242개인가 |

</details>

In [ ]:
# 미션 2 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 2 에 쓸 변수를 직접 만듭니다
if "CLAIMS" not in globals():                                   # 이 미션만 따로 열어도 이 셀부터 실행하면 됩니다
    from sklearn.datasets import fetch_openml                    # (미션 1 을 이미 돌렸으면 통째로 건너뜁니다)
    df = fetch_openml(data_id=41214, as_frame=True).frame
    df["IDpol"] = df["IDpol"].astype(int)
    for c in ["Area", "VehBrand", "VehGas", "Region"]:
        df[c] = df[c].astype(str).str.strip("'").astype("category")
    df["ClaimNb"] = df["ClaimNb"].astype(int).clip(upper=4)       # 첫날 고정한 전처리 2개 규칙 — 미션 1 과 같습니다
    df["Exposure"] = df["Exposure"].clip(upper=1.0)
    SEGS242 = (df.groupby(["Region", "VehBrand"], observed=True)
                  .agg(n=("IDpol", "size"), claims=("ClaimNb", "sum"), expo=("Exposure", "sum"))
                  .reset_index().sort_values(["Region", "VehBrand"]).reset_index(drop=True))
    CLAIMS = SEGS242["claims"].to_numpy(dtype=float)   # 쓸 변수 1 — 세그먼트별 사고 건수 (단위 건)
    EXPO = SEGS242["expo"].to_numpy(dtype=float)       # 쓸 변수 2 — 세그먼트별 노출 (단위 년)
    J = int(len(CLAIMS))                                # 쓸 변수 3 — 세그먼트 수
    I662 = int(np.argmin(np.abs(EXPO - 6.62)))          # 쓸 변수 4 — 노출 6.62년 세그먼트의 번호

THRESH_RHAT = 1.01    # 쓸 변수 5 — 이론에서 정한 R-hat 기준값
THRESH_ESS = 400      # 쓸 변수 6 — 이론에서 정한 유효표본수 기준값 (단위 개)

print(f"세그먼트 {J} 개 · 미션 1 과 같은 모형을 같은 기록에 다시 적용합니다 — 바뀌는 것은 표집기를 호출하는 코드 행뿐입니다")
print(f"진단값은 모수마다 따로 나옵니다 — 공통 평균 하나 · 세그먼트 간 표준편차 하나 · 세그먼트마다 하나씩 {J} 개")
print(f"이론에서 정한 기준값 2개 — R-hat 은 {THRESH_RHAT} 이하 · 유효표본수는 {THRESH_ESS} 개 이상")
print("쓸 수 있는 이름 : CLAIMS (세그먼트별 사고 건수, 건) · EXPO (세그먼트별 노출, 년) · J (세그먼트 수)"
      " · I662 (노출 6.62년 세그먼트 번호) · THRESH_RHAT (R-hat 기준값) · THRESH_ESS (유효표본수 기준값, 개)")

In [ ]:
# 미션 2 — 프롬프트 (함정)   # ai: prompt
# 이 미션은 함정입니다. 아래는 전임 담당자가 남긴 코드로, 돌아가고 값도 그럴듯하지만 그 값을 신뢰할 수 있는지는 확인하지 않았습니다.
# 이 코드는 그대로 두고 그 아래에 무엇을 더 실행할지 PROMPT 에 적으면 도우미가 이어서 작성합니다.
PROMPT = """

"""
# --- 여기부터 AI 코드 ---
# ↓ 전임 담당자의 코드 (그대로 둡니다) — PROMPT 를 채우고 실행하면 무엇이 빠졌는지 채점이 짚어 줍니다
with pm.Model() as m_prev:
    mu = pm.Normal("mu", np.log(0.10), 1.0)
    tau = pm.HalfNormal("tau", 1.0)
    z = pm.Normal("z", 0.0, 1.0, shape=J)
    theta = pm.Deterministic("theta", tau * z)
    pm.Poisson("obs", mu=pm.math.exp(mu + theta) * EXPO, observed=CLAIMS)
    prev = pm.sample(draws=1000, tune=1000, chains=4,
                     step=pm.Metropolis(),          # NUTS 가 느려서 이걸로 바꿔 뒀습니다
                     random_seed=49, progressbar=False)

s = az.summary(prev, var_names=["mu", "tau"])
print(s[["mean", "r_hat", "ess_bulk"]])
lam_prev = np.exp(prev.posterior["mu"].values[..., None]
                  + prev.posterior["theta"].values).reshape(-1, J)
print(f"노출 6.62년 세그먼트 사후평균 = {lam_prev[:, I662].mean():.4f} 건/년")
print("표집 완료 — 이 값을 요율표에 사용합니다")

> **[문제] 미션 2 에서 고친 뒤의 코드는 모수마다 따로 계산된 R-hat 을 마지막에 `.max()` 한 번으로 줄입니다. 이 코드 행이 하는 일로 맞는 것은?**
>
> 1. 체인 4개가 산출한 R-hat 을 평균 내어 하나로 만든다 — 체인 하나가 수렴하지 않아도 평균에 희석되어 드러나지 않는다
> 2. 모수마다 따로 계산된 R-hat 을 한곳에 모아 그중 가장 큰 값 하나를 골라 기준값과 비교한다 — 하나라도 기준값을 초과하면 그 표본은 요율표에 사용하지 않는다는 뜻이다
> 3. 사후 표본의 평균을 R-hat 으로 바꾼다 — 값이 하나뿐이라 세그먼트마다 따로 볼 일이 없다

## 계약 63건 세그먼트의 신용구간 — 2,323개 세그먼트 일괄 추정

### 미션 3 — 사고 경험률을 다루는 계층 이항과 2,323개 세그먼트의 신용구간

#### 할 일 — 미션 3 (25분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다. 3번에서 세그먼트 2,323개를 한 번에 추정하므로 몇 분까지 걸립니다.
2. 아래 만들 것대로 「PROMPT 셀」의 PROMPT 에 시킵니다.
3. 미션 1 과 같은 순서로 AI 튜터에게 코드를 받아 ▶ 하고, 채점 결과가 ❌ 면 힌트대로 PROMPT 를 고쳐 다시 합니다.

**이 미션의 질문** — 계약 63건인 세그먼트의 구간을, 첫날 쓴 표준오차, 곧 같은 크기의 표본을 다시 뽑을 때 추정값이 흔들리는 정도로 계산한 ±2 표준오차 구간보다 얼마나 짧게 만들 수 있습니까?

**쓸 수 있는 것** — 「준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 | 다룸 |
|---|---|---|
| `N23` | 세그먼트별 계약 수, 단위 건 | 고정 |
| `K23` | 세그먼트별 사고를 겪은 계약 수, 단위 건 | 고정 |
| `RAW23` | 세그먼트별 무 풀링 값, 단위 비율 | 고정 |
| `P_POOL` | 완전 풀링 값 하나, 단위 비율 | 고정 |
| `J23` | 세그먼트 수 2,323 | 고정 |
| `I63` | 계약 63건·사고 4건 세그먼트의 번호 | 고정 |
| `SE63` | 그 세그먼트의 표준오차 | 고정 |
| `LO_D45` · `HI_D45` | 첫날 ±2 표준오차 구간의 아래끝·위끝 | 고정 |

#### 만들 것

「PROMPT 셀」 한 곳에서 아래가 다 나오게 합니다. 굵은 이름은 글자 그대로 사용합니다.

1. **계약 63건 세그먼트의 부분 풀링 값** — 단위 비율 · 소수점 아래 4자리 · 계층 이항의 사후 표본에서 그 세그먼트의 평균
   - 사후 표본은 **로짓 스케일**에 있으므로 확률로 변환한 뒤에 평균을 계산합니다. 로짓 스케일은 확률 $p$ 를 $\log\frac{p}{1-p}$ 로 변환해 음의 무한대부터 양의 무한대까지 확장한 좌표이고, 그 값을 다시 확률로 변환하는 함수가 시그모이드입니다
2. **그 세그먼트의 95% 신용구간 길이** — 단위 %p · 소수점 아래 2자리
   - 사후 표본의 2.5% 분위수와 97.5% 분위수의 차에 100 을 곱한 값입니다
3. **대조용 한 행** — 같은 세그먼트를 첫날의 ±2 표준오차 구간(`LO_D45` · `HI_D45`)으로 계산한 구간 길이도 같은 단위로 출력합니다
   - 채점 대상은 아닙니다. 두 수가 나란히 있어야 오늘 무엇이 달라졌는지 확인됩니다
4. **그래프 한 장** — 가로축은 세그먼트의 계약 수, 세로축은 상대 구간 길이인 산점도
   - 점 : 세그먼트 하나에 점 하나, 모두 2,323개
   - 상대 구간 길이 : 세그먼트마다 그 세그먼트의 구간 길이를 **그 세그먼트의 부분 풀링 값**으로 나눈 값입니다. 2번과 달리 여기서는 100 을 곱하지 않은 비율 그대로 나눕니다. 2,323개를 같은 수 하나로 나누지 않습니다
   - 가로축 눈금 : 로그 눈금 · 가로선 : 상대 구간 길이 1 위치에 그림을 좌우로 가로지르는 선 하나
   - 가로축 이름 : 「세그먼트의 계약 수 (건, 로그 눈금)」
   - 세로축 이름 : 「상대 구간 길이 (구간 길이 ÷ 추정값)」

**조건** — 체인 4개 · 시드 49 로 고정하고, 반복 횟수와 워밍업 횟수는 미션 1 에 지정한 반복 1,000회와 워밍업 1,000회의 각각 2배로 지정합니다. 미션 1 과 같은 횟수로 두면 세그먼트 간 표준편차의 R-hat 이 기준값 1.01 을 초과할 때가 있습니다.

**출력 형식** — 값은 「이름 = 값 단위」 형태로 한 행씩 `print` 합니다. 그래프는 한 장입니다.

**통과 기준** — 1번과 2번이 각각 굵은 이름 그대로 한 행씩 출력되고 그 값이 모형이 내는 범위에 들며, 4번 그래프가 점 2,323개·로그 눈금·가로선 1개와 가로축·세로축 이름을 갖추면 통과입니다.

#### 프롬프트에 넣을 것(힌트)

- **고정할 이름** — 위 표의 이름 9개는 이미 있으므로 다시 만들지 말라고 적습니다.
- **바뀌는 층 2개** — 미션 1 의 3층 가운데 맨 아래가 포아송에서 이항으로 바뀝니다. 이항의 시행 수 인자에 그 세그먼트의 계약 수가, 관측 인자에 사고를 겪은 계약 수가 들어갑니다. 맨 위 층의 중심도 로짓 스케일의 값 −2.94 로 바꿉니다. −2.94 는 사고 경험률 0.0502 를 로짓으로 변환한 값입니다.
- **출력** — 값 2개의 이름과 단위, 소수점 아래 몇 자리까지 출력할지, 그리고 둘째 값은 비율이 아니라 퍼센트포인트라는 것.
- **그래프** — 가로축과 세로축에 배치할 값 · 가로축 눈금 · 가로선의 위치 · 가로축과 세로축의 이름.

**막히면** — AI 튜터에게 「사후 표본에서 세그먼트마다 2.5% · 97.5% 분위수를 한꺼번에 구하려면 어떻게 하나요」 처럼 질문하세요.

#### 실행 전에 — 무사고 세그먼트 추정값의 예측

> **[예측] 2,323개 세그먼트 가운데 756개 세그먼트는 사고를 한 건도 겪지 않아 무 풀링 값이 0 입니다. 부분 풀링을 쓰면 이 세그먼트들의 추정값은 어디에 위치할까요?**
>
> 1. 0 에 그대로 남는다 — 그 세그먼트의 기록에 사고가 없으므로
> 2. 약 0.03 에서 0.05 사이 — 0 보다는 위, 계약 전체의 값보다는 아래
> 3. 계약 전체의 값 0.0502 와 정확히 일치한다 — 그 세그먼트만의 기록이 없으므로 계약 전체의 값을 그대로 사용한다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

756개 세그먼트가 모두 같은 값일지도 한 문장 적어 두세요.

<details class="lms-extra"><summary>배경 — 다루는 값을 사고 경험률로 바꾸는 이유와 상대 구간 길이</summary>

이론의 「7. 요율표에 적는 값: 신용구간과 상대 구간 길이」에서 확인한 계산을 오늘은 세그먼트 2,323개 전부에 적용합니다.

**다루는 값이 바뀝니다.** 앞의 미션 2개가 측정한 것은 연간 사고율이고, 이번에 다루는 값은 **사고 경험률**입니다. 계약 하나를 「사고를 1번 이상 겪었는가」로만 보고 세그먼트의 계약 수 가운데 몇 건이 그랬는지를 집계하므로, 앞 미션의 모형이 포아송이면 이번 모형은 이항입니다.

계약 전체로 계산해도 사고 경험률은 0.0502, 연간 사고율은 0.1006 이라 요율표의 한 열에 함께 적으면 그 열이 2배로 읽힙니다. 요율표의 한 열은 둘 중 하나로만 적습니다.

세그먼트 편차는 확률 스케일에서 그대로 더하면 0 과 1 을 벗어나므로 로짓 스케일에서 더하고, 더한 결과를 시그모이드로 확률에 대응시킵니다.

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Sigmoid_function_01.png" width="400" height="400"/>
<p align="center"><em>▲ 로짓 스케일의 값을 0 과 1 사이의 확률로 변환하는 시그모이드 곡선 — 가로축이 세그먼트 편차를 더하는 로짓 스케일이고, 세로축이 그 합을 대응시킨 확률이다. 가로축 0 이 확률 0.5, −3 이 약 0.05 다 (출처: Wikimedia Commons, Sigmoid function 01.png)</em></p>

**상대 구간 길이.** 구간 길이는 %p 로 적고 세그먼트의 값은 비율로 적으므로, 나누기 전에 구간 길이를 100 으로 나눠 두 수의 단위를 맞춥니다. 구간 길이를 그 세그먼트의 값으로 나눈 값이 **상대 구간 길이** 이고, **1** 은 구간 길이가 값과 같아지는 기준선입니다.

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Example_of_a_symmetrical_funnel_plot_created_with_MetaXL_Sept_2015.jpg" width="587" height="360"/>
<p align="center"><em>▲ 같은 것을 측정한 연구 20개를 세로축에 표준오차를 두고 그린 깔때기 그림 — 표준오차가 약 0.2 인 위쪽 점들은 좁게 모여 있는데 약 1.1 인 아래쪽 점들은 −1.7 부터 1.0 까지 흩어져 있어, 관측이 적을수록 값이 더 넓게 흩어진다 (출처: Wikimedia Commons, Example of a symmetrical funnel plot created with MetaXL Sept 2015.jpg)</em></p>

이 본보기 그림은 가로축이 각 연구가 산출한 값이고 세로축이 그 값의 표준오차라, 가로축이 계약 수이고 세로축이 상대 구간 길이인 오늘 그릴 그림과는 배치한 값이 서로 다릅니다.

**이 값에는 노출이 들어가지 않습니다.** 계약 하나를 사고 유무 0 아니면 1 로만 보므로, 포아송의 평균에 노출을 곱한 미션 1 과 달리 평균 보장 기간이 짧은 세그먼트가 낮게 나올 수 있습니다. 그래서 이 표는 세그먼트마다 수축의 크기를 대조하는 데 쓰고, 실제 요율은 노출을 오프셋으로 둔 빈도 모형에서 따로 계산합니다.

</details>

<details class="lms-extra"><summary>주의 — AI 튜터에게 맡길 일과 내가 확인할 일</summary>

| AI 튜터에게 맡길 일 | 내가 반드시 확인할 일 |
|---|---|
| 2,323개 세그먼트의 계층 이항을 만들고 표집기를 호출하는 코드 | 이항의 시행 수 인자에 그 세그먼트의 **계약 수**가, 관측 인자에 **사고를 겪은 계약 수**가 들어갔는가 |
| 사후 표본을 확률 스케일로 변환해 세그먼트마다 평균과 분위수 계산 | 값이 **비율**로 타당한가, 사고가 없던 세그먼트가 0 에 남지 않고 어딘가에 위치하는가, 실행 로그의 발산 경고가 0건인가 |
| 상대 구간 길이를 계약 수에 대해 산점도로 배치 | 가로축이 로그 눈금인가, 가로선이 상대 구간 길이 1 위치인가, 점이 2,323개인가 |

</details>

In [ ]:
# 미션 3 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 3 에 쓸 변수를 직접 만듭니다
if "df" not in globals():                                       # 이 미션만 따로 열어도 이 셀부터 실행하면 됩니다
    from sklearn.datasets import fetch_openml                    # (앞 미션을 이미 돌렸으면 통째로 건너뜁니다)
    df = fetch_openml(data_id=41214, as_frame=True).frame
    df["IDpol"] = df["IDpol"].astype(int)
    for c in ["Area", "VehBrand", "VehGas", "Region"]:
        df[c] = df[c].astype(str).str.strip("'").astype("category")
    df["ClaimNb"] = df["ClaimNb"].astype(int).clip(upper=4)       # 첫날 고정한 전처리 2개 규칙 — 앞 미션과 같습니다
    df["Exposure"] = df["Exposure"].clip(upper=1.0)

# 다루는 값이 바뀝니다 — 사고를 몇 번 겪었는가가 아니라 「1번 이상 겪었는가」 로 계약 하나를 0 아니면 1 로 봅니다
df["사고겪음"] = (df["ClaimNb"] > 0).astype(int)

# 지역 x 브랜드 x 차량 출력 등급 2,323개 세그먼트 — 앞 두 미션의 242개 세그먼트를 출력 등급으로 한 번 더 나눈 세그먼트입니다
SEGS2323 = (df.groupby(["Region", "VehBrand", "VehPower"], observed=True)
               .agg(n=("IDpol", "size"), k=("사고겪음", "sum"))
               .reset_index().sort_values(["Region", "VehBrand", "VehPower"]).reset_index(drop=True))

N23 = SEGS2323["n"].to_numpy(dtype=float)     # 쓸 변수 1 — 세그먼트별 계약 수 (단위 건)
K23 = SEGS2323["k"].to_numpy(dtype=float)     # 쓸 변수 2 — 세그먼트별 사고를 겪은 계약 수 (단위 건)
RAW23 = K23 / N23                              # 쓸 변수 3 — 세그먼트별 무 풀링 값 (비율)
P_POOL = float(K23.sum() / N23.sum())          # 쓸 변수 4 — 완전 풀링 값 하나 (비율)
J23 = int(len(N23))                            # 쓸 변수 5 — 세그먼트 수
I63 = int(SEGS2323.index[(SEGS2323["n"] == 63) & (SEGS2323["k"] == 4)][0])   # 쓸 변수 6 — 계약 63건·사고 4건 세그먼트의 번호

# 첫날 이 세그먼트에 적용한 구간 — 무 풀링 비율에 ±2 표준오차를 붙인 구간입니다 (공식으로 바로 구하는 해(closed-form)라 여기서 다시 계산합니다)
SE63 = float(np.sqrt(RAW23[I63] * (1 - RAW23[I63]) / N23[I63]))   # 쓸 변수 7 — 그 세그먼트의 표준오차
LO_D45 = float(RAW23[I63] - 2 * SE63)                             # 쓸 변수 8 — 첫날 구간의 아래끝
HI_D45 = float(RAW23[I63] + 2 * SE63)                             # 쓸 변수 9 — 첫날 구간의 위끝

print(f"세그먼트 {J23:,} 개 · 세그먼트의 계약 수 최소 {int(N23.min()):,} / 중앙 {np.median(N23):,.0f} / 최대 {int(N23.max()):,} 건"
      f" · 사고를 한 건도 겪지 않은 세그먼트 {int((K23 == 0).sum()):,} 개 ({(K23 == 0).mean() * 100:.1f}%)")
print(f"여기서 다루는 값은 사고 경험률(비율)입니다 — 계약 전체 {P_POOL:.6f}"
      f" · 앞 두 미션이 쓴 연간 사고율과는 다른 값입니다")
print(f"계약 63건·사고 4건 세그먼트(번호 {I63}) — 무 풀링 {RAW23[I63]:.6f} · 표준오차 {SE63:.6f}"
      f" · 첫날 ±2 표준오차 구간 [{LO_D45:.6f}, {HI_D45:.6f}] · 그 구간 길이 {(HI_D45 - LO_D45) * 100:.2f} %p")
print("쓸 수 있는 이름 : N23 (세그먼트별 계약 수, 건) · K23 (세그먼트별 사고를 겪은 계약 수, 건) · RAW23 (무 풀링, 비율)"
      " · P_POOL (완전 풀링, 비율) · J23 (세그먼트 수) · I63 (계약 63건·사고 4건 세그먼트 번호)"
      " · SE63 (그 세그먼트의 표준오차) · LO_D45 · HI_D45 (첫날 ±2 표준오차 구간의 두 끝)")

In [ ]:
# 미션 3 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

> **[문제] 미션 3 의 AI 코드 표시선 아래 코드는 세그먼트 j 의 확률을 `sigmoid(mu + theta[j])` 로 적고 이항의 시행 수 인자에 `N23` 을 전달합니다. 이 두 인자가 하는 일로 맞는 것은?**
>
> 1. `N23` 은 그 세그먼트에서 사고를 겪은 계약 수라서 모형이 집계하는 것은 사고 건수이고, `sigmoid` 는 그 건수를 계약 수로 나눠 비율로 바꿔 준다
> 2. `N23` 은 그림의 가로축에만 쓰이는 값이고, `sigmoid` 는 값을 보기 좋게 다듬는 항이라 빼도 같은 사후분포가 나온다
> 3. `N23` 은 그 세그먼트의 계약 수, 곧 시행 수다 — 같은 확률로 `N23` 번 시행해 `K23` 번 나왔다고 두는 것이라 계약이 많은 세그먼트일수록 그 세그먼트의 데이터가 추정에 미치는 영향이 커지고, `sigmoid` 는 로짓 스케일에서 더한 `mu + theta[j]` 를 0 과 1 사이의 확률로 변환한다

## 사전 강도 102건과 약 300건으로 만드는 요율표 — 미니 프로젝트

> **[과제] 과제 제출 — 세그먼트별 요율표 한 장과 사람 심사 대상 세그먼트**
>
> - **무엇을** 세그먼트마다 요율과 구간과 판정을 적은 요율표 한 장과 리포트 하나를 만듭니다.
> - **무엇으로** 5일 동안 쓴 계약 데이터 하나로 만들고, 준비 셀이 값과 구간까지 만들어 줍니다.
> - **인정** 값 2개와 그래프가 모두 통과한 노트북과 요율표 파일을 함께 제출하면 인정됩니다.
> - **예** 표 한 행은 세그먼트 · 계약수 · 무풀링 · 부분풀링 · 하한 · 상한 · 상대 구간 길이 · 판정 8개 열입니다.
> 
> - 내가 쓴 PROMPT 를 지우지 말고 둔 채 노트북 .ipynb 를 제출할 것
> - 과제 앞부분의 「나의 범위 선언」 3개 항목을 프롬프트보다 먼저 채우고 그대로 쓸 것
> - 표는 요율표_세그먼트별요율_v2.csv 로 행 번호 없이 저장할 것
> - 표의 판정 열에는 자동승인 과 사람심사 2개 값만 적을 것
> - 과제 끝의 「프로젝트 리포트」 6개 항목을 채워 노트북에 남길 것
>
> 마감: 2026-10-05T23:59+09:00
>
> **평가 기준**
>
> | 기준 | 이렇게 하면 충족 | 배점 |
> | --- | --- | --- |
> | 문제 정의 | 「PROMPT 셀」의 PROMPT 가 범위 선언 3개 항목대로 목표·쓸 변수·출력 형식을 담았다 | 25 |
> | 검증 설계 | 구간을 낸 표본의 가장 큰 R-hat 과 가장 작은 유효표본수를 두 수로 적고 기준값과 비교해 판정했다 | 25 |
> | 오류 탐지 | 함정 자가 점검표 3개 항목에 O 를 적고, 그렇게 확인한 내용이 코드의 어느 행이었는지 가운데 열에 적었다 | 25 |
> | 해석·보고 | 결론 한 문장에 값과 단위와 결정 동사가 함께 있고, 한계 한 항목이 내가 고른 방법의 확인 질문에 답한다 | 25 |
>
> 제출은 LMS 레슨 화면의 과제 카드에서 GitHub 또는 Google Drive 링크로 합니다(이 파일에 적으면 제출되지 않습니다).

### 과제 — 세그먼트별 요율표 한 장과 사람 심사 대상 세그먼트

#### 할 일 — 과제 (35분)

1. 바로 아래 「결정 3개」를 읽고 「나의 범위 선언」 3개 항목을 먼저 채웁니다.
2. 그 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다. 세그먼트 2,323개를 한 번에 다루는 모형을 한 번 더 적합하므로 몇 분까지 걸립니다.
3. 그 선언대로, 아래 만들 것대로 「PROMPT 셀」의 PROMPT 에 시킵니다.
4. 미션 1 과 같은 순서로 AI 튜터에게 코드를 받아 ▶ 하고, 채점 결과가 ❌ 면 힌트대로 PROMPT 를 고쳐 다시 합니다.
5. 「AI 코드 표시선」 아래 코드를 보며 「함정 자가 점검표」 3개 항목에 O 나 X 와 확인한 코드 행을 적습니다(루브릭 오류 탐지 25점).
6. 마지막으로 「프로젝트 리포트」 6개 항목을 채우고, 노트북과 표 파일을 함께 제출합니다.

**이 과제의 질문** — 요율표 2,323개 세그먼트 가운데 어디를 그대로 내보내고 어디를 사람이 다시 봐야 합니까?

**사전 강도**는 사전분포가 가진 정보량을 가상 관측 건수로 나타낸 값입니다. 102건이면 계약 102건을 미리 관측한 것과 같은 정도로 작용하고, 클수록 관측이 적은 세그먼트의 추정값이 사전분포의 중심 쪽으로 더 많이 이동합니다.

**쓸 수 있는 것** — 「준비 셀」이 만들어 둔 이름입니다. **새로 적합할 모형은 하나도 없습니다.** 아래 「스코프」 결정에서 242개 세그먼트를 고른 경우 하나만 예외이고, 그때는 미션 1 의 모형을 한 번 다시 실행하고 표집을 1분에서 2분 더 기다립니다.

| 이름 | 무엇 | 다룸 |
|---|---|---|
| `SEG_NAME` | 세그먼트 이름 | 고정 |
| `N23` | 세그먼트별 계약 수, 단위 건 | 고정 |
| `K23` | 세그먼트별 사고를 겪은 계약 수, 단위 건 | 고정 |
| `RAW23` | 세그먼트별 무 풀링 값, 단위 비율 | 고정 |
| `P_POOL` | 완전 풀링 값 하나, 단위 비율 | 고정 |
| `P_HIER` | 세그먼트별 부분 풀링 값, 단위 비율 | 고정 |
| `LO_HIER` · `HI_HIER` | 95% 신용구간의 아래끝·위끝 | 고정 |
| `MAP_HAND` · `MAP_DATA` | 사전분포 2벌이 낸 사후분포의 최빈값, 단위 비율 — 둘 중 무엇을 쓸지는 아래 「결정 3개」의 「방법」에서 고릅니다 | 고정 |
| `RHAT_MAX` · `ESS_MIN` | 이 표본의 진단값 2개 | 고정 |

`ESS_MIN` 은 요약표의 `ess_bulk` 열에서 나온 값이고, 이 열은 사후평균처럼 분포의 가운데를 요약할 때의 유효표본수입니다. 표에 적는 하한과 상한은 분포의 양쪽 끝이라, 그 두 열을 보고할 때에는 같은 요약표의 `ess_tail` 열 최솟값도 기준값 400 과 대조합니다.

#### 만들 것

「PROMPT 셀」 한 곳에서 아래 5개가 다 나오게 합니다. 굵은 이름은 글자 그대로 사용합니다.

1. **저장한 표의 컬럼 수** — 단위 개 · 정수 · 저장한 파일을 **다시 읽어** 계산한 열 개수
2. **전체 계약의 사고 경험률** — 단위 비율 · 소수점 아래 6자리 · 세그먼트로 나누기 전, 계약 678,013건 가운데 사고를 겪은 계약의 비율
   - `K23` 의 합을 `N23` 의 합으로 나눈 값이고, 「준비 셀」이 출력한 `P_POOL` 과 같은 수입니다
3. **저장할 파일 한 장** — `요율표_세그먼트별요율_v2.csv` · 열은 8개

   | 세그먼트 | 계약수 | 무풀링 | 적을값 | 하한 | 상한 | 상대 구간 길이 | 판정 |
   |---|---|---|---|---|---|---|---|
   | 세그먼트 이름 | 그 세그먼트의 계약 수 | 그 세그먼트만의 기록으로 낸 값 | **「방법」 결정에서 고른 값** | `LO_HIER` 의 값 | `HI_HIER` 의 값 | 구간 길이 ÷ 적을 값 | `자동승인` 또는 `사람심사` |

   - **행 번호는 파일에 쓰지 않습니다.** `to_csv` 는 기본 설정에서 행 번호를 함께 저장하므로, 그대로 두면 다시 읽은 열이 9개가 되어 1번이 통과하지 않습니다
   - 본문에서 부르는 이름은 자동 승인과 사람 심사이고, 판정 열에 적는 문자열만 띄어쓰기 없이 `자동승인` · `사람심사` 입니다
   - 이 표의 값은 사고 경험률이라 노출이 들어가지 않습니다. 실제 요율로 옮기려면 노출을 오프셋으로 둔 빈도 모형이 따로 필요하다는 것을 리포트 한계 항목에 적습니다
4. **대조용 2행** — 판정 열의 `자동승인` 세그먼트 수와 `사람심사` 세그먼트 수를 각각 한 행으로 출력합니다
   - 채점 대상은 아닙니다. 두 수의 합이 저장한 표의 행 수와 같은지 대조하세요. 합이 맞지 않으면 판정 조건이 일부 세그먼트를 어느 쪽으로도 분류하지 못한 것입니다
5. **그래프 한 장** — 가로축은 세그먼트의 계약 수, 세로축은 상대 구간 길이인 산점도
   - 점 : 세그먼트 하나에 점 하나이고, 점의 개수는 「스코프」 결정에서 고른 세그먼트 수와 같습니다 · 가로축 눈금 : 로그 눈금
   - 가로선 : **내가 기준 결정에서 고른 값** 위치에 그림을 좌우로 가로지르는 선 하나
   - 가로축 이름 : 「세그먼트의 계약 수 (건, 로그 눈금)」
   - 세로축 이름 : 「상대 구간 길이 (구간 길이 ÷ 추정값)」

**출력 형식** — 값은 「이름 = 값 단위」 형태로 한 행씩 `print` 합니다. 그래프는 한 장이고, 표는 파일로 저장합니다.

**통과 기준** — 1번과 2번이 각각 굵은 이름 그대로 한 행씩 출력되고 그 값이 모형이 내는 범위에 들며, 5번 그래프가 로그 눈금·가로선 1개와 가로축·세로축 이름을 갖추면 통과입니다. 그래프의 점 개수는 채점하지 않습니다.

#### 프롬프트에 넣을 것(힌트)

- **고정할 이름** — `P_HIER` · `LO_HIER` · `HI_HIER` 는 「준비 셀」이 이미 계산해 둔 값입니다. 다시 적합하지 말고 그대로 쓰라고 적습니다.
- **판정 한 문장** — 상대 구간 길이를 (`HI_HIER` − `LO_HIER`) ÷ 적을 값으로 구하고, 내가 고른 기준값보다 작으면 `자동승인`, 아니면 `사람심사`. 기준값은 수로 확정해 적습니다.
- **저장** — 행 번호 없이 저장한다는 것과, 저장한 파일을 다시 읽어 열 수를 계산한다는 것.
- **그래프** — 가로축과 세로축에 배치할 값 · 가로축 눈금 · 가로선의 위치 · 가로축과 세로축의 이름.

**막히면** — AI 튜터에게 「판정 열이 두 값으로만 나오게 하려면 조건을 어떻게 적나요」 처럼 질문하세요.

<details class="lms-extra"><summary>배경 — 재무팀에 제출할 요율표 한 장</summary>

재무팀에 제출할 것은 미션 3 에서 답한 세그먼트 하나가 아니라 **표 한 장**입니다. 이론의 「8. 오늘의 정리와 팀 토론 질문」이 팀 문항으로 남긴 물음이 여기에서 다시 나옵니다. 관측이 적은 세그먼트에 적을 값을 **둘째 날 사람이 지정한 사전분포**와 **오늘 데이터로 추정한 사전분포** 가운데 무엇으로 낼지, 사람 심사로 분류할 세그먼트를 어디에서 나눌지를 아래 「결정 3개」에서 정합니다.

</details>

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_tweedie_regression_insurance_claims_001.png" width="800" height="400"/>
<p align="center"><em>▲ 같은 프랑스 자동차보험 포트폴리오를 운전자 나이(학습·시험 2장)·차량 나이·보너스말러스로 나눠 사고 빈도를 그린 4장 — 계약이 집중된 운전자 나이 30～60세에서는 관측과 모형 값이 거의 겹치는데 계약이 드문 양끝에서는 관측이 0.30 까지 증가하거나 0 으로 감소해, 오늘 요율표에서 사람 심사로 분류할 세그먼트가 이 양끝에 해당한다 (출처: scikit-learn documentation)</em></p>

4개 패널 모두 파란 점이 그 조건에서 실제로 관측된 빈도, 주황 점이 모형이 산출한 값이고, 연한 초록 면적은 그 조건에 계약이 얼마나 있는지를 나타냅니다. 왼쪽 아래 차량 나이 30년 이상과 오른쪽 아래 보너스말러스 150 이상에서 두 색의 차이가 커지는 부분이, 관측이 적어 모형이 다른 세그먼트의 정보를 많이 사용한 구간입니다.

**결정 3개** — 아래 3개에는 정답이 없습니다. 다만 「스코프」의 첫 선택지 하나만 미션 1 의 모형을 다시 실행해야 합니다. 채점 대상인 값 2개와 그래프는 어느 쪽을 골라도 나오게 되어 있습니다. 달라지는 것은 **그 표를 읽고 내리는 결정**이고, 리포트에 적어야 하는 것도 그 결정입니다. 참고로 적은 수는 「준비 셀」이 만든 값으로 미리 집계한 것이라, 다시 실행하면 마지막 자릿수가 조금 달라집니다.

| 결정 | 고를 수 있는 선택지 | 얻는 것 | 내주는 것 |
|---|---|---|---|
| **스코프**<br/>요율표를 어느 세그먼트 구분으로 적는가 | 지역 × 브랜드 **242개 세그먼트** | 가장 작은 세그먼트도 계약 6건 이상이라 값의 변동이 작습니다 | 차량 출력 등급이 합쳐지고, 다루는 값이 건/년이라 「준비 셀」의 비율 표를 쓸 수 없어 미션 1 의 모형에서 값과 구간을 다시 계산해야 합니다(앞의 「새로 적합할 모형은 하나도 없습니다」의 유일한 예외이고, 그때도 값 2번은 비율로 따로 출력합니다). 이 스코프에서는 「방법」도 계층 모형의 부분 풀링 값 하나로 정해집니다. 「준비 셀」의 사전분포 2벌이 낸 값은 2,323개 세그먼트의 비율이라 242개 세그먼트에 대응하는 값이 없습니다 |
| | 지역 × 브랜드 × 차량 출력 등급 **2,323개 세그먼트** | 구분이 더 세밀하고 「준비 셀」의 값을 그대로 씁니다 | 756개 세그먼트(32.5%)가 무사고이고 계약 수 중앙값이 41건이라, 관측이 적은 세그먼트에 적을 주석이 그만큼 늘어납니다 |
| | 2,323개 세그먼트 중 **계약 수 기준값 이상인 세그먼트만** 적고 나머지는 상위 세그먼트 값을 그대로 적용하기 | 요율표가 짧아지고 관측이 적은 세그먼트가 표에서 사라집니다 | 값을 그대로 적용한 세그먼트가 어느 상위 세그먼트에서 왔는지 적을 열이 필요하고, 계약 1,000건 이상이 156개 세그먼트뿐이라 기준값을 높이면 표가 너무 짧아집니다 |
| **방법**<br/>관측이 적은 세그먼트에 적을 값을 무엇으로 산출하는가 | **사람이 지정한 사전분포**의 값(사전 강도 102건 · 중심 0.0588) | 사전 강도를 사람이 지정했으므로 근거를 말로 설명할 수 있고, 다음 분기에도 그대로입니다 | 계약 63건 세그먼트에서 0.0552 로 3개 가운데 가장 높습니다. 다만 이 값은 사후분포의 최빈값이고 같은 사후분포의 평균은 0.0606 이라 0.54%p 더 큽니다. 요율은 기대 손해액에 대응하는 값이라 표에는 사후평균을 적는 것이 원칙이고, 사고 경험률처럼 오른쪽으로 치우친 사후분포에서는 최빈값이 평균보다 작아 관측이 적은 세그먼트일수록 낮게 적힙니다. 하한·상한 열에는 계층 모형이 낸 `LO_HIER` · `HI_HIER` 를 그대로 적으므로, 한 행 안에서 값은 최빈값 공식에서 구간은 계층 모형에서 왔다는 것도 리포트 한계 항목에 적습니다. 리포트에서 답할 질문 : 102건과 중심 0.0588 을 무엇을 근거로 정했고, 최빈값과 사후평균의 0.54%p 차이를 요율표에서 어떻게 다루는가 |
| | **데이터로 추정한 사전분포**의 값(사전 강도 약 300건 · 중심 0.0502) | 사전 강도를 사람이 지정하지 않아도 되고, 같은 데이터에서 나온 수라 설명이 짧습니다 | 계약 63건 세그먼트에서 0.0501 이고, 다음 분기에 다시 실행하면 그 약 300건이 달라집니다. 이 값도 앞 행과 같은 사후분포의 최빈값이라 같은 사후분포의 평균 0.0525 보다 0.24%p 작고, 기대 손해액을 적어야 하는 요율표에서 관측이 적은 세그먼트가 그만큼 낮게 적힙니다. 하한·상한 열은 이 선택지에서도 계층 모형이 낸 구간이라 앞 행과 같은 한계가 남습니다. 리포트에서 답할 질문 : 약 300건이 데이터로 추정한 수라는 사실과, 다음 분기에 다시 추정하면 요율표가 어떻게 달라지는가, 그리고 최빈값 대신 사후평균 0.0525 를 적으면 무엇이 달라지는가 |
| | **계층 모형의 부분 풀링 값** | 값과 구간이 같은 모형에서 한꺼번에 나와 표의 한 행이 서로 맞고, 3개 가운데 이 값만 사후분포의 평균이라 이론이 요율표에 적은 0.0503 과 계산 방식이 같습니다 | 계약 63건 세그먼트에서 0.0502 로 완전 풀링 값 0.0502 와 소수점 아래 넷째 자리까지 같습니다. 이 값은 표집에서 나오므로 실행할 때마다 넷째 자리가 달라집니다. 미션 3 에서 0.0504 가, 이론에서 0.0503 이 나온 것도 같은 이유이므로, 값과 함께 R-hat 과 유효표본수를 보고합니다. 리포트에서 답할 질문 : 값과 구간을 낸 표본의 R-hat 과 유효표본수가 얼마이고 그 판정은 통과인가 |
| **기준**<br/>자동 승인과 사람 심사를 나누는 기준값 | 상대 구간 길이 **1.0** | 자동 승인 2,254개 세그먼트 · 사람 심사 69개 세그먼트로, 사람이 볼 세그먼트가 가장 적습니다 | 구간 길이가 값과 거의 같은 세그먼트가 그대로 요율표에 실립니다 |
| | 상대 구간 길이 **0.75** | 자동 승인 525개 세그먼트 · 사람 심사 1,798개 세그먼트로, 요율표에 그대로 나가는 세그먼트는 구간이 확실히 짧은 세그먼트뿐입니다 | 사람 심사 1,798개 세그먼트는 기준 1.0 을 골랐을 때의 69개 세그먼트보다 26배 많습니다 |
| | 상대 구간 길이 **0.5** | 자동 승인 172개 세그먼트 · 사람 심사 2,151개 세그먼트로, 자동으로 나가는 값은 거의 의심할 여지가 없습니다 | 사실상 표 전부를 사람이 봅니다. 자동 승인을 두는 의미가 없어집니다 |

「스코프」에서 고른 선택지가 요율표의 행 수와 그래프의 점 수를 함께 정합니다. 242개 세그먼트를 고르면 242행에 점 242개이고, 2,323개 세그먼트면 2,323행에 점 2,323개이며, 계약 수 기준값 이상만 적으면 통과한 세그먼트 수만큼입니다.

위 표에 적은 집계 수는 「방법」을 계층 모형의 부분 풀링 값으로 두고 센 수입니다. 상대 구간 길이의 분모가 그 「방법」이 낸 값이므로, 사람이 지정한 사전분포를 고르면 같은 기준값 1.0 에서 사람 심사가 358개 세그먼트로 늘어납니다. 부분 풀링 값으로 센 69개 세그먼트의 5배가 넘는 수입니다. 다만 상대 구간 길이가 1.0 바로 근처인 세그먼트가 많고 값과 구간이 표집에서 나오므로, 같은 코드를 다시 실행하면 이 두 수가 각각 10 안팎 움직입니다. 첫날의 ±2 표준오차 구간으로 같은 계산을 하면 상대 구간 길이 1.0 미만이 1,166개 세그먼트인데, 그중 756개 세그먼트는 값도 구간 길이도 0 이라 처음부터 요율표에 적을 수 없는 세그먼트였고, 실제로 적을 수 있던 세그먼트는 410개입니다.

<details class="lms-extra"><summary>배경 — 세그먼트를 잘게 나눌 때의 값 분포와 관측이 적은 세그먼트의 형태</summary>

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_poisson_regression_non_normal_loss_002.png" width="800" height="300"/>
<p align="center"><em>▲ 관측 1장과 모형 3개를 학습·시험에서 각각 그린 히스토그램 8장 — 관측은 0 부터 30 까지 분포하는데 모형이 산출한 값은 아무리 넓어도 0 과 4 사이에 모여 있어, 요율표에 적히는 값은 관측보다 좁게 분포한다 (출처: scikit-learn documentation)</em></p>

위쪽 4장이 학습 데이터, 아래쪽 4장이 시험 데이터이고 각각 맨 왼쪽 `Data` 가 실제 관측이며 오른쪽 3개가 모형이 산출한 값입니다. 가로축은 사고 빈도, 세로축은 그 값에 해당하는 계약 수입니다. 세로축이 로그 눈금이라 한 단계 올라갈 때마다 값이 10배가 됩니다. 「스코프」 결정에서 세그먼트를 잘게 나눌수록 이 분포가 넓어지고 관측이 적은 세그먼트도 함께 증가합니다.

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_tweedie_regression_insurance_claims_002.png" width="800" height="300"/>
<p align="center"><em>▲ 같은 포트폴리오를 운전자 나이로 나눠 건당 평균 손해액을 그린 2장 — 계약이 드문 20세 미만과 80세 초과에서 관측이 7,000 이나 20,000 까지 변동하는 동안 모형 값은 평평하게 남아, 「방법」 결정에서 어느 선택지를 골라도 관측이 적은 세그먼트를 처리하는 방식이 이 주황 점과 같다 (출처: scikit-learn documentation)</em></p>

위가 학습 데이터, 아래가 시험 데이터이고 가로축이 운전자 나이, 세로축이 그 나이대의 평균 손해액입니다. 파란 점이 실제 관측이고 주황 점이 모형이 산출한 값이며, 세로축 단위가 유로라 오늘 표에 그대로 옮길 수는 없습니다.

</details>

**약 300건은 어디에서 나온 수인가** — 이론의 「6. 사전 강도를 데이터로 추정」이 세그먼트 간 표준편차 τ 를 가상 관측 건수로 환산해 얻은 수이고, 둘째 날 사람이 지정한 102건의 3배에 가깝습니다.

<details class="lms-extra"><summary>자세히 — 두 사전분포가 낸 값의 계산식</summary>

환산식과 그 유도는 이론에서 확인할 수 있습니다. 그 식에 세그먼트 간 표준편차 τ 를 대입해 얻은 301.8건을 본문과 「준비 셀」의 코드가 약 300건으로 반올림해 사용하므로, 아래 표의 가상 사고 건수도 300건을 대입한 15.1건입니다. 「준비 셀」의 `MAP_HAND` · `MAP_DATA` 는 두 사전 강도를 각각 대입해 사후분포의 최빈값을 공식으로 바로 계산한 값이고(이름의 MAP 이 사후분포의 최빈값입니다), 식은 이 하나입니다.

$$
\frac{K + a - 1}{N + m - 2}
$$

| 기호 | 무엇 | 사람이 지정한 사전분포 | 데이터로 추정한 사전분포 |
|---|---|---|---|
| $K$ | 그 세그먼트에서 사고를 겪은 계약 수 | `K23` | `K23` |
| $N$ | 그 세그먼트의 계약 수 | `N23` | `N23` |
| $a$ | 사전분포가 중심 쪽에 배정한 가상 사고 건수 | 6건 | 0.05024 × 300 = 15.1건 |
| $m$ | 사전 강도 — 가상 관측 전체 건수 | 102건 | 300건 |

</details>

두 사전분포가 얼마나 다른지도 한 문장으로 적어 둡니다. 2,323개 세그먼트 전부에서 두 요율표의 차이는 세그먼트당 중앙값 0.24%p 로 작은데, **144개 세그먼트에서는 1%p 를 초과합니다.** 평균만 보면 같은 표처럼 보이고 세그먼트를 하나씩 보면 다른 표입니다.

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_poisson_regression_non_normal_loss_003.png" width="600" height="400"/>
<p align="center"><em>▲ 요율표에 적은 값이 실제와 맞는지 계약을 예측 위험 순으로 정렬해 본 그림 4장 — 앞의 3장은 예측과 관측 두 선이 거의 겹치는데 오른쪽 아래 <code>DummyRegressor()</code> 만 0.10 에서 평평해, 이것이 오늘 미션 1 의 완전 풀링 값 0.1006 건/년이다 (출처: scikit-learn documentation)</em></p>

가로축은 계약을 예측 위험이 낮은 쪽부터 매긴 순위이고, 세로축은 그 구간에서 실제로 난 사고 빈도입니다. 파란 × 가 모형이 산출한 값이고 주황 ● 가 실제로 관측된 값이며, 세로축이 연간 사고율이라 오늘 과제에서 다루는 비율과는 다른 값입니다.

프롬프트를 쓰기 전에 3개 항목을 먼저 채웁니다. 각 항목에 「왜」까지 적어 두면 리포트 2번 항목이 그대로 완성됩니다.

> 나의 범위 선언:
> - 스코프 :
> - 방법 :
> - 기준 :

In [ ]:
# 과제 · 준비 — 읽고 실행만 합니다. 이 셀이 과제에 쓸 변수를 직접 만듭니다
if "N23" not in globals():                                       # 이 과제만 따로 열어도 이 셀부터 실행하면 됩니다
    from sklearn.datasets import fetch_openml                     # (미션 3 을 이미 돌렸으면 통째로 건너뜁니다)
    df = fetch_openml(data_id=41214, as_frame=True).frame
    df["IDpol"] = df["IDpol"].astype(int)
    for c in ["Area", "VehBrand", "VehGas", "Region"]:
        df[c] = df[c].astype(str).str.strip("'").astype("category")
    df["ClaimNb"] = df["ClaimNb"].astype(int).clip(upper=4)        # 5일 내내 고정해 온 전처리 2개 규칙
    df["Exposure"] = df["Exposure"].clip(upper=1.0)
    df["사고겪음"] = (df["ClaimNb"] > 0).astype(int)
    SEGS2323 = (df.groupby(["Region", "VehBrand", "VehPower"], observed=True)
                   .agg(n=("IDpol", "size"), k=("사고겪음", "sum"))
                   .reset_index().sort_values(["Region", "VehBrand", "VehPower"])
                   .reset_index(drop=True))
    N23 = SEGS2323["n"].to_numpy(dtype=float)
    K23 = SEGS2323["k"].to_numpy(dtype=float)
    RAW23 = K23 / N23
    P_POOL = float(K23.sum() / N23.sum())
    J23 = int(len(N23))

SEG_NAME = (SEGS2323["Region"].astype(str) + " x " + SEGS2323["VehBrand"].astype(str)
             + " x " + SEGS2323["VehPower"].astype(str) + "등급").to_numpy()   # 쓸 변수 1 — 세그먼트 이름

# 미션 3 의 계층 이항을 시드 7 로 한 번 더 실행합니다. 표에 적을 2,323개 세그먼트의 값과 구간이
# 한 표본에서 함께 나와야 하는데, 미션 3 의 사후 표본은 학생이 붙인 이름이 제각각이라 이 셀이 가져다
# 쓸 수 없습니다. 시드가 미션 3 의 49 와 다르므로 미션 3 에서 본 값과 마지막 자릿수가 다릅니다
if "P_HIER" not in globals():                     # 한 번 얻어 두면 이 셀을 다시 실행해도 표집하지 않습니다
    with pm.Model():
        mu = pm.Normal("mu", -2.94, 1.0)
        tau = pm.HalfNormal("tau", 1.0)
        z = pm.Normal("z", 0.0, 1.0, shape=J23)
        theta = pm.Deterministic("theta", tau * z)
        pm.Binomial("obs", n=N23, p=pm.math.sigmoid(mu + theta), observed=K23)
        _posterior = pm.sample(draws=2000, tune=2000, chains=4, random_seed=7, progressbar=False)
    _log_vals = _posterior.posterior["mu"].values[..., None] + _posterior.posterior["theta"].values
    _seg_vals = (1 / (1 + np.exp(-_log_vals))).reshape(-1, J23)
    P_HIER = _seg_vals.mean(axis=0)                          # 쓸 변수 2 — 세그먼트별 부분 풀링 값 (비율)
    LO_HIER = np.percentile(_seg_vals, 2.5, axis=0)          # 쓸 변수 3 — 95% 신용구간의 아래끝
    HI_HIER = np.percentile(_seg_vals, 97.5, axis=0)         # 쓸 변수 4 — 95% 신용구간의 위끝
    _summary = az.summary(_posterior, var_names=["mu", "tau", "theta"])
    RHAT_MAX = float(_summary["r_hat"].max())               # 쓸 변수 5 — 이 표본의 가장 큰 R-hat
    ESS_MIN = float(_summary["ess_bulk"].min())             # 쓸 변수 6 — 이 표본에서 ess_bulk 열의 최솟값

# 사전분포 2벌의 공식으로 바로 구하는 해(closed-form) — 둘째 날의 그 계산이라 표집 없이 한 번의 계산으로 나옵니다
M_HAND, A_HAND = 102.0, 6.0        # 사람이 지정한 사전분포 — 가상 관측 102건에 사고 6건 (중심 6/102)
M_DATA = 300.0                     # 데이터로 추정한 사전 강도 — 세그먼트 간 표준편차에서 환산한 수
MAP_HAND = (K23 + A_HAND - 1) / (N23 + M_HAND - 2)                    # 쓸 변수 7 — 사람이 지정한 사전분포의 값
MAP_DATA = (K23 + M_DATA * P_POOL - 1) / (N23 + M_DATA - 2)           # 쓸 변수 8 — 데이터로 추정한 사전분포의 값

_i63 = int(np.flatnonzero((N23 == 63) & (K23 == 4))[0])
print(f"세그먼트 {J23:,} 개 · 계약 수 중앙 {np.median(N23):,.0f} 건 · 사고를 한 건도 겪지 않은 세그먼트 "
      f"{int((K23 == 0).sum()):,} 개 ({(K23 == 0).mean() * 100:.1f}%) · 계약 1,000건 이상 "
      f"{int((N23 >= 1000).sum()):,} 개")
print(f"여기서 다루는 값은 사고 경험률(비율) 하나입니다 — 완전 풀링 값 {P_POOL:.6f}")
print(f"계층 사후 진단 — 가장 큰 R-hat {RHAT_MAX:.4f} (기준값 1.01) · 가장 작은 유효표본수 "
      f"{ESS_MIN:,.0f} 개 (기준값 400)")
print(f"계약 63건·사고 4건 세그먼트 — 무 풀링 {RAW23[_i63]:.6f} · 사람이 지정한 사전분포 {MAP_HAND[_i63]:.6f}"
      f" · 데이터로 추정한 사전분포 {MAP_DATA[_i63]:.6f} · 부분 풀링 {P_HIER[_i63]:.6f}"
      f" [{LO_HIER[_i63]:.4f}, {HI_HIER[_i63]:.4f}]")
print("쓸 수 있는 이름 : SEG_NAME (세그먼트 이름) · N23 (세그먼트별 계약 수, 건) · K23 (사고를 겪은 계약 수, 건)"
      " · RAW23 (무 풀링, 비율) · P_POOL (계약 전체, 비율) · J23 (세그먼트 수)"
      " · P_HIER (부분 풀링 값, 비율) · LO_HIER · HI_HIER (95% 신용구간의 두 끝)"
      " · MAP_HAND · MAP_DATA (사전분포 2벌이 낸 값, 비율) · M_HAND · M_DATA (그 사전 강도, 건)"
      " · RHAT_MAX · ESS_MIN (이 표본의 진단값 2개)")

In [ ]:
# 과제 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

**함정 자가 점검표** — 「PROMPT 셀」을 실행한 뒤, 「AI 코드 표시선」 아래 코드를 보며 3개 항목에 O 나 X 를 적고, 그렇게 확인한 내용이 코드의 어느 행이었는지도 가운데 열에 적습니다. 하나라도 X 면 그 코드 행을 고쳐 달라고 PROMPT 에 한 문장 추가하고 다시 실행하세요.

| # | 확인할 것 | 코드의 어느 행 | O / X |
|---|---|---|---|
| 1 | 한 표에 사고 경험률(비율)과 연간 사고율(건/년)을 **함께 적지 않았다**, 그리고 한 행의 적을값·하한·상한과 상대 구간 길이의 분모가 모두 같은 값이다 (요율표의 한 열에는 한 가지 값만 적는다) |  |  |
| 2 | 표의 하한·상한을 낸 표본의 가장 큰 R-hat 과 가장 작은 유효표본수를 어느 모수의 값인지 이름과 함께 이론에서 정한 기준값 2개와 비교해 통과·미통과를 판정하고, 그 판정을 리포트에 적었다 |  |  |
| 3 | 「방법」 결정에서 고른 선택지의 근거와, 그 선택지 때문에 다음 분기에 무엇이 달라지는지를 결정표의 확인 질문대로 리포트에 적었다 |  |  |

**제출 전 마지막 한 장** — 값 2개와 그래프가 모두 통과로 판정되었으면, 아래 6개 항목을 이 셀에 채워 둡니다(제출 노트북에서 이 셀이 리포트에 해당합니다). 채점자가 가장 먼저 읽는 것이 이 6개 항목입니다. 1 · 3 · 5 번 항목이 재무팀에 그대로 읽어 줄 브리핑 3문장입니다.

> 프로젝트 리포트:
> 1. 내가 좁힌 질문 — 한 문장. 재무팀의 물음을 내 말로 다시 적습니다
> 2. 범위 선언과 이유 — 3문장. 스코프 · 방법 · 기준을 각각 「무엇을 골랐고 왜」로
> 3. 결론 — 한 문장. 요율표에 적을 값이 무엇이고, 몇 개 세그먼트를 그대로 내보내고 몇 개 세그먼트를 사람 심사로 분류하는지를 수와 단위와 결정 동사로
> 4. 근거 — 2문장. 한 문장은 반드시 이 표본의 **가장 큰 R-hat** 과 **가장 작은 유효표본수** 두 값과 그 통과·미통과 판정입니다
> 5. 한계 — 2문장. 두 번째 문장은 「방법」 결정표에서 내가 고른 선택지의 「내주는 것」 열에 적힌 확인 질문에 답합니다
> 6. 다음 확인 사항 — 한 문장. 이 표를 실제로 쓰기 전에 무엇을 더 확인할지(한 예 : 적을값 × 계약수의 합이 `K23` 의 합과 맞는지)

6개 항목을 다 적었으면 노트북과 `요율표_세그먼트별요율_v2.csv` 를 함께 제출합니다.

## 5일 차 정리 — 사고율 추정에서 요율표까지

프라이싱 분석가로 진행한 5일 과정을 마칩니다. 오늘 아침의 물음은 「계약 63건인 세그먼트에 적을 요율을 다른 세그먼트의 기록까지 함께 사용해 정해도 되는가」였고, 답은 한 문장입니다. **관측이 적은 세그먼트의 값은 세그먼트 하나의 기록이 아니라 세그먼트 간 표준편차 안에서 함께 추정하고, 그렇게 나온 값은 R-hat 과 유효표본수를 먼저 통과한 뒤에야 요율표에 적습니다.**

| 오늘의 질문 | 우리가 얻은 답 |
|---|---|
| 관측이 적은 세그먼트의 추정값은 어느 쪽으로 이동하는가? | 노출 6.62년에 사고 2건이던 세그먼트의 무 풀링 값 0.3021 건/년이 완전 풀링 값에 근접한 위치까지 이동했습니다. 반대로 노출 35,949년인 세그먼트는 0.0890 그대로입니다. **수축의 크기는 그 세그먼트에 누적된 관측의 양이 정합니다** |
| 이 사후 표본을 요율표에 사용해도 되는가? | 노출 6.62년 세그먼트의 부분 풀링 값은 전임 담당자 표본에서 0.1041, 표집기를 바꾼 표본에서 0.1038 로 소수점 아래 3자리까지 같았습니다. 틀린 것은 값이 아니라 화면에 출력된 R-hat 1.09 · 유효표본수 34 를 읽지 않은 것이었습니다. **값이 타당해 보이는 것과 그 값을 신뢰할 수 있는 것은 서로 다른 진단값으로 판정합니다** |
| 계약 63건 세그먼트의 구간은 얼마나 짧아지는가? | 첫날의 ±2 표준오차 구간 12.29%p 이던 구간 길이가 4.58%p 로 짧아졌습니다(표집이라 실행할 때마다 소수점 아래 첫째 자리가 달라집니다. 이론에서는 반복 20,000회로 계산해 4.46%p 를 얻었습니다). 상대 구간 길이 1 미만인 세그먼트가 2,254개이고, 첫날 그대로 적을 수 있던 세그먼트는 410개였습니다. **같은 데이터에서 요율표에 실을 수 있는 세그먼트가 5배 이상 증가했습니다** |
| 그래서 요율표에는 무엇을 적는가? | 그 결정은 과제에서 각자 정합니다. 두 사전분포는 평균만 보면 같은 표처럼 보이고 세그먼트를 하나씩 보면 다른 표라서, 그 차이는 아래 팀 토론 문항에서 수로 답합니다. 오늘 산출한 것은 정답 하나가 아니라, 어느 쪽을 골라도 근거를 수로 제시할 수 있는 표 한 장입니다 |

5일을 한 문장으로 이으면 다음과 같습니다. 첫날에는 세그먼트마다 사고율에 ±2 표준오차로 구간을 계산했고, 둘째 날에는 관측이 적은 세그먼트에 사람이 지정한 사전분포를 적용해 값을 조정했고, 셋째 날에는 계약 특성으로 계약 하나하나에 요율을 부여했고, 넷째 날에는 그 표를 재표본과 새 데이터로 민감도 검사했습니다. 오늘은 세그먼트를 하나씩 따로 다루지 않고 2,323개를 한 번에 묶어 추정했고, 둘째 날 직접 지정한 사전 강도 102건도 데이터로 추정해 약 300건을 얻었습니다.

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_poisson_regression_non_normal_loss_001.png" width="800" height="200"/>
<p align="center"><em>▲ 5일 동안 쓴 freMTPL2freq 를 전처리하기 전 그대로 그린 3장 — 계약당 사고 건수는 16건까지, 노출은 2.0년까지, 노출로 나눈 빈도는 약 700 건/년까지 꼬리가 이어져, <code>ClaimNb</code> 를 4건에서 <code>Exposure</code> 를 1.0년에서 상한 처리한 이유가 이 3개 꼬리다 (출처: scikit-learn documentation)</em></p>

3개 패널 모두 세로축이 계약 수이고 로그 눈금입니다. 오른쪽 빈도가 그만큼 큰 값이 되는 것은 노출이 며칠뿐인 계약에서 사고가 1건 발생하면 연간 사고율이 급격히 커지기 때문입니다.

## 다음 모듈 예고 — 순서가 있는 데이터

오늘로 모형추정 모듈을 마칩니다. 5일 내내 다룬 계약 678,013건에는 **시간**이 없었습니다. 어느 계약을 먼저 보든 결과가 같았고, 세그먼트를 지역과 브랜드와 차량 출력 등급으로 자유롭게 다시 나눌 수 있었던 것도 같은 이유입니다. 다음 모듈의 데이터는 앞 시점의 값이 뒤 시점의 값에 영향을 주고, 계절이 반복되며, 행의 순서를 무작위로 바꾸는 순간 미래를 보고 과거를 예측하는 일이 됩니다.

그래도 5일 동안 만든 절차는 그대로 사용합니다. 값과 함께 구간을 적고, 관측이 적은 세그먼트에는 다른 세그먼트의 정보를 함께 쓰고, 표본을 신뢰할 수 있는지 먼저 판정하는 절차입니다. 바뀌는 것은 「무엇을 다른 세그먼트라고 부를 것인가」 하나이고, 다음 모듈에서 그 이름에 해당하는 것은 **앞 시점**입니다.

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Random-data-plus-trend-r2.png" width="560" height="416"/>
<p align="center"><em>▲ 다음 모듈이 다룰 데이터의 형태 — 시점 순서를 가로축에 배치하자 값이 −20 에서 30 사이로 변동하는 가운데에도 1,000 시점 동안 0 에서 10 까지 증가하는 추세가 드러나, 오늘까지 다룬 계약 678,013건에는 없던 시간축이 하나 생긴다 (출처: Wikimedia Commons)</em></p>

검은 선이 시점마다 기록한 값, 빨간 선이 인접한 시점끼리 평균 낸 이동평균, 파란 직선이 전체 추세입니다.

마무리로 두 문장을 남깁니다. 오늘 채점 뒤에 열린 완성본과 내 코드가 달랐던 부분 한 곳, 그리고 과제에서 요율표에 적기로 고른 값과 그 이유입니다. 팀 채널에 게시하고 다른 사람의 두 문장을 읽는 것이 이 모듈의 마지막 복습이 됩니다.

<div style="background:#F2F2F2;color:#000000;padding:28px;border-radius:12px;border:1px solid #E6E6E6;margin-top:16px;border-left:4px solid #051C2C">
<h2 style="color:#000000;margin:0 0 8px 0">수고했습니다 — 모형추정 심화통계 모듈 완료</h2>
<div style="color:#656565">사고율 하나에서 시작해 계약별 요율을 부여하고, 구간과 검증을 추가하고, 마지막으로 관측이 적은 세그먼트를 다른 세그먼트의 정보와 함께 추정해 마쳤습니다.<br/>다음 모듈에서는 순서가 있는 데이터를 다룹니다.</div>
</div>

> **[문제] 사전 강도 102건과 약 300건 가운데 무엇을 적는가 — 팀에서 3문장으로 답하기**
>
> 재무팀이 마지막으로 질문합니다. "관측이 적은 세그먼트에 적을 값을 정하는 그 사전 강도는 둘째 날에 우리가 102건으로 정했습니다. 오늘은 데이터로 추정하니 약 300건이라고 들었습니다. 요율표에는 어느 쪽을 적습니까?" 오늘 만든 표만으로 답해 보세요. (가) 두 값이 실제로 얼마나 다른지 수로, (나) 데이터로 추정한 사전 강도를 쓸 때 다음 분기에 무엇이 달라질 수 있는지, (다) 그래도 그 표를 이사회에 제출할 수 있게 하려면 표에 무엇을 함께 적어야 하는지를 3문장으로 적어 보세요.